Setup

In [ ]:
import os, shutil, subprocess, sys, torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")

for path in ["/kaggle/working", "/kaggle/tmp", "/tmp"]:
    if os.path.exists(path):
        total, used, free = shutil.disk_usage(path)
        print(f"Disk {path}: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java status:", "OK" if java_check.returncode == 0 else "MISSING")
if java_check.stderr:
    print(" ", java_check.stderr.splitlines()[0])

In [ ]:
import os

os.makedirs("/kaggle/tmp/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U peft accelerate huggingface_hub wandb pycocoevalcap easyocr rapidocr onnxruntime open-clip-torch

import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        val = secrets.get_secret(name)
        print(f"Secret {name}: FOUND (len={len(val)})")
    except Exception as e:
        print(f"Secret {name}: MISSING ({type(e).__name__})")

BLIP

In [ ]:
import time
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# Create a crisp synthetic test image with visible text (no network dependency)
img = Image.new("RGB", (512, 512), color=(240, 235, 225))
draw = ImageDraw.Draw(img)
draw.rectangle([120, 90, 392, 430], fill=(200, 45, 45), outline=(40, 40, 40), width=4)
draw.rectangle([145, 160, 367, 320], fill=(255, 255, 255))
font = ImageFont.load_default()
draw.text((165, 190), "ORGANIC\nTOMATO SOUP\nNET WT 14 OZ", fill=(20, 20, 20), font=font)

model_id = "Salesforce/blip2-opt-2.7b"
t0 = time.time()
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"Allocated after load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

prompt = "Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:"
inputs = processor(images=img, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)

print("input_ids shape:", inputs["input_ids"].shape)
print("decoded input_ids:", repr(processor.tokenizer.decode(inputs["input_ids"][0])))

torch.cuda.reset_peak_memory_stats(0)
t1 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs,
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
    )
elapsed = time.time() - t1

full_text = processor.batch_decode(out, skip_special_tokens=True)[0]
new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
print(f"Generated output shape: {out.shape} | Time: {elapsed:.2f}s | Peak VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")
print("Raw decoded output:", repr(full_text))

loRA

In [ ]:
import torch
from peft import LoraConfig, get_peft_model

# Freeze vision + Q-Former explicitly
for p in model.vision_model.parameters():
    p.requires_grad = False
for p in model.qformer.parameters():
    p.requires_grad = False
model.query_tokens.requires_grad = False
for p in model.language_projection.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model.language_model = get_peft_model(model.language_model, lora_cfg)
model.language_model.print_trainable_parameters()

# Verify LoRA weights are fp32 while base weights are fp16
lora_dtypes = {p.dtype for n, p in model.named_parameters() if p.requires_grad}
print("Trainable param dtypes:", lora_dtypes)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()

# Build 1 training example: prompt + target caption + EOS, prompt tokens masked to -100
prompt_text = "Visible text: ORGANIC TOMATO SOUP. Alt-text:"
target_text = " A red can of organic tomato soup with a white label."

prompt_enc = processor(images=img, text=prompt_text, return_tensors="pt")
full_enc = processor(
    images=img,
    text=prompt_text + target_text + processor.tokenizer.eos_token,
    return_tensors="pt",
)

input_ids = full_enc["input_ids"].to("cuda:0")
attention_mask = full_enc["attention_mask"].to("cuda:0")
pixel_values = full_enc["pixel_values"].to("cuda:0", torch.float16)

labels = input_ids.clone()
prompt_len = prompt_enc["input_ids"].shape[1]
labels[:, :prompt_len] = -100  # mask prompt (and any image placeholder tokens)

torch.cuda.reset_peak_memory_stats(0)
scaler = torch.amp.GradScaler("cuda")

with torch.amp.autocast("cuda", dtype=torch.float16):
    outputs = model(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )
    loss = outputs.loss

scaler.scale(loss).backward()
print(f"Dry-run loss: {loss.item():.4f} (finite={torch.isfinite(loss).item()})")
print(f"Peak VRAM during 1-step train: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")

model.zero_grad(set_to_none=True)

OCR Benchmark

In [ ]:
import time
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

# Make a slightly blurred & tilted label image to test OCR realism
ocr_img = Image.new("RGB", (400, 260), (235, 230, 220))
d = ImageDraw.Draw(ocr_img)
d.text((40, 60), "TYLENOL Extra Strength\n500 mg Acetaminophen\n100 Caplets", fill=(15, 15, 15))
ocr_img = ocr_img.rotate(6, expand=True, fillcolor=(235, 230, 220)).filter(ImageFilter.GaussianBlur(0.6))
ocr_np = np.array(ocr_img)

# 1. EasyOCR (CPU + GPU)
import easyocr

t0 = time.time()
reader_cpu = easyocr.Reader(["en"], gpu=False, verbose=False)
init_easy_cpu = time.time() - t0
t1 = time.time()
res_easy_cpu = reader_cpu.readtext(ocr_np)
lat_easy_cpu = time.time() - t1

t0 = time.time()
reader_gpu = easyocr.Reader(["en"], gpu=True, verbose=False)
init_easy_gpu = time.time() - t0
t1 = time.time()
res_easy_gpu = reader_gpu.readtext(ocr_np)
lat_easy_gpu = time.time() - t1

print(f"[EasyOCR CPU] init={init_easy_cpu:.2f}s | infer={lat_easy_cpu:.3f}s")
print("  texts:", [(t, round(c, 2)) for _, t, c in res_easy_cpu])
print(f"[EasyOCR GPU] init={init_easy_gpu:.2f}s | infer={lat_easy_gpu:.3f}s")

# 2. RapidOCR (ONNXRuntime CPU - reflects free HF Space UX)
from rapidocr import RapidOCR

t0 = time.time()
rapid = RapidOCR()
init_rapid = time.time() - t0
t1 = time.time()
res_rapid = rapid(ocr_np)
lat_rapid = time.time() - t1

print(f"[RapidOCR CPU] init={init_rapid:.2f}s | infer={lat_rapid:.3f}s")
if res_rapid and getattr(res_rapid, "txts", None):
    print("  texts:", [(t, round(float(s), 2)) for t, s in zip(res_rapid.txts, res_rapid.scores)])
else:
    print("  raw:", res_rapid)

VizWiz captions

In [ ]:
import subprocess, zipfile
from pathlib import Path

# PASTE YOUR CHALLENGE ANNOTATIONS GOOGLE DRIVE LINK HERE:
CHALLENGE_GDRIVE_URL = "https://drive.google.com/file/d/1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH/view?usp=sharing"

!pip install -q gdown

DATA_DIR = Path("/kaggle/tmp/vizwiz")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR = DATA_DIR / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

# 1. Download Challenge Annotations from Google Drive (supports both file and folder URLs)
if "/folders/" in CHALLENGE_GDRIVE_URL:
    subprocess.run(
        ["gdown", "--folder", CHALLENGE_GDRIVE_URL, "-O", str(ANN_DIR)],
        check=True,
    )
else:
    ann_file = DATA_DIR / "annotations_dl"
    subprocess.run(
        ["gdown", "--fuzzy", CHALLENGE_GDRIVE_URL, "-O", str(ann_file)],
        check=True,
    )
    if zipfile.is_zipfile(ann_file):
        print("Extracting annotations zip ...")
        with zipfile.ZipFile(ann_file, "r") as zf:
            zf.extractall(ANN_DIR)
    else:
        # In case it downloaded a single JSON directly
        ann_file.rename(ANN_DIR / "downloaded_annotations.json")

# Also extract any nested zips inside ANN_DIR just in case
for zp in list(ANN_DIR.rglob("*.zip")):
    print("Extracting nested zip:", zp.name)
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(zp.parent)

# 2. Download & extract official train and val images into /kaggle/tmp/vizwiz
image_urls = {
    "train_images": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip",
    "val_images": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip",
}

for name, url in image_urls.items():
    out_zip = DATA_DIR / f"{name}.zip"
    if not out_zip.exists():
        print(f"Downloading {name} (~3.5-10GB) ...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(out_zip), url], check=True)
    extract_dir = DATA_DIR / name
    if not extract_dir.exists():
        print(f"Extracting {out_zip.name} ...")
        with zipfile.ZipFile(out_zip, "r") as zf:
            zf.extractall(extract_dir)

print("\nExtracted annotation files:")
for p in sorted(ANN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p} ({p.stat().st_size / 1e6:.2f} MB)")

Dataset audit

In [ ]:
import json, statistics
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("/kaggle/tmp/vizwiz")

# Locate train.json and val.json inside extracted annotations
ann_files = {p.name: p for p in (DATA_DIR / "annotations").rglob("*.json")}
print("Annotation JSONs:", list(ann_files.keys()))

train_json_path = ann_files.get("train.json")
val_json_path = ann_files.get("val.json")
assert train_json_path and val_json_path, f"Expected train.json and val.json, got {ann_files}"

def find_image_dir(base: Path) -> Path:
    jpgs = list(base.rglob("*.jpg"))
    assert jpgs, f"No jpgs found under {base}"
    return jpgs[0].parent

train_img_dir = find_image_dir(DATA_DIR / "train_images")
val_img_dir = find_image_dir(DATA_DIR / "val_images")
print("Train img dir:", train_img_dir, "| count:", len(list(train_img_dir.glob("*.jpg"))))
print("Val img dir:  ", val_img_dir,   "| count:", len(list(val_img_dir.glob("*.jpg"))))

def audit_split(json_path: Path, img_dir: Path, split_name: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images = data["images"]
    anns = data["annotations"]
    img_by_id = {im["id"]: im for im in images}

    anns_by_img = defaultdict(list)
    missing_files = 0
    for im in images:
        if not (img_dir / im["file_name"]).exists():
            missing_files += 1

    precanned_caps = 0
    rejected_caps = 0
    valid_word_lens = []
    cr_count = 0

    for a in anns:
        cap = a["caption"]
        if "\r" in cap:
            cr_count += 1
        anns_by_img[a["image_id"]].append(a)
        if a.get("is_precanned", False):
            precanned_caps += 1
        elif a.get("is_rejected", False):
            rejected_caps += 1
        else:
            words = cap.strip().split()
            if words:
                valid_word_lens.append(len(words))

    # Image-level unclear & text_detected breakdowns
    maj_precanned_imgs = 0
    all_precanned_or_rejected_imgs = 0
    text_det_imgs = sum(1 for im in images if im.get("text_detected", False))

    for im in images:
        im_anns = anns_by_img[im["id"]]
        n_pre = sum(1 for a in im_anns if a.get("is_precanned", False))
        n_valid = sum(1 for a in im_anns if not a.get("is_precanned", False) and not a.get("is_rejected", False) and a["caption"].strip())
        if n_pre >= 3:  # majority of 5 captions
            maj_precanned_imgs += 1
        if n_valid == 0:
            all_precanned_or_rejected_imgs += 1

    valid_word_lens.sort()
    n_w = len(valid_word_lens)
    summary = {
        "split": split_name,
        "num_images": len(images),
        "num_annotations": len(anns),
        "missing_image_files": missing_files,
        "captions_with_carriage_return": cr_count,
        "precanned_captions": precanned_caps,
        "precanned_caption_pct": round(100 * precanned_caps / len(anns), 2),
        "rejected_captions": rejected_caps,
        "rejected_caption_pct": round(100 * rejected_caps / len(anns), 2),
        "images_text_detected": text_det_imgs,
        "images_text_detected_pct": round(100 * text_det_imgs / len(images), 2),
        "images_majority_precanned_ge3": maj_precanned_imgs,
        "images_majority_precanned_pct": round(100 * maj_precanned_imgs / len(images), 2),
        "images_zero_valid_captions": all_precanned_or_rejected_imgs,
        "valid_caption_words_mean": round(statistics.mean(valid_word_lens), 2),
        "valid_caption_words_median": valid_word_lens[n_w // 2],
        "valid_caption_words_p10": valid_word_lens[int(0.10 * n_w)],
        "valid_caption_words_p90": valid_word_lens[int(0.90 * n_w)],
        "valid_caption_words_max": valid_word_lens[-1],
    }
    return summary, data, anns_by_img

train_summary, train_raw, train_anns_by_img = audit_split(train_json_path, train_img_dir, "train")
val_summary, val_raw, val_anns_by_img = audit_split(val_json_path, val_img_dir, "val")

print("\n=== TRAIN AUDIT ===")
for k, v in train_summary.items(): print(f"  {k}: {v}")
print("\n=== VAL AUDIT ===")
for k, v in val_summary.items(): print(f"  {k}: {v}")

HF Save

In [ ]:
import json, random
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

ART_DIR = Path("/kaggle/working/artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

# Build stratification key per train image: (text_detected, majority_precanned)
img_ids = []
strata = []
for im in train_raw["images"]:
    im_anns = train_anns_by_img[im["id"]]
    is_unclear = sum(1 for a in im_anns if a.get("is_precanned", False)) >= 3
    has_text = bool(im.get("text_detected", False))
    img_ids.append(im["id"])
    strata.append(f"text={int(has_text)}_unclear={int(is_unclear)}")

train_ids, dev_ids = train_test_split(
    img_ids,
    test_size=1000,
    random_state=42,
    stratify=strata,
)

# Verify zero overlap and matching distributions
assert set(train_ids).isdisjoint(set(dev_ids))
val_ids = [im["id"] for im in val_raw["images"]]

def strata_counts(id_list):
    id_set = set(id_list)
    c = defaultdict(int)
    for iid, st in zip(img_ids, strata):
        if iid in id_set:
            c[st] += 1
    return dict(c)

splits_payload = {
    "seed": 42,
    "train_ids": sorted(train_ids),
    "dev_ids": sorted(dev_ids),
    "val_ids": sorted(val_ids),
    "counts": {
        "train": len(train_ids),
        "dev": len(dev_ids),
        "val": len(val_ids),
    },
    "strata_train": strata_counts(train_ids),
    "strata_dev": strata_counts(dev_ids),
}

splits_path = ART_DIR / "splits_v1.json"
report_path = ART_DIR / "data_report_v1.json"

with open(splits_path, "w") as f:
    json.dump(splits_payload, f, indent=2)
with open(report_path, "w") as f:
    json.dump({"train": train_summary, "val": val_summary, "splits": splits_payload["counts"]}, f, indent=2)

print("Split counts:", splits_payload["counts"])
print("Train strata:", splits_payload["strata_train"])
print("Dev strata:  ", splits_payload["strata_dev"])

# Push frozen split & report to a private/public HF Hub dataset repo for persistence
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
user = api.whoami()["name"]
REPO_ID = f"{user}/vizwiz-alttext-artifacts"
api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True, private=False)

for local_p in [splits_path, report_path]:
    api.upload_file(
        path_or_fileobj=str(local_p),
        path_in_repo=f"phase1/{local_p.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message="Phase 1: frozen VizWiz splits and data audit report",
    )
print(f"Uploaded Phase 1 artifacts to https://huggingface.co/datasets/{REPO_ID}")

OCR calibration

In [ ]:
import os, json
from pathlib import Path
from PIL import Image

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")

def find_file(base: Path, filename: str) -> Path:
    matches = [p for p in base.rglob(filename) if "__MACOSX" not in p.parts]
    assert matches, f"Could not find {filename} under {base}"
    return matches[0]

TRAIN_JSON = find_file(DATA_DIR / "annotations", "train.json")
VAL_JSON = find_file(DATA_DIR / "annotations", "val.json")
TRAIN_IMG_DIR = next((DATA_DIR / "train_images").rglob("*.jpg")).parent
VAL_IMG_DIR = next((DATA_DIR / "val_images").rglob("*.jpg")).parent

print("TRAIN_JSON:   ", TRAIN_JSON)
print("VAL_JSON:     ", VAL_JSON)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("VAL_IMG_DIR:  ", VAL_IMG_DIR)

with open(TRAIN_JSON) as f:
    train_json = json.load(f)

sizes = []
for im in train_json["images"][:200]:
    with Image.open(TRAIN_IMG_DIR / im["file_name"]) as p:
        sizes.append(p.size)

widths, heights = zip(*sizes)
print(f"Sampled 200 image sizes: W min/med/max = {min(widths)}/{sorted(widths)[100]}/{max(widths)}, "
      f"H min/med/max = {min(heights)}/{sorted(heights)[100]}/{max(heights)}")

In [ ]:
import json, logging
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from rapidocr import RapidOCR

logging.getLogger("RapidOCR").setLevel(logging.ERROR)
rapid = RapidOCR()

DATA_DIR = Path("/kaggle/tmp/vizwiz")
TRAIN_JSON = DATA_DIR / "annotations" / "annotations" / "train.json"
TRAIN_IMG_DIR = DATA_DIR / "train_images" / "train"

with open(TRAIN_JSON) as f:
    train_json = json.load(f)

anns_map = {}
for a in train_json["annotations"]:
    if not a.get("is_precanned") and not a.get("is_rejected"):
        anns_map.setdefault(a["image_id"], []).append(a["caption"])

if "model" not in globals():
    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    model = Blip2ForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map={"": 0}
    )
    model.eval()

probe_imgs = [im for im in train_json["images"] if im.get("text_detected")][:3] + \
             [im for im in train_json["images"] if not im.get("text_detected")][:2]

print("=== ZERO-SHOT PROMPT COMPARISON (5 REAL VIZWIZ IMAGES) ===")
for im in probe_imgs:
    pil = ImageOps.exif_transpose(Image.open(TRAIN_IMG_DIR / im["file_name"])).convert("RGB")
    ocr_pil = pil.copy()
    ocr_pil.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    res = rapid(np.array(ocr_pil))
    ocr_words = " ".join(
        [t.strip() for t, s in zip(res.txts or [], res.scores or []) if float(s) >= 0.5]
    ).split()[:16]
    ocr_str = " ".join(ocr_words) if ocr_words else "none"

    prompts = {
        "empty": None,
        "a_photo_of": "a photo of",
        "qa": "Question: Describe what is in this image. Answer:",
        "alt_ocr": f"Visible text: {ocr_str}. Alt-text:",
    }
    print(f"\nImage: {im['file_name']} | OCR: {ocr_str}")
    print(f"  Ref[0]: {anns_map.get(im['id'], ['<precanned>'])[0]}")
    for pname, ptxt in prompts.items():
        inp = processor(images=pil, text=ptxt, return_tensors="pt").to("cuda:0", torch.float16)
        in_len = inp["input_ids"].shape[1] if "input_ids" in inp else 0
        with torch.no_grad():
            out = model.generate(**inp, num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3)
        gen = processor.tokenizer.decode(out[0, in_len:], skip_special_tokens=True).strip()
        print(f"  [{pname}]: {gen}")

In [ ]:
import os, time, json, logging
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

TRAIN_JSON = Path("/kaggle/tmp/vizwiz/annotations/annotations/train.json")
TRAIN_IMG_DIR = Path("/kaggle/tmp/vizwiz/train_images/train")
with open(TRAIN_JSON) as f:
    sample_paths = [str(TRAIN_IMG_DIR / im["file_name"]) for im in json.load(f)["images"][:40]]

_WORKER_OCR = None
def _init_worker():
    global _WORKER_OCR
    os.environ["OMP_NUM_THREADS"] = "1"
    logging.getLogger("RapidOCR").setLevel(logging.ERROR)
    from rapidocr import RapidOCR
    _WORKER_OCR = RapidOCR(params={
        "EngineConfig.onnxruntime.intra_op_num_threads": 1,
        "EngineConfig.onnxruntime.inter_op_num_threads": 1,
    })

def _bench_one(path):
    pil = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    pil.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    res = _WORKER_OCR(np.array(pil))
    return len(res.txts) if (res and getattr(res, "txts", None)) else 0

for n_workers in [2, 4]:
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=n_workers, initializer=_init_worker) as ex:
        _ = list(ex.map(_bench_one, sample_paths))
    dt = time.time() - t0
    rate = len(sample_paths) / dt
    print(f"workers={n_workers}: {dt:.1f}s for 40 imgs ({rate:.2f} img/s) | "
          f"ETA dev+val (8.75k)={8750/rate/60:.1f} min | ETA all (31.2k)={31181/rate/60:.1f} min")

In [ ]:
import time, json, logging
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps

# Install onnxruntime-gpu and check if CUDAExecutionProvider works on CUDA 13
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("ORT version:", ort.__version__)
print("ORT available providers:", ort.get_available_providers())

# Inspect RapidOCR's exact config keys so we pass valid params
import rapidocr, yaml
cfg_candidates = list(Path(rapidocr.__file__).parent.rglob("*.yaml"))
print("RapidOCR config files:", [str(p.name) for p in cfg_candidates])
if cfg_candidates:
    with open(cfg_candidates[0]) as f:
        cfg = yaml.safe_load(f)
    print("Top-level config keys:", list(cfg.keys()))
    print("EngineConfig keys:", cfg.get("EngineConfig"))
    print("Global keys:", cfg.get("Global"))

logging.getLogger("RapidOCR").setLevel(logging.ERROR)
from rapidocr import RapidOCR

TRAIN_JSON = Path("/kaggle/tmp/vizwiz/annotations/annotations/train.json")
TRAIN_IMG_DIR = Path("/kaggle/tmp/vizwiz/train_images/train")
with open(TRAIN_JSON) as f:
    sample_paths = [str(TRAIN_IMG_DIR / im["file_name"]) for im in json.load(f)["images"][:40]]

# Test GPU RapidOCR if CUDAExecutionProvider is available
if "CUDAExecutionProvider" in ort.get_available_providers():
    try:
        ocr_gpu = RapidOCR(params={"EngineConfig.onnxruntime.use_cuda": True})
        # Warmup 2 images
        for p in sample_paths[:2]:
            im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
            _ = ocr_gpu(np.array(im))

        t0 = time.time()
        hits = 0
        for p in sample_paths:
            im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
            res = ocr_gpu(np.array(im))
            if res and getattr(res, "txts", None):
                hits += 1
        dt = time.time() - t0
        rate = len(sample_paths) / dt
        print(f"\n[RapidOCR GPU] {dt:.2f}s for 40 imgs ({rate:.2f} img/s) | hits={hits}/40 | "
              f"ETA 31.2k (1 GPU)={31181/rate/60:.1f} min | (2 GPUs)={31181/(rate*2)/60:.1f} min")
    except Exception as e:
        print("GPU RapidOCR failed:", type(e).__name__, e)
else:
    print("CUDAExecutionProvider not available; will optimize CPU multiprocessing.")

Image OCR Cache

In [ ]:
import os, json, time, subprocess
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
TMP_DIR = Path("/kaggle/tmp/ocr_work")
ART_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

# 1. Load frozen splits
splits_path = ART_DIR / "splits_v1.json"
if not splits_path.exists():
    splits_path = Path(hf_hub_download(REPO_ID, "phase1/splits_v1.json", repo_type="dataset", token=hf_token))
with open(splits_path) as f:
    splits = json.load(f)

with open(DATA_DIR / "annotations" / "annotations" / "train.json") as f:
    train_imgs = {im["id"]: im for im in json.load(f)["images"]}
with open(DATA_DIR / "annotations" / "annotations" / "val.json") as f:
    val_imgs = {im["id"]: im for im in json.load(f)["images"]}

train_dir = DATA_DIR / "train_images" / "train"
val_dir = DATA_DIR / "val_images" / "val"

# Ordered tasks: dev (1k) + val (7.75k) first, then train (22.4k)
tasks = []
for split_name, id_list, meta, img_dir in [
    ("dev", splits["dev_ids"], train_imgs, train_dir),
    ("val", splits["val_ids"], val_imgs, val_dir),
    ("train", splits["train_ids"], train_imgs, train_dir),
]:
    for iid in id_list:
        im = meta[iid]
        tasks.append({
            "image_id": iid,
            "split": split_name,
            "path": str(img_dir / im["file_name"]),
            "text_detected_gt": bool(im.get("text_detected", False)),
        })

# 2. Load any prior cached records (local or HF Hub) to support clean resume
partial_path = ART_DIR / "ocr_cache_partial.jsonl"
shard_paths = [TMP_DIR / "shard_0.jsonl", TMP_DIR / "shard_1.jsonl"]

if not partial_path.exists():
    try:
        dl = hf_hub_download(REPO_ID, "phase2/ocr_cache_partial.jsonl", repo_type="dataset", token=hf_token)
        partial_path.write_text(Path(dl).read_text(encoding="utf-8"), encoding="utf-8")
        print("Resumed partial cache from HF Hub.")
    except Exception:
        pass

done = {}
for p in [partial_path, *shard_paths]:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done[rec["image_id"]] = rec

# Consolidate into partial_path and clear old shards before launching fresh workers
if done:
    with open(partial_path, "w", encoding="utf-8") as f:
        for rec in done.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
for sp in shard_paths:
    if sp.exists():
        sp.unlink()

todo = [t for t in tasks if t["image_id"] not in done]
print(f"Total tasks: {len(tasks)} | Already done: {len(done)} | Remaining: {len(todo)}")

# 3. Write standalone worker script (avoids any notebook pickling/CUDA-fork issues)
worker_script = TMP_DIR / "ocr_worker.py"
worker_script.write_text('''
import sys, json, warnings
import numpy as np
from PIL import Image, ImageOps
from rapidocr import RapidOCR, EngineType

warnings.filterwarnings("ignore", message="Using padding='same'")

todo_file, out_file = sys.argv[1], sys.argv[2]
with open(todo_file, "r", encoding="utf-8") as f:
    items = json.load(f)

ocr = RapidOCR(params={
    "Global.log_level": "error",
    "Det.engine_type": EngineType.TORCH,
    "Cls.engine_type": EngineType.TORCH,
    "Rec.engine_type": EngineType.TORCH,
    "EngineConfig.torch.use_cuda": True,
    "EngineConfig.torch.cuda_ep_cfg.device_id": 0,
})

with open(out_file, "a", encoding="utf-8") as fout:
    for idx, item in enumerate(items, start=1):
        pil = ImageOps.exif_transpose(Image.open(item["path"])).convert("RGB")
        orig_w, orig_h = pil.size
        pil.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
        res = ocr(np.array(pil))
        dets = []
        if res and getattr(res, "txts", None):
            boxes = res.boxes if getattr(res, "boxes", None) is not None else [None] * len(res.txts)
            for txt, sc, bx in zip(res.txts, res.scores, boxes):
                sc_f = float(sc)
                t_clean = txt.strip()
                if sc_f >= 0.30 and t_clean:
                    if bx is not None:
                        arr = np.asarray(bx)
                        y_min = round(float(arr[:, 1].min()), 1)
                        x_min = round(float(arr[:, 0].min()), 1)
                    else:
                        y_min, x_min = 0.0, 0.0
                    dets.append({"text": t_clean, "conf": round(sc_f, 4), "y": y_min, "x": x_min})
        rec = {
            "image_id": item["image_id"],
            "split": item["split"],
            "text_detected_gt": item["text_detected_gt"],
            "w": orig_w,
            "h": orig_h,
            "dets": dets,
        }
        fout.write(json.dumps(rec, ensure_ascii=False) + "\\n")
        if idx % 25 == 0:
            fout.flush()
    fout.flush()
''', encoding="utf-8")

# 4. Pre-warm model weights once so workers don't race on downloads, then run 2 GPU workers
if todo:
    from rapidocr import RapidOCR, EngineType
    _ = RapidOCR(params={
        "Global.log_level": "error",
        "Det.engine_type": EngineType.TORCH,
        "Cls.engine_type": EngineType.TORCH,
        "Rec.engine_type": EngineType.TORCH,
        "EngineConfig.torch.use_cuda": True,
        "EngineConfig.torch.cuda_ep_cfg.device_id": 0,
    })

    # Split remaining work 50/50 across GPU 0 and GPU 1
    todo_files = [TMP_DIR / "todo_0.json", TMP_DIR / "todo_1.json"]
    todo_files[0].write_text(json.dumps(todo[0::2]), encoding="utf-8")
    todo_files[1].write_text(json.dumps(todo[1::2]), encoding="utf-8")

    procs = []
    for gpu_id in [0, 1]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        p = subprocess.Popen(
            ["python3", str(worker_script), str(todo_files[gpu_id]), str(shard_paths[gpu_id])],
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        procs.append(p)

    t0 = time.time()
    last_push = time.time()
    dev_val_ids = set(splits["dev_ids"] + splits["val_ids"])
    dev_val_pushed = dev_val_ids.issubset(done.keys())

    def count_lines(p):
        if not p.exists():
            return 0
        with open(p, "r", encoding="utf-8") as f:
            return sum(1 for line in f if line.strip())

    def merge_and_push(push_msg):
        merged = dict(done)
        for sp in shard_paths:
            if sp.exists():
                with open(sp, "r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            rec = json.loads(line)
                            merged[rec["image_id"]] = rec
        with open(partial_path, "w", encoding="utf-8") as f:
            for rec in merged.values():
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        api.upload_file(
            path_or_fileobj=str(partial_path),
            path_in_repo="phase2/ocr_cache_partial.jsonl",
            repo_id=REPO_ID,
            repo_type="dataset",
            commit_message=push_msg,
        )
        return merged

    while any(p.poll() is None for p in procs):
        time.sleep(20)
        new_done = count_lines(shard_paths[0]) + count_lines(shard_paths[1])
        dt = time.time() - t0
        rate = new_done / max(dt, 1e-6)
        rem_min = max(len(todo) - new_done, 0) / max(rate, 1e-6) / 60
        print(f"[{dt/60:.1f}m] Done: {len(done)+new_done}/{len(tasks)} | {rate:.2f} img/s | ETA: {rem_min:.1f} min")

        if time.time() - last_push >= 240:
            merged = merge_and_push(f"Phase 2B: checkpoint ({len(done)+new_done}/{len(tasks)})")
            print(f"  -> Checkpoint pushed ({len(merged)}/{len(tasks)})")
            if not dev_val_pushed and dev_val_ids.issubset(merged.keys()):
                dv_path = ART_DIR / "ocr_cache_dev_val_v1.json"
                dv_dict = {str(i): merged[i] for i in splits["dev_ids"] + splits["val_ids"]}
                dv_path.write_text(json.dumps(dv_dict, ensure_ascii=False), encoding="utf-8")
                api.upload_file(
                    path_or_fileobj=str(dv_path),
                    path_in_repo="phase2/ocr_cache_dev_val_v1.json",
                    repo_id=REPO_ID,
                    repo_type="dataset",
                    commit_message="Phase 2B: dev+val RapidOCR cache",
                )
                dev_val_pushed = True
                print("  -> Uploaded phase2/ocr_cache_dev_val_v1.json!")
            last_push = time.time()

    # Check exit codes
    for gpu_id, p in enumerate(procs):
        if p.returncode != 0:
            print(f"Worker GPU {gpu_id} error (code={p.returncode}):\n{p.stderr.read()}")
            raise RuntimeError(f"Worker GPU {gpu_id} failed")

    # Final merge
    merged = merge_and_push(f"Phase 2B: final partial ({len(tasks)}/{len(tasks)})")
    print(f"All OCR finished! Total records: {len(merged)}")

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

ART_DIR = Path("/kaggle/working/artifacts")
partial_path = ART_DIR / "ocr_cache_partial.jsonl"
splits = json.loads((ART_DIR / "splits_v1.json").read_text())

final_dict = {}
with open(partial_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            final_dict[str(rec["image_id"])] = rec

assert len(final_dict) == 31181, f"Expected 31181 records, got {len(final_dict)}"

final_path = ART_DIR / "ocr_cache_v1.json"
final_path.write_text(json.dumps(final_dict, ensure_ascii=False), encoding="utf-8")

dv_path = ART_DIR / "ocr_cache_dev_val_v1.json"
dv_dict = {str(i): final_dict[str(i)] for i in splits["dev_ids"] + splits["val_ids"]}
dv_path.write_text(json.dumps(dv_dict, ensure_ascii=False), encoding="utf-8")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

for p in [dv_path, final_path]:
    api.upload_file(
        path_or_fileobj=str(p),
        path_in_repo=f"phase2/{p.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Phase 2B: upload {p.name}",
    )
    print(f"Uploaded phase2/{p.name} ({p.stat().st_size/1e6:.2f} MB)")

print("\n=== OCR CACHE AUDIT ===")
for sp in ["dev", "val", "train"]:
    recs = [r for r in final_dict.values() if r["split"] == sp]
    for thr in [0.3, 0.5, 0.7]:
        all_hit = sum(1 for r in recs if any(d["conf"] >= thr for d in r["dets"]))
        gt_txt = [r for r in recs if r["text_detected_gt"]]
        gt_notxt = [r for r in recs if not r["text_detected_gt"]]
        tp = sum(1 for r in gt_txt if any(d["conf"] >= thr for d in r["dets"]))
        fp = sum(1 for r in gt_notxt if any(d["conf"] >= thr for d in r["dets"]))
        print(f"  [{sp}] conf>={thr}: overall={all_hit}/{len(recs)} ({100*all_hit/len(recs):.1f}%) | "
              f"recall_on_text_detected={100*tp/len(gt_txt):.1f}% | rate_on_no_text={100*fp/len(gt_notxt):.1f}%")

Reusable Evaluation Suite

In [ ]:
import os, json, time, sys, urllib.request, zipfile
from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import pycocoevalcap.spice.spice as spice_mod

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. Try fetching Stanford CoreNLP 3.6.0 from HTTPS mirror for SPICE (non-fatal if blocked/incompatible)
spice_dir = Path(spice_mod.__file__).parent
lib_dir = spice_dir / "lib"
jar_path = lib_dir / "stanford-corenlp-3.6.0.jar"
if not jar_path.exists():
    try:
        print("Fetching CoreNLP HTTPS mirror for SPICE ...")
        tmp_zip = Path("/kaggle/tmp/corenlp.zip")
        req = urllib.request.Request(
            "https://downloads.cs.stanford.edu/nlp/software/stanford-corenlp-full-2015-12-09.zip",
            headers={"User-Agent": "Mozilla/5.0"},
        )
        with urllib.request.urlopen(req, timeout=25) as resp, open(tmp_zip, "wb") as out:
            out.write(resp.read())
        with zipfile.ZipFile(tmp_zip, "r") as zf:
            for member in zf.namelist():
                if member.endswith("stanford-corenlp-3.6.0.jar") or member.endswith("stanford-corenlp-3.6.0-models.jar"):
                    lib_dir.mkdir(parents=True, exist_ok=True)
                    (lib_dir / Path(member).name).write_bytes(zf.read(member))
        tmp_zip.unlink(missing_ok=True)
        print("CoreNLP jars extracted:", jar_path.exists())
    except Exception as e:
        print(f"CoreNLP HTTPS fetch skipped ({type(e).__name__}: {e})")

# 2. Write final unified eval_suite.py
CODE_DIR = Path("/kaggle/working/src")
CODE_DIR.mkdir(parents=True, exist_ok=True)

eval_code = r'''
import re, json
from collections import defaultdict
import numpy as np
import torch
from PIL import Image, ImageOps
import spacy
from nltk.corpus import wordnet as wn
import open_clip

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

_NLP = None
def _get_nlp():
    global _NLP
    if _NLP is None:
        _NLP = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    return _NLP

GENERIC_FRAME_NOUNS = {"image", "photo", "picture", "photograph"}
UNCLEAR_PATTERN = re.compile(
    r"(too blurry|too dark|too bright|out of frame|cut off|cannot be identified|"
    r"unable to (?:tell|identify|read|describe|recognize)|quality issues|too severe|"
    r"too unclear|hard to see|cannot make out|can't make out|can't tell|cannot tell)",
    re.IGNORECASE,
)

def format_ocr_text(dets, conf_thr=0.50, max_words=16):
    kept = [d for d in dets if float(d.get("conf", 0.0)) >= conf_thr and d.get("text", "").strip()]
    if not kept:
        return "none"
    kept.sort(key=lambda d: (round(float(d.get("y", 0.0)) / 25.0), float(d.get("x", 0.0))))
    words = " ".join(d["text"].strip() for d in kept).split()
    return " ".join(words[:max_words]) if words else "none"

def build_prompt(ocr_str):
    return f"Visible text: {ocr_str}. Alt-text:"

def load_ground_truth(ann_json_path, target_ids, ocr_cache):
    with open(ann_json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    anns_by_img = defaultdict(list)
    for a in raw["annotations"]:
        anns_by_img[a["image_id"]].append(a)
    target_set = set(target_ids)
    gt = {}
    for im in raw["images"]:
        iid = im["id"]
        if iid not in target_set:
            continue
        im_anns = anns_by_img[iid]
        n_pre = sum(1 for a in im_anns if a.get("is_precanned", False))
        valid_refs = [
            re.sub(r"\s+", " ", a["caption"]).strip()
            for a in im_anns
            if not a.get("is_precanned", False) and not a.get("is_rejected", False) and a["caption"].strip()
        ]
        ocr_rec = ocr_cache.get(str(iid), {})
        ocr_str = format_ocr_text(ocr_rec.get("dets", []))
        gt[iid] = {
            "image_id": iid,
            "file_name": im["file_name"],
            "text_detected": bool(im.get("text_detected", False)),
            "n_precanned": n_pre,
            "is_unclear_gt": n_pre >= 3,
            "is_strict_clear": n_pre == 0,
            "valid_refs": valid_refs,
            "ocr_str": ocr_str,
            "ocr_all_text": " ".join(d["text"] for d in ocr_rec.get("dets", []) if float(d.get("conf", 0)) >= 0.5),
        }
    return gt

def _clean_for_coco(s):
    s = s.encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"\s+", " ", s.lower()).strip()
    return s if re.search(r"[a-z0-9]", s) else "none"

def _extract_nouns(text):
    nlp = _get_nlp()
    doc = nlp(text.lower())
    nouns = []
    for tok in doc:
        if tok.pos_ in ("NOUN", "PROPN") and tok.is_alpha and len(tok.text) > 1:
            lemma = tok.lemma_.lower()
            if lemma not in GENERIC_FRAME_NOUNS:
                nouns.append(lemma)
    return nouns

def _expand_wordnet(nouns):
    expanded = set(nouns)
    for n in nouns:
        for syn in wn.synsets(n, pos=wn.NOUN):
            for lem in syn.lemmas():
                expanded.add(lem.name().lower().replace("_", " "))
    return expanded

def compute_coco_metrics(gts_raw, res_raw, run_spice=True):
    if not gts_raw:
        return {}
    tokenizer = PTBTokenizer()
    gts = tokenizer.tokenize(gts_raw)
    res = tokenizer.tokenize(res_raw)

    scorer_specs = [
        (Bleu, (4,), ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]),
        (Meteor, (), "METEOR"),
        (Rouge, (), "ROUGE-L"),
        (Cider, (), "CIDEr"),
    ]
    if run_spice:
        scorer_specs.append((Spice, (), "SPICE"))
    out = {}
    for cls, args, method in scorer_specs:
        try:
            scorer = cls(*args)
            score, _ = scorer.compute_score(gts, res)
            if isinstance(method, list):
                out["BLEU-4"] = round(float(score[-1]) * 100, 2)
            else:
                out[method] = round(float(score) * 100, 2)
        except Exception as e:
            print(f"  [Warning] Scorer {method} failed: {type(e).__name__}: {e}")
    return out

@torch.no_grad()
def compute_clip_scores(preds_by_id, gt_by_id, img_dir, eval_ids, batch_size=64, device="cuda:0"):
    if not eval_ids:
        return {}, {}
    clip_model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai", quick_gelu=True, device=device
    )
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    clip_model.eval()

    clip_scores, ref_clip_scores = {}, {}
    for i in range(0, len(eval_ids), batch_size):
        b_ids = eval_ids[i:i + batch_size]
        imgs, cand_texts, ref_lists = [], [], []
        for iid in b_ids:
            p = ImageOps.exif_transpose(Image.open(img_dir / gt_by_id[iid]["file_name"])).convert("RGB")
            imgs.append(preprocess(p))
            cand_texts.append("A photo depicts " + preds_by_id[iid])
            ref_lists.append(["A photo depicts " + r for r in gt_by_id[iid]["valid_refs"]])

        img_t = torch.stack(imgs).to(device)
        cand_tok = tokenizer(cand_texts).to(device)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            img_feat = clip_model.encode_image(img_t)
            cand_feat = clip_model.encode_text(cand_tok)
        img_feat = torch.nn.functional.normalize(img_feat.float(), dim=-1)
        cand_feat = torch.nn.functional.normalize(cand_feat.float(), dim=-1)

        cs = 2.5 * (img_feat * cand_feat).sum(dim=-1).clamp(min=0.0)

        flat_refs = [r for rl in ref_lists for r in rl]
        ref_tok = tokenizer(flat_refs).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            flat_ref_feat = clip_model.encode_text(ref_tok)
        flat_ref_feat = torch.nn.functional.normalize(flat_ref_feat.float(), dim=-1)

        idx_ptr = 0
        for j, iid in enumerate(b_ids):
            n_r = len(ref_lists[j])
            r_feats = flat_ref_feat[idx_ptr:idx_ptr + n_r]
            idx_ptr += n_r
            max_cos_cr = float((cand_feat[j:j+1] @ r_feats.T).max().clamp(min=0.0).item())
            c_val = float(cs[j].item())
            clip_scores[iid] = c_val
            ref_clip_scores[iid] = (2.0 * c_val * max_cos_cr / (c_val + max_cos_cr)) if (c_val > 0 and max_cos_cr > 0) else 0.0

    del clip_model
    torch.cuda.empty_cache()
    return clip_scores, ref_clip_scores

def evaluate_predictions(preds_by_id, gt_by_id, img_dir, run_spice=True, run_clip=True, device="cuda:0"):
    all_ids = [iid for iid in gt_by_id if iid in preds_by_id]
    valid_ids = [iid for iid in all_ids if len(gt_by_id[iid]["valid_refs"]) > 0]

    clip_s, ref_clip_s = ({}, {})
    if run_clip:
        clip_s, ref_clip_s = compute_clip_scores(preds_by_id, gt_by_id, img_dir, valid_ids, device=device)

    noun_unsup_counts, cap_has_unsup = {}, {}
    quoted_total, quoted_faithful = 0, 0
    ocr_hit_imgs, ocr_nonempty_imgs = 0, 0

    for iid in valid_ids:
        pred = preds_by_id[iid]
        gt = gt_by_id[iid]

        if not UNCLEAR_PATTERN.search(pred):
            pred_nouns = _extract_nouns(pred)
            ref_nouns = set()
            for r in gt["valid_refs"]:
                ref_nouns.update(_extract_nouns(r))
            ref_expanded = _expand_wordnet(ref_nouns)
            unsup = [n for n in pred_nouns if n not in ref_expanded]
            noun_unsup_counts[iid] = (len(unsup), len(pred_nouns))
            cap_has_unsup[iid] = 1 if len(unsup) > 0 else 0

        quotes = re.findall(r'"([^"]{2,})"|\'([^\']{2,})\'', pred)
        q_strs = [q[0] or q[1] for q in quotes]
        haystack = (gt["ocr_all_text"] + " " + " ".join(gt["valid_refs"])).lower()
        for q in q_strs:
            quoted_total += 1
            qn = re.sub(r"[^a-z0-9 ]", "", q.lower()).strip()
            hn = re.sub(r"[^a-z0-9 ]", "", haystack)
            if qn and qn in hn:
                quoted_faithful += 1

        if gt["text_detected"] and gt["ocr_str"] != "none":
            ocr_toks = re.findall(r"[a-z0-9]{4,}", gt["ocr_str"].lower())
            if ocr_toks:
                ocr_nonempty_imgs += 1
                pred_low = pred.lower()
                if any(t in pred_low for t in ocr_toks):
                    ocr_hit_imgs += 1

    def score_slice(slice_ids):
        gts_raw = {iid: [{"caption": _clean_for_coco(r)} for r in gt_by_id[iid]["valid_refs"]] for iid in slice_ids}
        res_raw = {iid: [{"caption": _clean_for_coco(preds_by_id[iid])}] for iid in slice_ids}
        out = {"n_images": len(slice_ids)}
        out.update(compute_coco_metrics(gts_raw, res_raw, run_spice=run_spice))
        if run_clip and slice_ids:
            out["CLIPScore"] = round(float(np.mean([clip_s[i] for i in slice_ids])), 4)
            out["RefCLIPScore"] = round(float(np.mean([ref_clip_s[i] for i in slice_ids])), 4)
        s_nouns = [noun_unsup_counts[i] for i in slice_ids if i in noun_unsup_counts]
        if s_nouns:
            tot_u = sum(u for u, _ in s_nouns)
            tot_n = sum(n for _, n in s_nouns)
            s_caps = [cap_has_unsup[i] for i in slice_ids if i in cap_has_unsup]
            out["unsup_noun_pct"] = round(100.0 * tot_u / max(tot_n, 1), 2)
            out["cap_with_unsup_pct"] = round(100.0 * sum(s_caps) / len(s_caps), 2)
        out["avg_words"] = round(float(np.mean([len(preds_by_id[i].split()) for i in slice_ids])), 2)
        return out

    results = {
        "overall": score_slice(valid_ids),
        "text_detected": score_slice([i for i in valid_ids if gt_by_id[i]["text_detected"]]),
        "no_text": score_slice([i for i in valid_ids if not gt_by_id[i]["text_detected"]]),
    }

    unclear_ids = [i for i in all_ids if gt_by_id[i]["is_unclear_gt"]]
    clear_ids = [i for i in all_ids if gt_by_id[i]["is_strict_clear"]]
    unclear_hits = sum(1 for i in unclear_ids if UNCLEAR_PATTERN.search(preds_by_id[i]))
    false_unclear = sum(1 for i in clear_ids if UNCLEAR_PATTERN.search(preds_by_id[i]))

    results["behavior"] = {
        "n_total": len(all_ids),
        "n_unclear_gt": len(unclear_ids),
        "unclear_recall_pct": round(100.0 * unclear_hits / max(len(unclear_ids), 1), 2),
        "n_strict_clear": len(clear_ids),
        "false_unclear_pct": round(100.0 * false_unclear / max(len(clear_ids), 1), 2),
        "quoted_count": quoted_total,
        "quoted_faithful_pct": round(100.0 * quoted_faithful / quoted_total, 2) if quoted_total else None,
        "ocr_token_adoption_pct": round(100.0 * ocr_hit_imgs / max(ocr_nonempty_imgs, 1), 2),
    }
    return results
'''

(CODE_DIR / "eval_suite.py").write_text(eval_code.strip() + "\n", encoding="utf-8")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"
api.upload_file(
    path_or_fileobj=str(CODE_DIR / "eval_suite.py"),
    path_in_repo="code/eval_suite.py",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Phase 2C: unified eval_suite.py",
)
print("Saved & uploaded code/eval_suite.py")

# 3. 32-image sanity check
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import importlib, eval_suite
importlib.reload(eval_suite)

import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
gt_dev = eval_suite.load_ground_truth(
    DATA_DIR / "annotations/annotations/train.json", splits["dev_ids"], ocr_dv
)

dev_32 = splits["dev_ids"][:32]
gt_32 = {k: gt_dev[k] for k in dev_32}
train_img_dir = DATA_DIR / "train_images/train"

if "model" not in globals():
    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
    model.eval()
processor.tokenizer.padding_side = "left"

if "preds_empty" not in globals():
    preds_empty = {}
    t0 = time.time()
    for i in range(0, len(dev_32), 8):
        b_ids = dev_32[i:i + 8]
        pils = [ImageOps.exif_transpose(Image.open(train_img_dir / gt_dev[iid]["file_name"])).convert("RGB") for iid in b_ids]
        inp = processor(images=pils, return_tensors="pt").to("cuda:0", torch.float16)
        with torch.no_grad():
            out = model.generate(**inp, num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3)
        texts = processor.tokenizer.batch_decode(out, skip_special_tokens=True)
        for iid, t in zip(b_ids, texts):
            preds_empty[iid] = t.strip()
    print(f"Generated 32 empty predictions in {time.time()-t0:.1f}s")

metrics_32 = eval_suite.evaluate_predictions(preds_empty, gt_32, train_img_dir, run_spice=True, run_clip=True)
print("\n=== 32-IMAGE EVAL SUITE SANITY CHECK ===")
print(json.dumps(metrics_32, indent=2))

SESSION RESTART CELL

In [ ]:
import os, subprocess, zipfile, shutil
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/kaggle/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

# 1. Install only missing packages (without -U so Kaggle's core versions stay aligned)
!pip install -q peft accelerate pycocoevalcap open-clip-torch spacy nltk gdown
!python -m spacy download en_core_web_sm

import nltk
for pkg in ["wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# 2. Verify Kaggle Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
print("HF_TOKEN & WANDB_API_KEY: FOUND")

# 3. Restore Phase 1 & Phase 2B artifacts from Hugging Face Hub if missing
from huggingface_hub import HfApi, hf_hub_download

ART_DIR = Path("/kaggle/working/artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

for remote_p, local_name in [
    ("phase1/splits_v1.json", "splits_v1.json"),
    ("phase2/ocr_cache_dev_val_v1.json", "ocr_cache_dev_val_v1.json"),
]:
    dst = ART_DIR / local_name
    if not dst.exists():
        dl = hf_hub_download(REPO_ID, remote_p, repo_type="dataset", token=hf_token)
        shutil.copy(dl, dst)
        print(f"Restored {local_name} from HF Hub")
    else:
        print(f"Found local {local_name}")

# 4. Restore VizWiz images & annotations into /kaggle/tmp/vizwiz
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ANN_DIR = DATA_DIR / "annotations"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR.mkdir(parents=True, exist_ok=True)

if not (ANN_DIR / "annotations" / "train.json").exists():
    ann_zip = DATA_DIR / "annotations_dl.zip"
    subprocess.run(
        ["gdown", "1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH", "-O", str(ann_zip)],
        check=True,
    )
    with zipfile.ZipFile(ann_zip, "r") as zf:
        zf.extractall(ANN_DIR)
    ann_zip.unlink(missing_ok=True)
print("Annotations ready:", (ANN_DIR / "annotations" / "train.json").exists())

for name, url in [
    ("train_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip"),
    ("val_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"),
]:
    extract_dir = DATA_DIR / name
    if not extract_dir.exists():
        out_zip = DATA_DIR / f"{name}.zip"
        print(f"Downloading {name} ...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(out_zip), url], check=True)
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(out_zip, "r") as zf:
            zf.extractall(extract_dir)
        out_zip.unlink(missing_ok=True)
print("Images ready:", (DATA_DIR / "train_images" / "train").exists(), (DATA_DIR / "val_images" / "val").exists())

In [ ]:
import os, gc, json, time, subprocess, sys
from pathlib import Path
import torch
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
ART_DIR = Path("/kaggle/working/artifacts")
TMP_DIR = Path("/kaggle/tmp/zs_work")
TMP_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/kaggle/tmp/vizwiz")

# 1. Free GPU 0 memory from earlier cells so Worker 0 has all 15.6 GB
for var in ["model", "processor", "out", "inp", "preds_empty", "preds_ocr"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())

gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", splits["dev_ids"], ocr_dv)
gt_val = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/val.json", splits["val_ids"], ocr_dv)

all_tasks = []
for sp_name, gt_map, img_dir in [
    ("dev", gt_dev, DATA_DIR / "train_images/train"),
    ("val", gt_val, DATA_DIR / "val_images/val"),
]:
    for iid, info in gt_map.items():
        all_tasks.append({
            "image_id": iid,
            "split": sp_name,
            "path": str(img_dir / info["file_name"]),
            "prompt": eval_suite.build_prompt(info["ocr_str"]),
        })

# Sort by prompt length so batched padding overhead is minimal
all_tasks.sort(key=lambda x: len(x["prompt"]))

# 2. Write standalone dual-GPU inference worker
worker_py = TMP_DIR / "zs_worker.py"
worker_py.write_text('''
import sys, json
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_file, out_file, mode, bs = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map={"": 0}
)
model.eval()

with open(out_file, "a", encoding="utf-8") as fout:
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        if mode == "alt_ocr":
            prompts = [x["prompt"] for x in batch]
            inp = processor(images=pils, text=prompts, padding=True, return_tensors="pt").to("cuda:0", torch.float16)
            in_len = inp["input_ids"].shape[1]
        else:
            inp = processor(images=pils, return_tensors="pt").to("cuda:0", torch.float16)
            in_len = 0
        with torch.no_grad():
            out = model.generate(**inp, num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3)
        texts = processor.tokenizer.batch_decode(out[:, in_len:], skip_special_tokens=True)
        for x, t in zip(batch, texts):
            fout.write(json.dumps({"image_id": x["image_id"], "split": x["split"], "pred": t.strip()}, ensure_ascii=False) + "\\n")
        fout.flush()
''', encoding="utf-8")

def run_mode(mode_name, bs=12):
    out_json = ART_DIR / f"preds_zs_{mode_name}_v1.json"
    if not out_json.exists():
        try:
            dl = hf_hub_download(REPO_ID, f"phase2/{out_json.name}", repo_type="dataset", token=hf_token)
            out_json.write_text(Path(dl).read_text(encoding="utf-8"), encoding="utf-8")
        except Exception:
            pass
    if out_json.exists():
        existing = json.loads(out_json.read_text())
        if len(existing.get("dev", {})) == 1000 and len(existing.get("val", {})) == 7750:
            print(f"[{mode_name}] Already complete, skipping.")
            return

    todo_0 = TMP_DIR / f"todo_{mode_name}_0.json"
    todo_1 = TMP_DIR / f"todo_{mode_name}_1.json"
    sh_0 = TMP_DIR / f"shard_{mode_name}_0.jsonl"
    sh_1 = TMP_DIR / f"shard_{mode_name}_1.jsonl"
    for p in [sh_0, sh_1]:
        if p.exists(): p.unlink()

    todo_0.write_text(json.dumps(all_tasks[0::2]), encoding="utf-8")
    todo_1.write_text(json.dumps(all_tasks[1::2]), encoding="utf-8")

    print(f"\n=== Launching 2-GPU Zero-Shot Generation: {mode_name} (8,750 images) ===")
    procs = []
    for gpu_id, tf, sf in [(0, todo_0, sh_0), (1, todo_1, sh_1)]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        procs.append(subprocess.Popen(
            ["python3", str(worker_py), str(tf), str(sf), mode_name, str(bs)],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))

    t0 = time.time()
    def _cnt(p):
        return sum(1 for l in open(p) if l.strip()) if p.exists() else 0

    while any(p.poll() is None for p in procs):
        time.sleep(20)
        n_done = _cnt(sh_0) + _cnt(sh_1)
        dt = time.time() - t0
        rate = n_done / max(dt, 1e-6)
        eta = max(len(all_tasks) - n_done, 0) / max(rate, 1e-6) / 60
        print(f"  [{mode_name} {dt/60:.1f}m] {n_done}/{len(all_tasks)} ({rate:.2f} img/s, ETA: {eta:.1f} min)")

    for gpu_id, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gpu_id} failed:\n{p.stderr.read()}")

    merged = {"dev": {}, "val": {}}
    for sf in [sh_0, sh_1]:
        with open(sf, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    merged[r["split"]][str(r["image_id"])] = r["pred"]

    assert len(merged["dev"]) == 1000 and len(merged["val"]) == 7750
    out_json.write_text(json.dumps(merged, ensure_ascii=False), encoding="utf-8")
    api.upload_file(
        path_or_fileobj=str(out_json),
        path_in_repo=f"phase2/{out_json.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Phase 2C: {out_json.name}",
    )
    print(f"Uploaded phase2/{out_json.name} in {(time.time()-t0)/60:.1f} min")

run_mode("empty", bs=12)
run_mode("alt_ocr", bs=12)

In [ ]:
import os, json, sys
from pathlib import Path
import wandb
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

# Allow Java 21 reflection for SPICE FST serialization
os.environ["JDK_JAVA_OPTIONS"] = (
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED "
    "--add-opens java.base/java.nio=ALL-UNNAMED"
)

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
REPO_ID = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import importlib, eval_suite
importlib.reload(eval_suite)

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())

gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", splits["dev_ids"], ocr_dv)
gt_val = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/val.json", splits["val_ids"], ocr_dv)

all_results = {}
for mode in ["empty", "alt_ocr"]:
    preds_raw = json.loads((ART_DIR / f"preds_zs_{mode}_v1.json").read_text())
    preds_dev = {int(k): v for k, v in preds_raw["dev"].items()}
    preds_val = {int(k): v for k, v in preds_raw["val"].items()}

    print(f"\nEvaluating zs_{mode} on dev (1,000) ...")
    res_dev = eval_suite.evaluate_predictions(preds_dev, gt_dev, DATA_DIR / "train_images/train", run_spice=True, run_clip=True)
    print(f"Evaluating zs_{mode} on val (7,750) ...")
    res_val = eval_suite.evaluate_predictions(preds_val, gt_val, DATA_DIR / "val_images/val", run_spice=True, run_clip=True)

    all_results[f"zs_{mode}"] = {"dev": res_dev, "val": res_val}

    run = wandb.init(
        project="vizwiz-alttext-blip2",
        name=f"phase2-zeroshot-{mode}",
        config={"phase": "2C", "model": "Salesforce/blip2-opt-2.7b", "prompt_mode": mode, "num_beams": 5, "max_new_tokens": 60},
        reinit=True,
    )
    flat_metrics = {}
    for sp_name, sp_res in [("dev", res_dev), ("val", res_val)]:
        for grp, d in sp_res.items():
            for mk, mv in d.items():
                if mv is not None:
                    flat_metrics[f"{sp_name}/{grp}/{mk}"] = mv
    wandb.log(flat_metrics)

    table = wandb.Table(columns=["image_id", "text_detected", "unclear_gt", "ocr_str", "ref_0", "prediction"])
    for iid in splits["dev_ids"][:25]:
        g = gt_dev[iid]
        table.add_data(iid, g["text_detected"], g["is_unclear_gt"], g["ocr_str"], g["valid_refs"][0] if g["valid_refs"] else "<precanned>", preds_dev[iid])
    wandb.log({"dev_samples": table})
    run.finish()

res_path = ART_DIR / "baseline_results_v1.json"
res_path.write_text(json.dumps(all_results, indent=2), encoding="utf-8")
api.upload_file(
    path_or_fileobj=str(res_path),
    path_in_repo="phase2/baseline_results_v1.json",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Phase 2C: zero-shot baseline evaluation results",
)

print("\n=== PHASE 2C BASELINE RESULTS SUMMARY ===")
print(json.dumps(all_results, indent=2))

Cache Frozen Q-Former Features

In [ ]:
import os, gc, json, time, subprocess, shutil
from pathlib import Path
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
FEAT_DIR.mkdir(parents=True, exist_ok=True)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
REPO_ID = "B4BUK4/vizwiz-alttext-artifacts"

ocr_full_path = ART_DIR / "ocr_cache_v1.json"
if not ocr_full_path.exists():
    dl = hf_hub_download(REPO_ID, "phase2/ocr_cache_v1.json", repo_type="dataset", token=hf_token)
    shutil.copy(dl, ocr_full_path)
    print("Downloaded phase2/ocr_cache_v1.json")

splits = json.loads((ART_DIR / "splits_v1.json").read_text())

if "model" not in globals():
    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
    model.eval()

with open(DATA_DIR / "annotations/annotations/train.json") as f:
    train_raw = json.load(f)
img_meta = {im["id"]: im for im in train_raw["images"]}
sample_im = ImageOps.exif_transpose(
    Image.open(DATA_DIR / "train_images/train" / img_meta[splits["dev_ids"][0]]["file_name"])
).convert("RGB")

prompt = "Visible text: none. Alt-text:"
target = " A clear sample caption for testing."
p_enc = processor(images=sample_im, text=prompt, return_tensors="pt")
f_enc = processor(images=sample_im, text=prompt + target + processor.tokenizer.eos_token, return_tensors="pt").to("cuda:0")
f_enc["pixel_values"] = f_enc["pixel_values"].to(torch.float16)

labels = f_enc["input_ids"].clone()
labels[:, :p_enc["input_ids"].shape[1]] = -100

with torch.no_grad():
    loss_full = model(**f_enc, labels=labels).loss.item()

    v_out = model.vision_model(pixel_values=f_enc["pixel_values"])[0]
    v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
    q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
    q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
    proj_dtype = model.language_projection.weight.dtype
    vis_feat = model.language_projection(q_out.to(proj_dtype)).to(torch.float16)

    inputs_embeds = model.language_model.get_input_embeddings()(f_enc["input_ids"])
    img_mask = (f_enc["input_ids"] == model.config.image_token_id)
    assert img_mask.sum().item() == 32, f"Expected 32 image tokens, got {img_mask.sum().item()}"
    inputs_embeds[img_mask] = vis_feat.reshape(-1, vis_feat.shape[-1]).to(inputs_embeds.dtype)
    loss_cached = model.language_model(
        inputs_embeds=inputs_embeds, attention_mask=f_enc["attention_mask"], labels=labels
    ).loss.item()

diff = abs(loss_full - loss_cached)
print(f"Equivalence check -> full_loss={loss_full:.6f}, cached_loss={loss_cached:.6f}, diff={diff:.6e}")
assert diff < 1e-3, f"Loss mismatch too large: {diff}"

del model, f_enc, p_enc, v_out, q_out, vis_feat, inputs_embeds
gc.collect(); torch.cuda.empty_cache()

train_feat_file = FEAT_DIR / "vis_feats_train_v1.pt"
dev_feat_file = FEAT_DIR / "vis_feats_dev_v1.pt"

if not (train_feat_file.exists() and dev_feat_file.exists()):
    extract_py = FEAT_DIR / "extract_feats.py"
    extract_py.write_text('''
import sys, json
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_json, out_pt, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_json).read())

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
proj_dtype = model.language_projection.weight.dtype

feats = {}
with torch.no_grad():
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        px = processor(images=pils, return_tensors="pt")["pixel_values"].to("cuda:0", torch.float16)
        v_out = model.vision_model(pixel_values=px)[0]
        v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
        q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
        q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
        proj = model.language_projection(q_out.to(proj_dtype)).to(torch.float16).cpu()
        for j, x in enumerate(batch):
            feats[x["image_id"]] = proj[j]
torch.save(feats, out_pt)
''', encoding="utf-8")

    train_dir = DATA_DIR / "train_images/train"
    all_items = [
        {"image_id": iid, "path": str(train_dir / img_meta[iid]["file_name"])}
        for iid in (splits["train_ids"] + splits["dev_ids"])
    ]
    t_files = [FEAT_DIR / "todo_0.json", FEAT_DIR / "todo_1.json"]
    s_files = [FEAT_DIR / "shard_0.pt", FEAT_DIR / "shard_1.pt"]
    t_files[0].write_text(json.dumps(all_items[0::2]))
    t_files[1].write_text(json.dumps(all_items[1::2]))

    t0 = time.time()
    procs = []
    for gpu_id in [0, 1]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        procs.append(subprocess.Popen(
            ["python3", str(extract_py), str(t_files[gpu_id]), str(s_files[gpu_id]), "32"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))
    while any(p.poll() is None for p in procs):
        time.sleep(15)
        print(f"  Extracting frozen Q-Former features ... ({(time.time()-t0)/60:.1f} min elapsed)")
    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"Extractor GPU {gid} failed:\n{p.stderr.read()}")

    merged = {}
    for sf in s_files:
        merged.update(torch.load(sf, weights_only=True))
        sf.unlink()

    train_feats = {iid: merged[iid] for iid in splits["train_ids"]}
    dev_feats = {iid: merged[iid] for iid in splits["dev_ids"]}
    torch.save(train_feats, train_feat_file)
    torch.save(dev_feats, dev_feat_file)
    print(f"Saved {len(train_feats)} train + {len(dev_feats)} dev embeddings in {(time.time()-t0)/60:.1f} min "
          f"({train_feat_file.stat().st_size/1e9:.2f} GB)")
else:
    print("Cached features already exist.")

200-Step Track A LoRA Smoke Test (Loss Curve, VRAM & Step-0 vs Step-200 Samples)

In [ ]:
import os, re, json, time, random, sys
from collections import defaultdict
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

anns_by_img = defaultdict(list)
for a in train_raw["annotations"]:
    if not a.get("is_precanned", False) and not a.get("is_rejected", False):
        cap = re.sub(r"\s+", " ", a["caption"]).strip()
        if cap:
            anns_by_img[a["image_id"]].append(cap)

train_ids_valid = [iid for iid in splits["train_ids"] if len(anns_by_img[iid]) > 0]
print(f"Track A train images with >=1 valid caption: {len(train_ids_valid)} / {len(splits['train_ids'])}")

train_feats = torch.load(FEAT_DIR / "vis_feats_train_v1.pt", weights_only=True)
dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer
IMAGE_TOKEN_ID = 50265  # verify dynamically from config below

# Load ONLY the OPT language model + config to save ~3 GB VRAM!
full_model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
IMAGE_TOKEN_ID = full_model.config.image_token_id
opt_model = full_model.language_model
del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens
del full_model
import gc; gc.collect(); torch.cuda.empty_cache()

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], bias="none",
)
opt_model = get_peft_model(opt_model, lora_cfg)
opt_model.print_trainable_parameters()
opt_model.gradient_checkpointing_enable()
opt_model.enable_input_require_grads()

# Dataset: 1 random valid caption per image, returns vis_feat + input_ids + labels
class TrackADataset(Dataset):
    def __init__(self, ids, feats_dict, max_caption_tokens=64):
        self.ids = ids
        self.feats = feats_dict
        self.max_cap_tok = max_caption_tokens
        self.img_prefix_ids = [IMAGE_TOKEN_ID] * 32

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        iid = self.ids[idx]
        ocr_str = eval_suite.format_ocr_text(ocr_all.get(str(iid), {}).get("dets", []))
        prompt = eval_suite.build_prompt(ocr_str)
        cap = random.choice(anns_by_img[iid])

        # OPT tokenizer adds </s> at index 0
        p_ids = tokenizer.encode(prompt, add_special_tokens=True)
        c_ids = tokenizer.encode(" " + cap, add_special_tokens=False)[:self.max_cap_tok - 1] + [tokenizer.eos_token_id]

        input_ids = self.img_prefix_ids + p_ids + c_ids
        labels = ([-100] * (32 + len(p_ids))) + c_ids
        return {
            "vis_feat": self.feats[iid],
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = tokenizer.pad_token_id
    B = len(batch)
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attn_mask = torch.zeros((B, max_len), dtype=torch.long)
    labels = torch.full((B, max_len), -100, dtype=torch.long)
    vis_feats = torch.stack([x["vis_feat"] for x in batch])
    for i, x in enumerate(batch):
        L = x["input_ids"].shape[0]
        input_ids[i, :L] = x["input_ids"]
        attn_mask[i, :L] = 1
        labels[i, :L] = x["labels"]
    return {"vis_feats": vis_feats, "input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

def run_step_forward(batch):
    vis_feats = batch["vis_feats"].to("cuda:0", torch.float16)
    input_ids = batch["input_ids"].to("cuda:0")
    attn_mask = batch["attention_mask"].to("cuda:0")
    labels = batch["labels"].to("cuda:0")

    embeds = opt_model.get_input_embeddings()(input_ids).clone()
    embeds[:, :32, :] = vis_feats.to(embeds.dtype)
    return opt_model(inputs_embeds=embeds, attention_mask=attn_mask, labels=labels).loss

@torch.no_grad()
def preview_5_dev(tag):
    opt_model.eval()
    tokenizer.padding_side = "left"
    sample_ids = splits["dev_ids"][:5]
    print(f"\n--- Sample Predictions ({tag}) ---")
    for iid in sample_ids:
        ocr_str = eval_suite.format_ocr_text(ocr_all.get(str(iid), {}).get("dets", []))
        p_ids = tokenizer.encode(eval_suite.build_prompt(ocr_str), add_special_tokens=True)
        in_ids = torch.tensor([[IMAGE_TOKEN_ID]*32 + p_ids], device="cuda:0")
        attn = torch.ones_like(in_ids)
        embeds = opt_model.get_input_embeddings()(in_ids).clone()
        embeds[:, :32, :] = dev_feats[iid].unsqueeze(0).to("cuda:0", embeds.dtype)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = opt_model.generate(
                inputs_embeds=embeds, attention_mask=attn,
                num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
            )
        gen = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        ref = anns_by_img[iid][0] if anns_by_img[iid] else "<precanned>"
        print(f"  ID {iid} | OCR={ocr_str[:40]} | Ref={ref[:55]} | Pred={gen}")
    tokenizer.padding_side = "right"
    opt_model.train()

preview_5_dev("step=0")

# 200 optimizer steps with micro_batch=8 x grad_accum=4 = effective batch 32
ds = TrackADataset(train_ids_valid, train_feats)
loader = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2)

optimizer = torch.optim.AdamW([p for p in opt_model.parameters() if p.requires_grad], lr=1.5e-4, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=20, num_training_steps=200)
scaler = torch.amp.GradScaler("cuda")

torch.cuda.reset_peak_memory_stats(0)
opt_model.train()
step, micro, running_loss = 0, 0, 0.0
t0 = time.time()
optimizer.zero_grad(set_to_none=True)

while step < 200:
    for batch in loader:
        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss = run_step_forward(batch) / 4.0
        scaler.scale(loss).backward()
        running_loss += loss.item()
        micro += 1
        if micro % 4 == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in opt_model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 25 == 0:
                print(f"Step {step:3d}/200 | loss={running_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e} | "
                      f"VRAM={torch.cuda.max_memory_allocated(0)/1e9:.2f} GB | {time.time()-t0:.1f}s")
            running_loss = 0.0
            if step >= 200:
                break

preview_5_dev("step=200")

In [ ]:
!pip uninstall -y torchao

# 2-second check that PEFT LoRA wraps cleanly now
import torch.nn as nn
from peft import LoraConfig, get_peft_model

class _Dummy(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(16, 16)
        self.v_proj = nn.Linear(16, 16)

m = get_peft_model(_Dummy(), LoraConfig(r=4, lora_alpha=8, target_modules=["q_proj", "v_proj"]))
m.print_trainable_parameters()
print("PEFT LoRA check passed—now re-run Cell 1 and Cell 2!")

In [ ]:
import os, subprocess
from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
assert (FEAT_DIR / "vis_feats_train_v1.pt").exists(), "Missing cached train features!"
assert (FEAT_DIR / "vis_feats_dev_v1.pt").exists(), "Missing cached dev features!"

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
wandb_key = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-trackA"
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, private=False)

train_script = Path("/kaggle/tmp/train_trackA_ddp.py")
train_script.write_text(r'''
import os, re, sys, json, time, math, random, gc
from collections import defaultdict
from pathlib import Path

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import Blip2Processor, Blip2ForConditionalGeneration, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
import wandb
from huggingface_hub import HfApi

sys.path.insert(0, "/kaggle/working/src")
import eval_suite

EPOCHS = 4
PER_GPU_BS = 4
GRAD_ACCUM = 4          # 4 * 2 GPUs * 4 accum = 32 effective batch size
LR = 1.5e-4
WARMUP_RATIO = 0.03
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def main():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    seed_everything(SEED + rank)

    DATA_DIR = Path("/kaggle/tmp/vizwiz")
    ART_DIR = Path("/kaggle/working/artifacts")
    FEAT_DIR = Path("/kaggle/tmp/vis_feats")
    CKPT_DIR = Path("/kaggle/working/checkpoints/trackA")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
    train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

    anns_by_img = defaultdict(list)
    for a in train_raw["annotations"]:
        if not a.get("is_precanned", False) and not a.get("is_rejected", False):
            cap = re.sub(r"\s+", " ", a["caption"]).strip()
            if cap:
                anns_by_img[a["image_id"]].append(cap)
    del train_raw

    train_ids = [iid for iid in splits["train_ids"] if len(anns_by_img[iid]) > 0]
    dev_ids = splits["dev_ids"]
    gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", dev_ids, ocr_all)

    train_feats = torch.load(FEAT_DIR / "vis_feats_train_v1.pt", weights_only=True)
    dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)

    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    tokenizer = processor.tokenizer

    full_model = Blip2ForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    IMAGE_TOKEN_ID = full_model.config.image_token_id
    opt_model = full_model.language_model
    del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens
    del full_model
    gc.collect()
    opt_model = opt_model.to(device)

    opt_model.config.use_cache = False
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none",
    )
    opt_model = get_peft_model(opt_model, lora_cfg)
    if rank == 0:
        opt_model.print_trainable_parameters()

    ddp_model = DDP(opt_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    class TrackADataset(Dataset):
        def __init__(self, ids, feats_dict, is_train=True, max_cap_tok=64):
            self.ids = ids
            self.feats = feats_dict
            self.is_train = is_train
            self.max_cap_tok = max_cap_tok
            self.img_prefix = [IMAGE_TOKEN_ID] * 32

        def __len__(self):
            return len(self.ids)

        def __getitem__(self, idx):
            iid = self.ids[idx]
            ocr_str = eval_suite.format_ocr_text(ocr_all.get(str(iid), {}).get("dets", []))
            prompt = eval_suite.build_prompt(ocr_str)
            cap = random.choice(anns_by_img[iid]) if self.is_train else anns_by_img[iid][0]

            p_ids = tokenizer.encode(prompt, add_special_tokens=True)
            c_ids = tokenizer.encode(" " + cap, add_special_tokens=False)[:self.max_cap_tok - 1] + [tokenizer.eos_token_id]
            input_ids = self.img_prefix + p_ids + c_ids
            labels = ([-100] * (32 + len(p_ids))) + c_ids
            return {
                "vis_feat": self.feats[iid],
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

    def collate_fn(batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)
        pad_id = tokenizer.pad_token_id
        B = len(batch)
        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
        attn_mask = torch.zeros((B, max_len), dtype=torch.long)
        labels = torch.full((B, max_len), -100, dtype=torch.long)
        vis_feats = torch.stack([x["vis_feat"] for x in batch])
        for i, x in enumerate(batch):
            L = x["input_ids"].shape[0]
            input_ids[i, :L] = x["input_ids"]
            attn_mask[i, :L] = 1
            labels[i, :L] = x["labels"]
        return {"vis_feats": vis_feats, "input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

    train_ds = TrackADataset(train_ids, train_feats, is_train=True)
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED)
    loader = DataLoader(train_ds, batch_size=PER_GPU_BS, sampler=sampler, collate_fn=collate_fn, num_workers=0)

    dev_valid_ids = [iid for iid in dev_ids if len(anns_by_img[iid]) > 0]
    dev_loss_ds = TrackADataset(dev_valid_ids, dev_feats, is_train=False)
    dev_loss_loader = DataLoader(dev_loss_ds, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=0)

    steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = int(total_steps * WARMUP_RATIO)

    trainable_params = [p for p in ddp_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda")

    if rank == 0:
        run = wandb.init(
            project="vizwiz-alttext-blip2",
            name="phase3-trackA-lora-r16",
            config={
                "phase": "3B", "track": "A", "model": model_id,
                "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
                "target_modules": ["q_proj", "v_proj"],
                "epochs": EPOCHS, "per_gpu_bs": PER_GPU_BS, "grad_accum": GRAD_ACCUM,
                "effective_bs": PER_GPU_BS * world_size * GRAD_ACCUM,
                "lr": LR, "warmup_ratio": WARMUP_RATIO, "total_steps": total_steps,
            },
        )
        api = HfApi(token=os.environ["HF_TOKEN"])
        model_repo = os.environ["MODEL_REPO"]

    def forward_loss(batch):
        vf = batch["vis_feats"].to(device, torch.float16)
        in_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        emb = ddp_model.module.get_input_embeddings()(in_ids).clone()
        emb[:, :32, :] = vf.to(emb.dtype)
        return ddp_model(inputs_embeds=emb, attention_mask=attn, labels=lbls).loss

    @torch.no_grad()
    def eval_dev_epoch(epoch):
        base_peft = ddp_model.module
        base_peft.eval()
        base_peft.config.use_cache = True
        torch.cuda.empty_cache()

        dev_loss = 0.0
        if rank == 0:
            tot_l, n_b = 0.0, 0
            for b in dev_loss_loader:
                vf = b["vis_feats"].to(device, torch.float16)
                in_ids = b["input_ids"].to(device)
                attn = b["attention_mask"].to(device)
                lbls = b["labels"].to(device)
                emb = base_peft.get_input_embeddings()(in_ids).clone()
                emb[:, :32, :] = vf.to(emb.dtype)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    tot_l += base_peft(inputs_embeds=emb, attention_mask=attn, labels=lbls).loss.item()
                n_b += 1
            dev_loss = tot_l / max(n_b, 1)

        my_ids = sorted(dev_ids[rank::world_size], key=lambda x: len(gt_dev[x]["ocr_str"]))
        my_preds = {}
        emb_layer = base_peft.get_input_embeddings()
        bs_gen = 8
        for i in range(0, len(my_ids), bs_gen):
            b_ids = my_ids[i:i + bs_gen]
            seqs = []
            for iid in b_ids:
                p_ids = tokenizer.encode(eval_suite.build_prompt(gt_dev[iid]["ocr_str"]), add_special_tokens=True)
                seqs.append([IMAGE_TOKEN_ID] * 32 + p_ids)
            max_l = max(len(s) for s in seqs)
            B = len(seqs)
            in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device=device)
            attn = torch.zeros((B, max_l), dtype=torch.long, device=device)
            vf_batch = torch.stack([dev_feats[iid] for iid in b_ids]).to(device, torch.float16)
            emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device=device)
            for j, s in enumerate(seqs):
                L = len(s)
                pad = max_l - L
                in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device=device)
                attn[j, pad:] = 1
                row_emb = emb_layer(in_ids[j]).to(torch.float16)
                row_emb[pad:pad+32] = vf_batch[j]
                emb[j] = row_emb
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = base_peft.generate(
                    inputs_embeds=emb, attention_mask=attn,
                    num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
                    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
                )
            texts = tokenizer.batch_decode(out, skip_special_tokens=True)
            for iid, t in zip(b_ids, texts):
                my_preds[str(iid)] = t.strip()

        shard_p = Path(f"/kaggle/tmp/dev_preds_ep{epoch}_r{rank}.json")
        shard_p.write_text(json.dumps(my_preds), encoding="utf-8")
        torch.cuda.empty_cache()
        dist.barrier()

        metrics = None
        if rank == 0:
            all_preds = {}
            for r_i in range(world_size):
                sp = Path(f"/kaggle/tmp/dev_preds_ep{epoch}_r{r_i}.json")
                all_preds.update({int(k): v for k, v in json.loads(sp.read_text()).items()})
                sp.unlink()
            metrics = eval_suite.evaluate_predictions(
                all_preds, gt_dev, DATA_DIR / "train_images/train",
                run_spice=False, run_clip=True, device=f"cuda:{local_rank}",
            )
            metrics["dev_loss"] = round(dev_loss, 4)
            Path(CKPT_DIR / f"dev_preds_ep{epoch}.json").write_text(
                json.dumps({str(k): v for k, v in all_preds.items()}, ensure_ascii=False), encoding="utf-8"
            )

        base_peft.config.use_cache = False
        base_peft.train()
        dist.barrier()
        return metrics

    best_cider = -1.0
    best_ep = 0
    global_step = 0
    smooth_loss = []
    t_train0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        sampler.set_epoch(epoch)
        ddp_model.train()
        optimizer.zero_grad(set_to_none=True)
        t_ep = time.time()

        for step_idx, batch in enumerate(loader):
            is_sync = ((step_idx + 1) % GRAD_ACCUM == 0) or ((step_idx + 1) == len(loader))
            if not is_sync:
                with ddp_model.no_sync():
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        loss = forward_loss(batch) / GRAD_ACCUM
                    scaler.scale(loss).backward()
                    smooth_loss.append(loss.item() * GRAD_ACCUM)
            else:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = forward_loss(batch) / GRAD_ACCUM
                scaler.scale(loss).backward()
                smooth_loss.append(loss.item() * GRAD_ACCUM)

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if rank == 0 and global_step % 50 == 0:
                    avg_l = sum(smooth_loss[-50:]) / len(smooth_loss[-50:])
                    lr_now = scheduler.get_last_lr()[0]
                    vram = torch.cuda.max_memory_allocated(local_rank) / 1e9
                    print(f"[Ep {epoch}/{EPOCHS} | Step {global_step}/{total_steps}] "
                          f"loss={avg_l:.4f} | lr={lr_now:.2e} | VRAM={vram:.2f}GB | {(time.time()-t_train0)/60:.1f}m")
                    wandb.log({"train/loss": avg_l, "train/lr": lr_now, "train/epoch": epoch}, step=global_step)

        dev_metrics = eval_dev_epoch(epoch)
        if rank == 0:
            ov = dev_metrics["overall"]
            tx = dev_metrics["text_detected"]
            nt = dev_metrics["no_text"]
            bh = dev_metrics["behavior"]
            print(f"\n=== EPOCH {epoch} DEV RESULTS ({(time.time()-t_ep)/60:.1f} min) ===")
            print(f"  dev_loss={dev_metrics['dev_loss']:.4f} | CIDEr={ov['CIDEr']} (text={tx['CIDEr']}, no_text={nt['CIDEr']}) | "
                  f"BLEU-4={ov['BLEU-4']} | METEOR={ov['METEOR']} | ROUGE-L={ov['ROUGE-L']}")
            print(f"  CLIPScore={ov['CLIPScore']} | RefCLIPScore={ov['RefCLIPScore']} | "
                  f"unsup_noun={ov['unsup_noun_pct']}% | ocr_adopt={bh['ocr_token_adoption_pct']}% | avg_words={ov['avg_words']}")

            wandb.log({
                "dev/loss": dev_metrics["dev_loss"],
                "dev/overall/CIDEr": ov["CIDEr"],
                "dev/overall/BLEU-4": ov["BLEU-4"],
                "dev/overall/METEOR": ov["METEOR"],
                "dev/overall/ROUGE-L": ov["ROUGE-L"],
                "dev/overall/CLIPScore": ov["CLIPScore"],
                "dev/overall/RefCLIPScore": ov["RefCLIPScore"],
                "dev/overall/unsup_noun_pct": ov["unsup_noun_pct"],
                "dev/text_detected/CIDEr": tx["CIDEr"],
                "dev/no_text/CIDEr": nt["CIDEr"],
                "dev/behavior/ocr_token_adoption_pct": bh["ocr_token_adoption_pct"],
            }, step=global_step)

            ep_dir = CKPT_DIR / f"epoch_{epoch}"
            ddp_model.module.save_pretrained(ep_dir)
            (ep_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))

            if ov["CIDEr"] > best_cider:
                best_cider = ov["CIDEr"]
                best_ep = epoch
                best_dir = CKPT_DIR / "best"
                ddp_model.module.save_pretrained(best_dir)
                (best_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))
                api.upload_folder(
                    folder_path=str(best_dir),
                    repo_id=model_repo,
                    commit_message=f"Track A best adapter (epoch {epoch}, dev CIDEr={best_cider})",
                )
                print(f"  -> New BEST! Uploaded epoch {epoch} adapter (dev CIDEr={best_cider}) to {model_repo}")

    if rank == 0:
        print(f"\nTRAINING COMPLETE! Best epoch={best_ep} with dev CIDEr={best_cider}")
        run.finish()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["WANDB_API_KEY"] = wandb_key
env["MODEL_REPO"] = MODEL_REPO
env["ART_REPO"] = ART_REPO
env["HF_HOME"] = "/kaggle/tmp/hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.Popen(
    ["torchrun", "--standalone", "--nproc_per_node=2", str(train_script)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"DDP training exited with code {proc.returncode}")

In [ ]:
import os, subprocess, zipfile, shutil
from pathlib import Path
import torch

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/kaggle/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

# 1. Install dependencies & remove incompatible preinstalled torchao
!pip install -q peft accelerate pycocoevalcap open-clip-torch spacy nltk gdown
!pip uninstall -y -q torchao
!python -m spacy download en_core_web_sm

import nltk
for pkg in ["wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# 2. Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
print("HF_TOKEN & WANDB_API_KEY: FOUND")

# 3. Restore artifacts, eval_suite.py, and winning Track A LoRA adapter from HF Hub
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

ART_DIR = Path("/kaggle/working/artifacts")
CODE_DIR = Path("/kaggle/working/src")
BEST_TA = Path("/kaggle/working/checkpoints/trackA/best")
ART_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR.mkdir(parents=True, exist_ok=True)
BEST_TA.mkdir(parents=True, exist_ok=True)

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"
TA_REPO = f"{hf_user}/vizwiz-alttext-lora-trackA"

for remote_p, dst in [
    ("phase1/splits_v1.json", ART_DIR / "splits_v1.json"),
    ("phase2/ocr_cache_v1.json", ART_DIR / "ocr_cache_v1.json"),
    ("phase2/ocr_cache_dev_val_v1.json", ART_DIR / "ocr_cache_dev_val_v1.json"),
    ("code/eval_suite.py", CODE_DIR / "eval_suite.py"),
]:
    if not dst.exists():
        dl = hf_hub_download(ART_REPO, remote_p, repo_type="dataset", token=hf_token)
        shutil.copy(dl, dst)
        print(f"Restored {dst.name} from HF Hub")
    else:
        print(f"Found local {dst.name}")

if not (BEST_TA / "adapter_model.safetensors").exists():
    snapshot_download(repo_id=TA_REPO, local_dir=str(BEST_TA), token=hf_token)
    print(f"Restored Track A best adapter from {TA_REPO}")
else:
    print("Found local Track A best adapter")

# 4. Restore VizWiz annotations + images into /kaggle/tmp/vizwiz
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ANN_DIR = DATA_DIR / "annotations"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR.mkdir(parents=True, exist_ok=True)

if not (ANN_DIR / "annotations" / "train.json").exists():
    ann_zip = DATA_DIR / "annotations_dl.zip"
    subprocess.run(["gdown", "1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH", "-O", str(ann_zip)], check=True)
    with zipfile.ZipFile(ann_zip, "r") as zf:
        zf.extractall(ANN_DIR)
    ann_zip.unlink(missing_ok=True)

for name, url in [
    ("train_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip"),
    ("val_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"),
]:
    extract_dir = DATA_DIR / name
    if not extract_dir.exists():
        out_zip = DATA_DIR / f"{name}.zip"
        print(f"Downloading {name} ...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(out_zip), url], check=True)
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(out_zip, "r") as zf:
            zf.extractall(extract_dir)
        out_zip.unlink(missing_ok=True)

print(f"Ready! GPU 0 allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

In [ ]:
import os, gc, json, time, subprocess
from pathlib import Path
import torch

# 1. Free diagnostic model from GPU 0 before spawning 2-GPU workers
for var in ["m", "model", "inp", "out_m", "out_direct", "out_blip", "emb", "v_out", "q_out", "vis_feat"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
print(f"Kernel GPU 0 allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
FEAT_DIR.mkdir(parents=True, exist_ok=True)

val_feat_file = FEAT_DIR / "vis_feats_val_v1.pt"
if not val_feat_file.exists():
    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    val_raw = json.loads((DATA_DIR / "annotations/annotations/val.json").read_text())
    val_meta = {im["id"]: im for im in val_raw["images"]}
    val_dir = DATA_DIR / "val_images/val"

    extract_py = FEAT_DIR / "extract_feats.py"
    extract_py.write_text('''
import sys, json
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_json, out_pt, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_json).read())

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
proj_dtype = model.language_projection.weight.dtype

feats = {}
with torch.no_grad():
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        px = processor(images=pils, return_tensors="pt")["pixel_values"].to("cuda:0", torch.float16)
        v_out = model.vision_model(pixel_values=px)[0]
        v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
        q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
        q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
        proj = model.language_projection(q_out.to(proj_dtype)).to(torch.float16).cpu()
        for j, x in enumerate(batch):
            feats[x["image_id"]] = proj[j]
torch.save(feats, out_pt)
''', encoding="utf-8")

    all_items = [
        {"image_id": iid, "path": str(val_dir / val_meta[iid]["file_name"])}
        for iid in splits["val_ids"]
    ]
    t0_f, t1_f = FEAT_DIR / "val_todo_0.json", FEAT_DIR / "val_todo_1.json"
    s0_f, s1_f = FEAT_DIR / "val_shard_0.pt", FEAT_DIR / "val_shard_1.pt"
    t0_f.write_text(json.dumps(all_items[0::2]))
    t1_f.write_text(json.dumps(all_items[1::2]))

    t0 = time.time()
    procs = []
    for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        procs.append(subprocess.Popen(
            ["python3", str(extract_py), str(tf), str(sf), "32"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))
    while any(p.poll() is None for p in procs):
        time.sleep(20)
        print(f"  Extracting val Q-Former features ... ({(time.time()-t0)/60:.1f} min)")
    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

    merged = {}
    for sf in [s0_f, s1_f]:
        merged.update(torch.load(sf, weights_only=True))
        sf.unlink()
    torch.save(merged, val_feat_file)
    print(f"Saved {len(merged)} val embeddings in {(time.time()-t0)/60:.1f} min ({val_feat_file.stat().st_size/1e9:.2f} GB)")
else:
    print("vis_feats_val_v1.pt already exists.")

In [ ]:
import os, json, time, subprocess, sys
from pathlib import Path
import wandb
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-trackA"

ART_DIR = Path("/kaggle/working/artifacts")
DATA_DIR = Path("/kaggle/tmp/vizwiz")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
TMP_DIR = Path("/kaggle/tmp/trackA_val_v2")
TMP_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR = Path("/kaggle/working/checkpoints/trackA/best")

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import importlib, eval_suite
importlib.reload(eval_suite)

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
gt_val = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/val.json", splits["val_ids"], ocr_dv)

# 1. Recover any predictions already flushed in shard_0 / shard_1
preds_val = {}
for sf in sorted(TMP_DIR.glob("shard_*.jsonl")):
    with open(sf, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
                preds_val[int(r["image_id"])] = r["pred"]
            except json.JSONDecodeError:
                pass

remaining_tasks = [
    {"image_id": iid, "prompt": eval_suite.build_prompt(info["ocr_str"])}
    for iid, info in gt_val.items()
    if iid not in preds_val
]
remaining_tasks.sort(key=lambda x: len(x["prompt"]))
print(f"Recovered {len(preds_val)}/7750 predictions | Remaining: {len(remaining_tasks)}")

# 2. Write OOM-safe worker (bs=6 + automatic bs=1 fallback on long OCR outliers)
worker = TMP_DIR / "val_worker_opt.py"
worker.write_text('''
import sys, json, gc
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

todo_file, out_file, feat_pt, adapter_dir, bs = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())
if not items:
    sys.exit(0)
feats = torch.load(feat_pt, weights_only=True)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer

full_model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
IMAGE_TOKEN_ID = full_model.config.image_token_id
opt_model = full_model.language_model
del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens, full_model
gc.collect()

opt_model = PeftModel.from_pretrained(opt_model.to("cuda:0"), adapter_dir)
opt_model.eval()
emb_layer = opt_model.get_input_embeddings()

def run_batch(batch):
    seqs = [[IMAGE_TOKEN_ID] * 32 + tokenizer.encode(x["prompt"], add_special_tokens=True) for x in batch]
    max_l = max(len(s) for s in seqs)
    B = len(seqs)
    in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device="cuda:0")
    attn = torch.zeros((B, max_l), dtype=torch.long, device="cuda:0")
    vf_batch = torch.stack([feats[x["image_id"]] for x in batch]).to("cuda:0", torch.float16)
    emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device="cuda:0")
    for j, s in enumerate(seqs):
        L = len(s)
        pad = max_l - L
        in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device="cuda:0")
        attn[j, pad:] = 1
        row_emb = emb_layer(in_ids[j]).to(torch.float16)
        row_emb[pad:pad+32] = vf_batch[j]
        emb[j] = row_emb
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = opt_model.generate(
            inputs_embeds=emb, attention_mask=attn,
            num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

with open(out_file, "a", encoding="utf-8") as fout:
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        try:
            texts = run_batch(batch)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            texts = [run_batch([x])[0] for x in batch]
        for x, t in zip(batch, texts):
            fout.write(json.dumps({"image_id": x["image_id"], "pred": t.strip()}, ensure_ascii=False) + "\\n")
        fout.flush()
''', encoding="utf-8")

if remaining_tasks:
    t0 = time.time()
    t0_f, t1_f = TMP_DIR / "todo_rem_0.json", TMP_DIR / "todo_rem_1.json"
    s0_f, s1_f = TMP_DIR / "shard_rem_0.jsonl", TMP_DIR / "shard_rem_1.jsonl"
    for p in [s0_f, s1_f]:
        if p.exists(): p.unlink()
    t0_f.write_text(json.dumps(remaining_tasks[0::2]))
    t1_f.write_text(json.dumps(remaining_tasks[1::2]))

    procs = []
    for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        procs.append(subprocess.Popen(
            ["python3", str(worker), str(tf), str(sf), str(FEAT_DIR / "vis_feats_val_v1.pt"), str(BEST_DIR), "6"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))

    def _cnt(p):
        return sum(1 for l in open(p) if l.strip()) if p.exists() else 0

    while any(p.poll() is None for p in procs):
        time.sleep(20)
        n_done = _cnt(s0_f) + _cnt(s1_f)
        rate = n_done / max(time.time() - t0, 1e-6)
        print(f"  [val {(time.time()-t0)/60:.1f}m] {len(preds_val)+n_done}/7750 ({rate:.2f} img/s)")

    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

    for sf in [s0_f, s1_f]:
        for line in open(sf, "r", encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                preds_val[int(r["image_id"])] = r["pred"]

assert len(preds_val) == 7750, f"Expected 7750, got {len(preds_val)}"
preds_path = ART_DIR / "preds_trackA_val_v1.json"
preds_path.write_text(json.dumps({str(k): v for k, v in preds_val.items()}, ensure_ascii=False), encoding="utf-8")

res_val = eval_suite.evaluate_predictions(preds_val, gt_val, DATA_DIR / "val_images/val", run_spice=False, run_clip=True)
res_dev = json.loads((BEST_DIR / "dev_metrics.json").read_text())

trackA_results = {"dev_best": res_dev, "val": res_val}
res_path = ART_DIR / "trackA_results_v1.json"
res_path.write_text(json.dumps(trackA_results, indent=2), encoding="utf-8")

for p in [preds_path, res_path]:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=f"phase3/{p.name}", repo_id=ART_REPO, repo_type="dataset")

run = wandb.init(project="vizwiz-alttext-blip2", name="phase3-trackA-val-fixed", config={"model": MODEL_REPO, "eos_token_id": 2})
flat = {f"val/{grp}/{k}": v for grp, d in res_val.items() for k, v in d.items() if v is not None}
wandb.log(flat)
run.finish()

print("\n=== PHASE 3 TRACK A FINAL RESULTS (DEV + VAL) ===")
print(json.dumps(trackA_results, indent=2))

In [ ]:
import json, torch
from pathlib import Path
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

ART_DIR = Path("/kaggle/working/artifacts")
CKPT_DIR = Path("/kaggle/working/checkpoints/trackA")
DATA_DIR = Path("/kaggle/tmp/vizwiz")

dev_preds = json.loads((CKPT_DIR / "dev_preds_ep4.json").read_text())
val_preds = json.loads((ART_DIR / "preds_trackA_val_v1.json").read_text())

print("=== 5 DEV EPOCH-4 PREDICTIONS (CIDEr=101.25) ===")
for k in list(dev_preds.keys())[:5]:
    print(f"  {k}: {repr(dev_preds[k])}")

print("\n=== 5 VAL PREDICTIONS FROM val_worker (CIDEr=0.46) ===")
for k in list(val_preds.keys())[:5]:
    print(f"  {k}: {repr(val_preds[k])}")

# Check what val_worker actually produces on those exact 5 dev images!
import sys
if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", splits["dev_ids"], ocr_dv)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.language_model = PeftModel.from_pretrained(model.language_model, str(CKPT_DIR / "best"))
model.eval()

print("\n=== DIAGNOSTIC ON FIRST 3 DEV IMAGES VIA Blip2 + PeftModel ===")
for iid in splits["dev_ids"][:3]:
    info = gt_dev[iid]
    pil = ImageOps.exif_transpose(Image.open(DATA_DIR / "train_images/train" / info["file_name"])).convert("RGB")
    prompt = eval_suite.build_prompt(info["ocr_str"])
    inp = processor(images=pil, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)
    in_len = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inp, num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3)
    print(f"ID {iid} | in_len={in_len} | out.shape={tuple(out.shape)}")
    print("  raw_full:  ", repr(processor.tokenizer.decode(out[0], skip_special_tokens=False)))
    print("  sliced:    ", repr(processor.tokenizer.decode(out[0, in_len:], skip_special_tokens=True)))
    print("  expected:  ", repr(dev_preds[str(iid)]))

In [ ]:
import json, torch
from pathlib import Path
from safetensors.torch import load_file

CKPT_BEST = Path("/kaggle/working/checkpoints/trackA/best")
print("adapter_config.json:", (CKPT_BEST / "adapter_config.json").read_text())
sd = load_file(str(CKPT_BEST / "adapter_model.safetensors"))
print("Num tensors in adapter_model.safetensors:", len(sd))
print("Non-lora keys:", [k for k in sd.keys() if "lora_" not in k][:10])
print("Sample lora keys:", [k for k in sd.keys() if "lora_" in k][:4])

print("\nBlip2 generation_config:", model.generation_config)
print("LM generation_config:   ", model.language_model.generation_config)
print("Tokenizer eos/pad/bos:  ", processor.tokenizer.eos_token_id, processor.tokenizer.pad_token_id, processor.tokenizer.bos_token_id)
print("Active adapters:", model.language_model.active_adapters)

# Test calling model.language_model.generate(inputs_embeds=...) directly (like eval_dev_epoch did!)
info = gt_dev[37]
pil = ImageOps.exif_transpose(Image.open(DATA_DIR / "train_images/train" / info["file_name"])).convert("RGB")
prompt = eval_suite.build_prompt(info["ocr_str"])
inp = processor(images=pil, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)

with torch.no_grad():
    # A) Check Blip2's own vision+qformer+projection -> language_model.generate
    v_out = model.vision_model(pixel_values=inp["pixel_values"])[0]
    v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
    q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
    q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
    vis_feat = model.language_projection(q_out.to(model.language_projection.weight.dtype)).to(torch.float16)

    emb = model.language_model.get_input_embeddings()(inp["input_ids"]).clone()
    emb[:, :32, :] = vis_feat
    out_direct = model.language_model.generate(
        inputs_embeds=emb,
        attention_mask=inp["attention_mask"],
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    print("\n[A] Direct language_model.generate(inputs_embeds):", repr(processor.tokenizer.decode(out_direct[0], skip_special_tokens=True)))

    # B) Check Blip2ForConditionalGeneration.generate WITH explicit eos_token_id/pad_token_id
    out_blip = model.generate(
        **inp,
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    print("[B] model.generate(..., eos=2) raw:", repr(processor.tokenizer.decode(out_blip[0], skip_special_tokens=True)))

In [ ]:
import os, gc, json, time, inspect, subprocess
from pathlib import Path
import torch
from transformers import Blip2ForConditionalGeneration

# 1. Print Blip2ForConditionalGeneration.generate source so we see the exact transformers 5.20.2 bug
print("=== Blip2ForConditionalGeneration.generate source ===")
print(inspect.getsource(Blip2ForConditionalGeneration.generate))

# 2. Free GPU 0 memory before spawning 2-GPU extractor
for var in ["m", "model", "out_m", "inp"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
FEAT_DIR.mkdir(parents=True, exist_ok=True)

val_feat_file = FEAT_DIR / "vis_feats_val_v1.pt"
if not val_feat_file.exists():
    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    val_raw = json.loads((DATA_DIR / "annotations/annotations/val.json").read_text())
    val_meta = {im["id"]: im for im in val_raw["images"]}
    val_dir = DATA_DIR / "val_images/val"

    extract_py = FEAT_DIR / "extract_feats.py"  # already written in Phase 3A; rewrite safely just in case
    extract_py.write_text('''
import sys, json
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_json, out_pt, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_json).read())

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
proj_dtype = model.language_projection.weight.dtype

feats = {}
with torch.no_grad():
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        px = processor(images=pils, return_tensors="pt")["pixel_values"].to("cuda:0", torch.float16)
        v_out = model.vision_model(pixel_values=px)[0]
        v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
        q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
        q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
        proj = model.language_projection(q_out.to(proj_dtype)).to(torch.float16).cpu()
        for j, x in enumerate(batch):
            feats[x["image_id"]] = proj[j]
torch.save(feats, out_pt)
''', encoding="utf-8")

    all_items = [
        {"image_id": iid, "path": str(val_dir / val_meta[iid]["file_name"])}
        for iid in splits["val_ids"]
    ]
    t0_f, t1_f = FEAT_DIR / "val_todo_0.json", FEAT_DIR / "val_todo_1.json"
    s0_f, s1_f = FEAT_DIR / "val_shard_0.pt", FEAT_DIR / "val_shard_1.pt"
    t0_f.write_text(json.dumps(all_items[0::2]))
    t1_f.write_text(json.dumps(all_items[1::2]))

    t0 = time.time()
    procs = []
    for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        procs.append(subprocess.Popen(
            ["python3", str(extract_py), str(tf), str(sf), "32"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))
    while any(p.poll() is None for p in procs):
        time.sleep(20)
        print(f"  Extracting val Q-Former features ... ({(time.time()-t0)/60:.1f} min)")
    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

    merged = {}
    for sf in [s0_f, s1_f]:
        merged.update(torch.load(sf, weights_only=True))
        sf.unlink()
    torch.save(merged, val_feat_file)
    print(f"Saved {len(merged)} val embeddings in {(time.time()-t0)/60:.1f} min ({val_feat_file.stat().st_size/1e9:.2f} GB)")
else:
    print("vis_feats_val_v1.pt already exists.")

Phase 4

In [ ]:
import os, subprocess, shutil
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
print("=== SYSTEM MEMORY ===")
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)

SYN_DIR = Path("/kaggle/tmp/phase4_synth")
SYN_IMG_DIR = SYN_DIR / "images"
if SYN_IMG_DIR.exists():
    shutil.rmtree(SYN_IMG_DIR)
SYN_IMG_DIR.mkdir(parents=True, exist_ok=True)

# 1. Subprocess 1: CLIP Calibration + Synthetic Degradation (exits to free 100% RAM)
stage1_py = SYN_DIR / "stage1_clip_deg.py"
stage1_py.write_text(r'''
import os, re, gc, json, math, random
from collections import defaultdict
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
import torch
import open_clip

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
SYN_IMG_DIR = SYN_DIR / "images"

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())
file_by_id = {im["id"]: im["file_name"] for im in train_raw["images"]}

first_cap = {}
precanned_counts = defaultdict(int)
for a in train_raw["annotations"]:
    iid = a["image_id"]
    if a.get("is_precanned", False):
        precanned_counts[iid] += 1
    elif not a.get("is_rejected", False) and iid not in first_cap:
        c = re.sub(r"\s+", " ", a["caption"]).strip()
        if c:
            first_cap[iid] = c
del train_raw
gc.collect()

train_dir = DATA_DIR / "train_images/train"

def apply_degradation(pil: Image.Image, kind: str, level: int, rng: random.Random) -> Image.Image:
    im = pil.copy()
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    w, h = im.size
    scale = max(w, h) / 1024.0
    if kind == "gaussian":
        radii = {1: 5.0, 2: 12.0, 3: 24.0, 4: 40.0}
        return im.filter(ImageFilter.GaussianBlur(radius=max(1.5, radii[level] * scale)))
    elif kind == "motion":
        sizes = {1: 15, 2: 33, 3: 59, 4: 91}
        k = max(5, int(sizes[level] * scale) | 1)
        kernel = np.zeros((k, k), dtype=np.float32)
        if rng.random() < 0.5:
            kernel[k // 2, :] = 1.0 / k
        else:
            np.fill_diagonal(kernel, 1.0 / k)
        return Image.fromarray(cv2.filter2D(np.asarray(im), -1, kernel))
    elif kind == "dark":
        factors = {1: 0.25, 2: 0.13, 3: 0.07, 4: 0.03}
        return ImageEnhance.Brightness(im).enhance(factors[level])
    elif kind == "occlude":
        ratios = {1: 0.45, 2: 0.65, 3: 0.80, 4: 0.90}
        r = ratios[level]
        if rng.random() < 0.5:
            cw, ch = max(24, int(w * (1.0 - r * 0.78))), max(24, int(h * (1.0 - r * 0.78)))
            x0 = rng.randint(0, max(0, w - cw))
            y0 = rng.randint(0, max(0, h - ch))
            return im.crop((x0, y0, x0 + cw, y0 + ch)).resize((w, h), Image.Resampling.BICUBIC)
        else:
            arr = np.asarray(im).copy()
            bw, bh = int(w * math.sqrt(r)), int(h * math.sqrt(r))
            x0 = rng.randint(0, max(0, w - bw))
            y0 = rng.randint(0, max(0, h - bh))
            arr[y0:y0+bh, x0:x0+bw] = rng.randint(0, 20)
            return Image.fromarray(arr)
    return im

device = "cuda:0"
clip_model, _, clip_prep = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai", quick_gelu=True, device=device
)
clip_tok = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

@torch.no_grad()
def clip_sim_batch(pils, texts, bs=32):
    sims = []
    for i in range(0, len(pils), bs):
        ims = torch.stack([clip_prep(p) for p in pils[i:i+bs]]).to(device)
        toks = clip_tok(["A photo depicts " + t for t in texts[i:i+bs]]).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            im_f = torch.nn.functional.normalize(clip_model.encode_image(ims).float(), dim=-1)
            tx_f = torch.nn.functional.normalize(clip_model.encode_text(toks).float(), dim=-1)
        sims.extend((2.5 * (im_f * tx_f).sum(dim=-1).clamp(min=0.0)).cpu().tolist())
    return sims

dev_clear_ids = [iid for iid in splits["dev_ids"] if precanned_counts[iid] == 0 and iid in first_cap][:120]
rng = random.Random(42)

clean_pils = []
for iid in dev_clear_ids:
    with Image.open(train_dir / file_by_id[iid]) as raw:
        im = ImageOps.exif_transpose(raw).convert("RGB")
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    clean_pils.append(im)
clean_caps = [first_cap[iid] for iid in dev_clear_ids]
clean_sims = clip_sim_batch(clean_pils, clean_caps)

print("=== DEV CLIP CALIBRATION (120 strictly-clear dev images) ===", flush=True)
print(f"  Clean images CLIPScore: mean={np.mean(clean_sims):.4f}, p10={np.percentile(clean_sims, 10):.4f}, p25={np.percentile(clean_sims, 25):.4f}", flush=True)

for kind in ["gaussian", "motion", "dark", "occlude"]:
    row = []
    for lvl in [1, 2, 3, 4]:
        deg_pils = [apply_degradation(p, kind, lvl, rng) for p in clean_pils]
        deg_sims = clip_sim_batch(deg_pils, clean_caps)
        mean_s = float(np.mean(deg_sims))
        mean_drop = float(np.mean(np.array(clean_sims) - np.array(deg_sims)))
        row.append(f"L{lvl}={mean_s:.3f} (drop={mean_drop:+.3f})")
        del deg_pils
    print(f"  {kind:8s}: " + " | ".join(row), flush=True)

del clean_pils
gc.collect()

THRESH_CLIP = float(np.percentile(clean_sims, 10))
MIN_DROP = 0.08
print(f"\nCalibrated Unclear Rule -> deg_clip < {THRESH_CLIP:.4f} AND (clean_clip - deg_clip) >= {MIN_DROP:.2f}", flush=True)

# Build 50x4 Dev Blur Curve
curve_ids = dev_clear_ids[:50]
curve_records = []
for iid in curve_ids:
    with Image.open(train_dir / file_by_id[iid]) as raw:
        pil = ImageOps.exif_transpose(raw).convert("RGB")
    for lvl in [1, 2, 3, 4]:
        dpil = apply_degradation(pil, "gaussian", lvl, rng)
        fn = f"dev_blur_{iid}_L{lvl}.jpg"
        dpil.save(SYN_IMG_DIR / fn, quality=92)
        curve_records.append({"key": f"dev_blur_{iid}_L{lvl}", "image_id": iid, "level": lvl, "path": str(SYN_IMG_DIR / fn)})

# Build 1,800 synthetic train variants in tiny chunks of 16 (<200 MB RAM!)
train_clear_ids = [iid for iid in splits["train_ids"] if precanned_counts[iid] == 0 and iid in first_cap]
rng.shuffle(train_clear_ids)
synth_source_ids = train_clear_ids[:1800]
kinds = ["gaussian", "motion", "dark", "occlude"]

synth_train_records = []
for c_start in range(0, len(synth_source_ids), 16):
    chunk_ids = synth_source_ids[c_start:c_start + 16]
    c_pils, d_pils, caps, metas = [], [], [], []
    for offset, iid in enumerate(chunk_ids):
        idx = c_start + offset
        with Image.open(train_dir / file_by_id[iid]) as raw:
            pil = ImageOps.exif_transpose(raw).convert("RGB")
        pil.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
        kind = kinds[idx % len(kinds)]
        lvl = rng.choice([1, 2, 3, 4])
        dpil = apply_degradation(pil, kind, lvl, rng)
        key = f"syn_{iid}_{kind}_L{lvl}"
        out_p = SYN_IMG_DIR / f"{key}.jpg"
        dpil.save(out_p, quality=92)
        c_pils.append(pil)
        d_pils.append(dpil)
        caps.append(first_cap[iid])
        metas.append((key, iid, kind, lvl, str(out_p)))

    c_sims = clip_sim_batch(c_pils, caps, bs=16)
    d_sims = clip_sim_batch(d_pils, caps, bs=16)
    for (key, iid, kind, lvl, out_p), cs, ds in zip(metas, c_sims, d_sims):
        is_unclear = (ds < THRESH_CLIP) and ((cs - ds) >= MIN_DROP)
        synth_train_records.append({
            "key": key, "image_id": iid, "kind": kind, "level": lvl,
            "is_unclear": is_unclear, "clean_clip": round(cs, 4), "deg_clip": round(ds, 4),
            "path": out_p, "dets": [],
        })
    del c_pils, d_pils
    if (c_start // 16) % 25 == 0:
        gc.collect()
        print(f"  [synth] {min(c_start+16, len(synth_source_ids))}/{len(synth_source_ids)}", flush=True)

manifest = {
    "thresh_clip": round(THRESH_CLIP, 4),
    "min_drop": MIN_DROP,
    "train_synth": synth_train_records,
    "dev_blur_curve": curve_records,
}
(SYN_DIR / "phase4_manifest_no_ocr.json").write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
print("Stage 1 complete!", flush=True)
''', encoding="utf-8")

# 2. Subprocess 2: RapidOCR over saved synthetic images (exits to free 100% RAM)
stage2_py = SYN_DIR / "stage2_ocr.py"
stage2_py.write_text(r'''
import gc, json, logging, warnings
from pathlib import Path
import numpy as np
from PIL import Image
from rapidocr import RapidOCR, EngineType

warnings.filterwarnings("ignore", message="Using padding='same'")
logging.getLogger("RapidOCR").setLevel(logging.ERROR)

SYN_DIR = Path("/kaggle/tmp/phase4_synth")
manifest = json.loads((SYN_DIR / "phase4_manifest_no_ocr.json").read_text())

ocr_engine = RapidOCR(params={
    "Global.log_level": "error",
    "Det.engine_type": EngineType.TORCH,
    "Cls.engine_type": EngineType.TORCH,
    "Rec.engine_type": EngineType.TORCH,
    "EngineConfig.torch.use_cuda": True,
    "EngineConfig.torch.cuda_ep_cfg.device_id": 0,
})

def run_ocr_path(path_str):
    with Image.open(path_str) as raw:
        im = raw.convert("RGB")
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    res = ocr_engine(np.array(im))
    dets = []
    if res and getattr(res, "txts", None):
        boxes = res.boxes if getattr(res, "boxes", None) is not None else [None] * len(res.txts)
        for txt, sc, bx in zip(res.txts, res.scores, boxes):
            if float(sc) >= 0.50 and txt.strip():
                arr = np.asarray(bx) if bx is not None else None
                y_m = round(float(arr[:, 1].min()), 1) if arr is not None else 0.0
                x_m = round(float(arr[:, 0].min()), 1) if arr is not None else 0.0
                dets.append({"text": txt.strip(), "conf": round(float(sc), 4), "y": y_m, "x": x_m})
    return dets

all_recs = manifest["train_synth"] + manifest["dev_blur_curve"]
for idx, rec in enumerate(all_recs, start=1):
    rec["dets"] = run_ocr_path(rec["path"])
    if idx % 400 == 0:
        gc.collect()
        print(f"  [ocr] {idx}/{len(all_recs)}", flush=True)

n_unclear = sum(1 for r in manifest["train_synth"] if r["is_unclear"])
n_keep = len(manifest["train_synth"]) - n_unclear
kinds = ["gaussian", "motion", "dark", "occlude"]
by_kind = {k: sum(1 for r in manifest["train_synth"] if r["kind"] == k and r["is_unclear"]) for k in kinds}

manifest["n_synth_unclear"] = n_unclear
manifest["n_synth_robust_clean"] = n_keep
(SYN_DIR / "phase4_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
print(f"\nBuilt {len(manifest['train_synth'])} synthetic train examples -> unclear={n_unclear} {by_kind}, robust_clean={n_keep}", flush=True)
print(f"Built {len(manifest['dev_blur_curve'])} dev blur-curve images.", flush=True)
''', encoding="utf-8")

env = os.environ.copy()
env["HF_HOME"] = "/kaggle/tmp/hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"

for label, script in [("Stage 1 (CLIP + Degradations)", stage1_py), ("Stage 2 (RapidOCR)", stage2_py)]:
    print(f"\n--- Running {label} ---")
    proc = subprocess.Popen(["python3", str(script)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{label} failed with code {proc.returncode}")

In [ ]:
import os, json, time, subprocess
from pathlib import Path
import torch

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
FEAT_DIR.mkdir(parents=True, exist_ok=True)

extract_py = FEAT_DIR / "extract_feats.py"
if not extract_py.exists():
    extract_py.write_text('''
import sys, json
import torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_json, out_pt, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_json).read())

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
proj_dtype = model.language_projection.weight.dtype

feats = {}
with torch.no_grad():
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        px = processor(images=pils, return_tensors="pt")["pixel_values"].to("cuda:0", torch.float16)
        v_out = model.vision_model(pixel_values=px)[0]
        v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
        q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
        q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
        proj = model.language_projection(q_out.to(proj_dtype)).to(torch.float16).cpu()
        for j, x in enumerate(batch):
            feats[x["image_id"]] = proj[j]
torch.save(feats, out_pt)
''', encoding="utf-8")

def run_2gpu_extract(items, out_map_fn):
    t0_f, t1_f = FEAT_DIR / "tmp_todo_0.json", FEAT_DIR / "tmp_todo_1.json"
    s0_f, s1_f = FEAT_DIR / "tmp_shard_0.pt", FEAT_DIR / "tmp_shard_1.pt"
    t0_f.write_text(json.dumps(items[0::2]))
    t1_f.write_text(json.dumps(items[1::2]))
    t0 = time.time()
    procs = []
    for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        procs.append(subprocess.Popen(
            ["python3", str(extract_py), str(tf), str(sf), "32"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        ))
    while any(p.poll() is None for p in procs):
        time.sleep(20)
        print(f"  Extracting Q-Former features ({len(items)} imgs) ... ({(time.time()-t0)/60:.1f} min)")
    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")
    merged = {}
    for sf in [s0_f, s1_f]:
        merged.update(torch.load(sf, weights_only=True))
        sf.unlink()
    out_map_fn(merged, time.time() - t0)

# 1. Synthetic embeddings (2,000 imgs -> ~1 min)
syn_feat_file = SYN_DIR / "vis_feats_phase4_synth.pt"
manifest = json.loads((SYN_DIR / "phase4_manifest.json").read_text())
syn_items = [{"image_id": r["key"], "path": r["path"]} for r in (manifest["train_synth"] + manifest["dev_blur_curve"])]
run_2gpu_extract(syn_items, lambda m, dt: (torch.save(m, syn_feat_file), print(f"Saved {len(m)} synth embeddings in {dt/60:.1f} min")))

# 2. Train + Dev embeddings if missing
train_feat_file = FEAT_DIR / "vis_feats_train_v1.pt"
dev_feat_file = FEAT_DIR / "vis_feats_dev_v1.pt"
if not (train_feat_file.exists() and dev_feat_file.exists()):
    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())
    img_meta = {im["id"]: im for im in train_raw["images"]}
    train_dir = DATA_DIR / "train_images/train"
    td_items = [{"image_id": iid, "path": str(train_dir / img_meta[iid]["file_name"])} for iid in (splits["train_ids"] + splits["dev_ids"])]
    def _save_td(m, dt):
        torch.save({iid: m[iid] for iid in splits["train_ids"]}, train_feat_file)
        torch.save({iid: m[iid] for iid in splits["dev_ids"]}, dev_feat_file)
        print(f"Saved train + dev embeddings in {dt/60:.1f} min")
    run_2gpu_extract(td_items, _save_td)

# 3. Val embeddings if missing
val_feat_file = FEAT_DIR / "vis_feats_val_v1.pt"
if not val_feat_file.exists():
    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    val_raw = json.loads((DATA_DIR / "annotations/annotations/val.json").read_text())
    val_meta = {im["id"]: im for im in val_raw["images"]}
    val_dir = DATA_DIR / "val_images/val"
    v_items = [{"image_id": iid, "path": str(val_dir / val_meta[iid]["file_name"])} for iid in splits["val_ids"]]
    run_2gpu_extract(v_items, lambda m, dt: (torch.save(m, val_feat_file), print(f"Saved {len(m)} val embeddings in {dt/60:.1f} min")))

print("All Phase 4 features ready! Cached:", [p.name for p in FEAT_DIR.glob("*.pt")])

In [ ]:
import os, json, time, subprocess, zipfile, shutil
from pathlib import Path
import torch

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/kaggle/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

# 1. Packages & incompatible torchao removal
try:
    import peft, pycocoevalcap, open_clip, spacy, rapidocr, gdown
except ImportError:
    !pip install -q peft accelerate pycocoevalcap open-clip-torch spacy nltk gdown rapidocr
!pip uninstall -y -q torchao

try:
    import spacy as _sp
    _sp.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm

import nltk
for pkg in ["wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# 2. Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

# 3. Restore HF Hub artifacts, eval_suite.py & Track A adapter
ART_DIR = Path("/kaggle/working/artifacts")
CODE_DIR = Path("/kaggle/working/src")
BEST_TA = Path("/kaggle/working/checkpoints/trackA/best")
ART_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR.mkdir(parents=True, exist_ok=True)
BEST_TA.mkdir(parents=True, exist_ok=True)

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"

for remote_p, dst in [
    ("phase1/splits_v1.json", ART_DIR / "splits_v1.json"),
    ("phase2/ocr_cache_v1.json", ART_DIR / "ocr_cache_v1.json"),
    ("phase2/ocr_cache_dev_val_v1.json", ART_DIR / "ocr_cache_dev_val_v1.json"),
    ("code/eval_suite.py", CODE_DIR / "eval_suite.py"),
]:
    if not dst.exists():
        dl = hf_hub_download(ART_REPO, remote_p, repo_type="dataset", token=hf_token)
        shutil.copy(dl, dst)
        print(f"Restored {dst.name}")

if not (BEST_TA / "adapter_model.safetensors").exists():
    snapshot_download(repo_id=f"{hf_user}/vizwiz-alttext-lora-trackA", local_dir=str(BEST_TA), token=hf_token)
    print("Restored Track A best adapter")

# 4. Restore VizWiz data into /kaggle/tmp/vizwiz if missing
DATA_DIR = Path("/kaggle/tmp/vizwiz")
ANN_DIR = DATA_DIR / "annotations"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR.mkdir(parents=True, exist_ok=True)

if not (ANN_DIR / "annotations" / "train.json").exists():
    ann_zip = DATA_DIR / "annotations_dl.zip"
    subprocess.run(["gdown", "1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH", "-O", str(ann_zip)], check=True)
    with zipfile.ZipFile(ann_zip, "r") as zf:
        zf.extractall(ANN_DIR)
    ann_zip.unlink(missing_ok=True)

for name, url in [
    ("train_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip"),
    ("val_images", "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"),
]:
    extract_dir = DATA_DIR / name
    if not extract_dir.exists():
        out_zip = DATA_DIR / f"{name}.zip"
        print(f"Downloading {name} ...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(out_zip), url], check=True)
        with zipfile.ZipFile(out_zip, "r") as zf:
            zf.extractall(extract_dir)
        out_zip.unlink(missing_ok=True)

# 5. Rebuild Phase 4 synthetic set in isolated subprocesses if missing
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
SYN_IMG_DIR = SYN_DIR / "images"
manifest_path = SYN_DIR / "phase4_manifest.json"
if not manifest_path.exists():
    SYN_IMG_DIR.mkdir(parents=True, exist_ok=True)
    stage1_py = SYN_DIR / "stage1_clip_deg.py"
    stage1_py.write_text(r'''
import os, re, gc, json, math, random
from collections import defaultdict
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
import torch
import open_clip

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
SYN_IMG_DIR = SYN_DIR / "images"

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())
file_by_id = {im["id"]: im["file_name"] for im in train_raw["images"]}

first_cap = {}
precanned_counts = defaultdict(int)
for a in train_raw["annotations"]:
    iid = a["image_id"]
    if a.get("is_precanned", False):
        precanned_counts[iid] += 1
    elif not a.get("is_rejected", False) and iid not in first_cap:
        c = re.sub(r"\s+", " ", a["caption"]).strip()
        if c:
            first_cap[iid] = c
del train_raw; gc.collect()
train_dir = DATA_DIR / "train_images/train"

def apply_degradation(pil, kind, level, rng):
    im = pil.copy()
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    w, h = im.size
    scale = max(w, h) / 1024.0
    if kind == "gaussian":
        radii = {1: 5.0, 2: 12.0, 3: 24.0, 4: 40.0}
        return im.filter(ImageFilter.GaussianBlur(radius=max(1.5, radii[level] * scale)))
    elif kind == "motion":
        sizes = {1: 15, 2: 33, 3: 59, 4: 91}
        k = max(5, int(sizes[level] * scale) | 1)
        kernel = np.zeros((k, k), dtype=np.float32)
        if rng.random() < 0.5: kernel[k // 2, :] = 1.0 / k
        else: np.fill_diagonal(kernel, 1.0 / k)
        return Image.fromarray(cv2.filter2D(np.asarray(im), -1, kernel))
    elif kind == "dark":
        factors = {1: 0.25, 2: 0.13, 3: 0.07, 4: 0.03}
        return ImageEnhance.Brightness(im).enhance(factors[level])
    elif kind == "occlude":
        ratios = {1: 0.45, 2: 0.65, 3: 0.80, 4: 0.90}
        r = ratios[level]
        if rng.random() < 0.5:
            cw, ch = max(24, int(w * (1.0 - r * 0.78))), max(24, int(h * (1.0 - r * 0.78)))
            x0, y0 = rng.randint(0, max(0, w - cw)), rng.randint(0, max(0, h - ch))
            return im.crop((x0, y0, x0 + cw, y0 + ch)).resize((w, h), Image.Resampling.BICUBIC)
        else:
            arr = np.asarray(im).copy()
            bw, bh = int(w * math.sqrt(r)), int(h * math.sqrt(r))
            x0, y0 = rng.randint(0, max(0, w - bw)), rng.randint(0, max(0, h - bh))
            arr[y0:y0+bh, x0:x0+bw] = rng.randint(0, 20)
            return Image.fromarray(arr)
    return im

device = "cuda:0"
clip_model, _, clip_prep = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai", quick_gelu=True, device=device)
clip_tok = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

@torch.no_grad()
def clip_sim_batch(pils, texts, bs=32):
    sims = []
    for i in range(0, len(pils), bs):
        ims = torch.stack([clip_prep(p) for p in pils[i:i+bs]]).to(device)
        toks = clip_tok(["A photo depicts " + t for t in texts[i:i+bs]]).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            im_f = torch.nn.functional.normalize(clip_model.encode_image(ims).float(), dim=-1)
            tx_f = torch.nn.functional.normalize(clip_model.encode_text(toks).float(), dim=-1)
        sims.extend((2.5 * (im_f * tx_f).sum(dim=-1).clamp(min=0.0)).cpu().tolist())
    return sims

dev_clear_ids = [iid for iid in splits["dev_ids"] if precanned_counts[iid] == 0 and iid in first_cap][:120]
rng = random.Random(42)

clean_pils = []
for iid in dev_clear_ids:
    with Image.open(train_dir / file_by_id[iid]) as raw:
        im = ImageOps.exif_transpose(raw).convert("RGB")
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    clean_pils.append(im)
clean_sims = clip_sim_batch(clean_pils, [first_cap[i] for i in dev_clear_ids])
THRESH_CLIP = float(np.percentile(clean_sims, 10))
MIN_DROP = 0.08
del clean_pils; gc.collect()

curve_records = []
for iid in dev_clear_ids[:50]:
    with Image.open(train_dir / file_by_id[iid]) as raw:
        pil = ImageOps.exif_transpose(raw).convert("RGB")
    for lvl in [1, 2, 3, 4]:
        dpil = apply_degradation(pil, "gaussian", lvl, rng)
        fn = f"dev_blur_{iid}_L{lvl}.jpg"
        dpil.save(SYN_IMG_DIR / fn, quality=92)
        curve_records.append({"key": f"dev_blur_{iid}_L{lvl}", "image_id": iid, "level": lvl, "path": str(SYN_IMG_DIR / fn)})

train_clear_ids = [iid for iid in splits["train_ids"] if precanned_counts[iid] == 0 and iid in first_cap]
rng.shuffle(train_clear_ids)
synth_source_ids = train_clear_ids[:1800]
kinds = ["gaussian", "motion", "dark", "occlude"]

synth_train_records = []
for c_start in range(0, len(synth_source_ids), 16):
    chunk_ids = synth_source_ids[c_start:c_start + 16]
    c_pils, d_pils, caps, metas = [], [], [], []
    for offset, iid in enumerate(chunk_ids):
        idx = c_start + offset
        with Image.open(train_dir / file_by_id[iid]) as raw:
            pil = ImageOps.exif_transpose(raw).convert("RGB")
        pil.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
        kind = kinds[idx % len(kinds)]
        lvl = rng.choice([1, 2, 3, 4])
        dpil = apply_degradation(pil, kind, lvl, rng)
        key = f"syn_{iid}_{kind}_L{lvl}"
        out_p = SYN_IMG_DIR / f"{key}.jpg"
        dpil.save(out_p, quality=92)
        c_pils.append(pil); d_pils.append(dpil); caps.append(first_cap[iid])
        metas.append((key, iid, kind, lvl, str(out_p)))
    c_sims = clip_sim_batch(c_pils, caps, bs=16)
    d_sims = clip_sim_batch(d_pils, caps, bs=16)
    for (key, iid, kind, lvl, out_p), cs, ds in zip(metas, c_sims, d_sims):
        synth_train_records.append({
            "key": key, "image_id": iid, "kind": kind, "level": lvl,
            "is_unclear": (ds < THRESH_CLIP) and ((cs - ds) >= MIN_DROP),
            "clean_clip": round(cs, 4), "deg_clip": round(ds, 4), "path": out_p, "dets": [],
        })
    del c_pils, d_pils
    if (c_start // 16) % 25 == 0:
        gc.collect()

(SYN_DIR / "phase4_manifest_no_ocr.json").write_text(json.dumps({
    "thresh_clip": round(THRESH_CLIP, 4), "min_drop": MIN_DROP,
    "train_synth": synth_train_records, "dev_blur_curve": curve_records,
}, ensure_ascii=False), encoding="utf-8")
''', encoding="utf-8")

    stage2_py = SYN_DIR / "stage2_ocr.py"
    stage2_py.write_text(r'''
import gc, json, logging, warnings
from pathlib import Path
import numpy as np
from PIL import Image
from rapidocr import RapidOCR, EngineType

warnings.filterwarnings("ignore", message="Using padding='same'")
logging.getLogger("RapidOCR").setLevel(logging.ERROR)

SYN_DIR = Path("/kaggle/tmp/phase4_synth")
manifest = json.loads((SYN_DIR / "phase4_manifest_no_ocr.json").read_text())
ocr_engine = RapidOCR(params={
    "Global.log_level": "error", "Det.engine_type": EngineType.TORCH,
    "Cls.engine_type": EngineType.TORCH, "Rec.engine_type": EngineType.TORCH,
    "EngineConfig.torch.use_cuda": True, "EngineConfig.torch.cuda_ep_cfg.device_id": 0,
})

def run_ocr_path(path_str):
    with Image.open(path_str) as raw:
        im = raw.convert("RGB")
    im.thumbnail((1024, 1024), Image.Resampling.BICUBIC)
    res = ocr_engine(np.array(im))
    dets = []
    if res and getattr(res, "txts", None):
        boxes = res.boxes if getattr(res, "boxes", None) is not None else [None] * len(res.txts)
        for txt, sc, bx in zip(res.txts, res.scores, boxes):
            if float(sc) >= 0.50 and txt.strip():
                arr = np.asarray(bx) if bx is not None else None
                dets.append({
                    "text": txt.strip(), "conf": round(float(sc), 4),
                    "y": round(float(arr[:, 1].min()), 1) if arr is not None else 0.0,
                    "x": round(float(arr[:, 0].min()), 1) if arr is not None else 0.0,
                })
    return dets

for idx, rec in enumerate(manifest["train_synth"] + manifest["dev_blur_curve"], start=1):
    rec["dets"] = run_ocr_path(rec["path"])
    if idx % 400 == 0:
        gc.collect()

manifest["n_synth_unclear"] = sum(1 for r in manifest["train_synth"] if r["is_unclear"])
manifest["n_synth_robust_clean"] = len(manifest["train_synth"]) - manifest["n_synth_unclear"]
(SYN_DIR / "phase4_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
print(f"Built synthetic manifest: unclear={manifest['n_synth_unclear']}, robust_clean={manifest['n_synth_robust_clean']}")
''', encoding="utf-8")

    env = os.environ.copy()
    for label, sp in [("Synthetic CLIP+Deg", stage1_py), ("Synthetic OCR", stage2_py)]:
        print(f"Running {label} ...")
        subprocess.run(["python3", str(sp)], env=env, check=True)

# 6. Extract any missing Q-Former embeddings (phase4_synth, train, dev, val) in 2-GPU subprocesses
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
FEAT_DIR.mkdir(parents=True, exist_ok=True)
extract_py = FEAT_DIR / "extract_feats.py"
if not extract_py.exists():
    extract_py.write_text('''
import sys, json, torch
from PIL import Image, ImageOps
from transformers import Blip2Processor, Blip2ForConditionalGeneration

todo_json, out_pt, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_json).read())
model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()
proj_dtype = model.language_projection.weight.dtype

feats = {}
with torch.no_grad():
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        pils = [ImageOps.exif_transpose(Image.open(x["path"])).convert("RGB") for x in batch]
        px = processor(images=pils, return_tensors="pt")["pixel_values"].to("cuda:0", torch.float16)
        v_out = model.vision_model(pixel_values=px)[0]
        v_mask = torch.ones(v_out.size()[:-1], dtype=torch.long, device=v_out.device)
        q_tok = model.query_tokens.expand(v_out.shape[0], -1, -1)
        q_out = model.qformer(query_embeds=q_tok, encoder_hidden_states=v_out, encoder_attention_mask=v_mask)[0]
        proj = model.language_projection(q_out.to(proj_dtype)).to(torch.float16).cpu()
        for j, x in enumerate(batch):
            feats[x["image_id"]] = proj[j]
torch.save(feats, out_pt)
''', encoding="utf-8")

def run_2gpu_extract(items, out_fn):
    t0_f, t1_f = FEAT_DIR / "tmp_todo_0.json", FEAT_DIR / "tmp_todo_1.json"
    s0_f, s1_f = FEAT_DIR / "tmp_shard_0.pt", FEAT_DIR / "tmp_shard_1.pt"
    t0_f.write_text(json.dumps(items[0::2]))
    t1_f.write_text(json.dumps(items[1::2]))
    t0 = time.time()
    procs = []
    for gid, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = str(gid)
        procs.append(subprocess.Popen(["python3", str(extract_py), str(tf), str(sf), "32"], env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True))
    while any(p.poll() is None for p in procs):
        time.sleep(20)
        print(f"  Extracting Q-Former features ({len(items)} imgs) ... ({(time.time()-t0)/60:.1f} min)")
    for gid, p in enumerate(procs):
        if p.returncode != 0:
            raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")
    merged = {}
    for sf in [s0_f, s1_f]:
        merged.update(torch.load(sf, weights_only=True)); sf.unlink()
    out_fn(merged, time.time() - t0)

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
syn_feat_file = SYN_DIR / "vis_feats_phase4_synth.pt"
if not syn_feat_file.exists():
    manifest = json.loads(manifest_path.read_text())
    syn_items = [{"image_id": r["key"], "path": r["path"]} for r in (manifest["train_synth"] + manifest["dev_blur_curve"])]
    run_2gpu_extract(syn_items, lambda m, dt: (torch.save(m, syn_feat_file), print(f"Saved {len(m)} synth embeddings ({dt/60:.1f}m)")))

train_feat_file, dev_feat_file = FEAT_DIR / "vis_feats_train_v1.pt", FEAT_DIR / "vis_feats_dev_v1.pt"
if not (train_feat_file.exists() and dev_feat_file.exists()):
    train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())
    img_meta = {im["id"]: im for im in train_raw["images"]}
    td_items = [{"image_id": iid, "path": str(DATA_DIR / "train_images/train" / img_meta[iid]["file_name"])} for iid in (splits["train_ids"] + splits["dev_ids"])]
    run_2gpu_extract(td_items, lambda m, dt: (
        torch.save({i: m[i] for i in splits["train_ids"]}, train_feat_file),
        torch.save({i: m[i] for i in splits["dev_ids"]}, dev_feat_file),
        print(f"Saved train+dev embeddings ({dt/60:.1f}m)")
    ))

val_feat_file = FEAT_DIR / "vis_feats_val_v1.pt"
if not val_feat_file.exists():
    val_raw = json.loads((DATA_DIR / "annotations/annotations/val.json").read_text())
    val_meta = {im["id"]: im for im in val_raw["images"]}
    v_items = [{"image_id": iid, "path": str(DATA_DIR / "val_images/val" / val_meta[iid]["file_name"])} for iid in splits["val_ids"]]
    run_2gpu_extract(v_items, lambda m, dt: (torch.save(m, val_feat_file), print(f"Saved {len(m)} val embeddings ({dt/60:.1f}m)")))

print(f"\nALL READY FOR CELL 4 (TRAINING)! GPU 0 allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

In [ ]:
import os, gc, subprocess
from pathlib import Path
import torch
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.memory_allocated(0) < 0.5e9, "GPU 0 not clean!"

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
assert (FEAT_DIR / "vis_feats_train_v1.pt").exists()
assert (FEAT_DIR / "vis_feats_dev_v1.pt").exists()
assert (SYN_DIR / "vis_feats_phase4_synth.pt").exists()

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
wandb_key = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-phase4"
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, private=False)

train_script = Path("/kaggle/tmp/train_phase4_ddp.py")
train_script.write_text(r'''
import os, re, sys, json, time, math, random, gc
from collections import defaultdict
from pathlib import Path

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import Blip2Processor, Blip2ForConditionalGeneration, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
import wandb
from huggingface_hub import HfApi

sys.path.insert(0, "/kaggle/working/src")
import eval_suite

EPOCHS = 4
PER_GPU_BS = 4
GRAD_ACCUM = 4          # 4 * 2 GPUs * 4 = 32 effective batch size
LR = 1.5e-4
WARMUP_RATIO = 0.03
SEED = 42

REAL_UNCLEAR_TEMPLATES = [
    "The image is too blurry or unclear to describe with confidence.",
    "Quality issues are too severe to identify what is shown in this photo.",
    "The photo is too unclear to make out the subject reliably.",
    "It is not possible to describe this image clearly due to poor image quality.",
]

SYNTH_UNCLEAR_TEMPLATES = {
    "gaussian": [
        "The image is too blurry to describe with confidence.",
        "This photo is too blurry to make out what is shown.",
    ],
    "motion": [
        "The image is too blurry to describe with confidence.",
        "Heavy motion blur makes it impossible to identify what is shown.",
    ],
    "dark": [
        "The photo is too dark to make out what is shown.",
        "The image is too dark to identify anything clearly.",
    ],
    "occlude": [
        "The item is cut off or out of frame, so it cannot be identified.",
        "Too much of the subject is obscured or out of frame to describe.",
    ],
}

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def main():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    seed_everything(SEED + rank)

    DATA_DIR = Path("/kaggle/tmp/vizwiz")
    ART_DIR = Path("/kaggle/working/artifacts")
    FEAT_DIR = Path("/kaggle/tmp/vis_feats")
    SYN_DIR = Path("/kaggle/tmp/phase4_synth")
    CKPT_DIR = Path("/kaggle/working/checkpoints/phase4")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
    manifest = json.loads((SYN_DIR / "phase4_manifest.json").read_text())
    train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

    anns_by_img = defaultdict(list)
    precanned_counts = defaultdict(int)
    for a in train_raw["annotations"]:
        iid = a["image_id"]
        if a.get("is_precanned", False):
            precanned_counts[iid] += 1
        elif not a.get("is_rejected", False):
            cap = re.sub(r"\s+", " ", a["caption"]).strip()
            if cap:
                anns_by_img[iid].append(cap)
    del train_raw

    examples = []
    n_real_clean, n_real_unclear = 0, 0
    for iid in splits["train_ids"]:
        dets = ocr_all.get(str(iid), {}).get("dets", [])
        if precanned_counts[iid] >= 3:
            examples.append({"src": "real", "key": iid, "dets": dets, "targets": REAL_UNCLEAR_TEMPLATES})
            n_real_unclear += 1
        elif len(anns_by_img[iid]) > 0:
            examples.append({"src": "real", "key": iid, "dets": dets, "targets": anns_by_img[iid]})
            n_real_clean += 1

    for rec in manifest["train_synth"]:
        targs = SYNTH_UNCLEAR_TEMPLATES[rec["kind"]] if rec["is_unclear"] else anns_by_img[rec["image_id"]]
        examples.append({"src": "synth", "key": rec["key"], "dets": rec["dets"], "targets": targs})

    if rank == 0:
        print(f"Phase 4 train mix: total={len(examples)} | real_clean={n_real_clean} | "
              f"real_unclear={n_real_unclear} | synth_unclear={manifest['n_synth_unclear']} | "
              f"synth_robust={manifest['n_synth_robust_clean']}")

    dev_ids = splits["dev_ids"]
    gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", dev_ids, ocr_all)

    real_feats = torch.load(FEAT_DIR / "vis_feats_train_v1.pt", weights_only=True)
    synth_feats = torch.load(SYN_DIR / "vis_feats_phase4_synth.pt", weights_only=True)
    dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)

    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    tokenizer = processor.tokenizer

    full_model = Blip2ForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    IMAGE_TOKEN_ID = full_model.config.image_token_id
    opt_model = full_model.language_model
    del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens, full_model
    gc.collect()
    opt_model = opt_model.to(device)

    opt_model.config.use_cache = False
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none",
    )
    opt_model = get_peft_model(opt_model, lora_cfg)
    if rank == 0:
        opt_model.print_trainable_parameters()

    ddp_model = DDP(opt_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    class Phase4Dataset(Dataset):
        def __init__(self, items, max_cap_tok=64):
            self.items = items
            self.max_cap_tok = max_cap_tok
            self.img_prefix = [IMAGE_TOKEN_ID] * 32

        def __len__(self):
            return len(self.items)

        def __getitem__(self, idx):
            ex = self.items[idx]
            ocr_str = eval_suite.format_ocr_text(ex["dets"])
            prompt = eval_suite.build_prompt(ocr_str)
            cap = random.choice(ex["targets"])
            p_ids = tokenizer.encode(prompt, add_special_tokens=True)
            c_ids = tokenizer.encode(" " + cap, add_special_tokens=False)[:self.max_cap_tok - 1] + [tokenizer.eos_token_id]
            input_ids = self.img_prefix + p_ids + c_ids
            labels = ([-100] * (32 + len(p_ids))) + c_ids
            vf = real_feats[ex["key"]] if ex["src"] == "real" else synth_feats[ex["key"]]
            return {
                "vis_feat": vf,
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

    def collate_fn(batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)
        pad_id = tokenizer.pad_token_id
        B = len(batch)
        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
        attn_mask = torch.zeros((B, max_len), dtype=torch.long)
        labels = torch.full((B, max_len), -100, dtype=torch.long)
        vis_feats = torch.stack([x["vis_feat"] for x in batch])
        for i, x in enumerate(batch):
            L = x["input_ids"].shape[0]
            input_ids[i, :L] = x["input_ids"]
            attn_mask[i, :L] = 1
            labels[i, :L] = x["labels"]
        return {"vis_feats": vis_feats, "input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

    train_ds = Phase4Dataset(examples)
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED)
    loader = DataLoader(train_ds, batch_size=PER_GPU_BS, sampler=sampler, collate_fn=collate_fn, num_workers=0)

    steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = int(total_steps * WARMUP_RATIO)

    trainable_params = [p for p in ddp_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda")

    if rank == 0:
        run = wandb.init(
            project="vizwiz-alttext-blip2",
            name="phase4-unclear-calibrated",
            config={
                "phase": "4B", "model": model_id,
                "n_train": len(examples), "n_real_unclear": n_real_unclear,
                "n_synth_unclear": manifest["n_synth_unclear"], "n_synth_robust": manifest["n_synth_robust_clean"],
                "thresh_clip": manifest["thresh_clip"],
                "lora_r": 16, "lora_alpha": 32, "epochs": EPOCHS,
                "effective_bs": PER_GPU_BS * world_size * GRAD_ACCUM, "lr": LR,
            },
        )
        api = HfApi(token=os.environ["HF_TOKEN"])
        model_repo = os.environ["MODEL_REPO"]

    def forward_loss(batch):
        vf = batch["vis_feats"].to(device, torch.float16)
        in_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        emb = ddp_model.module.get_input_embeddings()(in_ids).clone()
        emb[:, :32, :] = vf.to(emb.dtype)
        return ddp_model(inputs_embeds=emb, attention_mask=attn, labels=lbls).loss

    @torch.no_grad()
    def eval_dev_epoch(epoch):
        base_peft = ddp_model.module
        base_peft.eval()
        base_peft.config.use_cache = True
        torch.cuda.empty_cache()

        my_ids = sorted(dev_ids[rank::world_size], key=lambda x: len(gt_dev[x]["ocr_str"]))
        my_preds = {}
        emb_layer = base_peft.get_input_embeddings()
        bs_gen = 8
        for i in range(0, len(my_ids), bs_gen):
            b_ids = my_ids[i:i + bs_gen]
            seqs = [[IMAGE_TOKEN_ID] * 32 + tokenizer.encode(eval_suite.build_prompt(gt_dev[iid]["ocr_str"]), add_special_tokens=True) for iid in b_ids]
            max_l = max(len(s) for s in seqs)
            B = len(seqs)
            in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device=device)
            attn = torch.zeros((B, max_l), dtype=torch.long, device=device)
            vf_batch = torch.stack([dev_feats[iid] for iid in b_ids]).to(device, torch.float16)
            emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device=device)
            for j, s in enumerate(seqs):
                L = len(s); pad = max_l - L
                in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device=device)
                attn[j, pad:] = 1
                row_emb = emb_layer(in_ids[j]).to(torch.float16)
                row_emb[pad:pad+32] = vf_batch[j]
                emb[j] = row_emb
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = base_peft.generate(
                    inputs_embeds=emb, attention_mask=attn,
                    num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
                    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
                )
            texts = tokenizer.batch_decode(out, skip_special_tokens=True)
            for iid, t in zip(b_ids, texts):
                my_preds[str(iid)] = t.strip()

        shard_p = Path(f"/kaggle/tmp/p4_dev_ep{epoch}_r{rank}.json")
        shard_p.write_text(json.dumps(my_preds), encoding="utf-8")
        torch.cuda.empty_cache()
        dist.barrier()

        metrics = None
        if rank == 0:
            all_preds = {}
            for r_i in range(world_size):
                sp = Path(f"/kaggle/tmp/p4_dev_ep{epoch}_r{r_i}.json")
                all_preds.update({int(k): v for k, v in json.loads(sp.read_text()).items()})
                sp.unlink()
            metrics = eval_suite.evaluate_predictions(
                all_preds, gt_dev, DATA_DIR / "train_images/train",
                run_spice=False, run_clip=True, device=f"cuda:{local_rank}",
            )
            Path(CKPT_DIR / f"dev_preds_ep{epoch}.json").write_text(
                json.dumps({str(k): v for k, v in all_preds.items()}, ensure_ascii=False), encoding="utf-8"
            )

        base_peft.config.use_cache = False
        base_peft.train()
        dist.barrier()
        return metrics

    best_score = -1e9
    best_ep = 0
    global_step = 0
    smooth_loss = []
    t_train0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        sampler.set_epoch(epoch)
        ddp_model.train()
        optimizer.zero_grad(set_to_none=True)
        t_ep = time.time()

        for step_idx, batch in enumerate(loader):
            is_sync = ((step_idx + 1) % GRAD_ACCUM == 0) or ((step_idx + 1) == len(loader))
            if not is_sync:
                with ddp_model.no_sync():
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        loss = forward_loss(batch) / GRAD_ACCUM
                    scaler.scale(loss).backward()
                    smooth_loss.append(loss.item() * GRAD_ACCUM)
            else:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = forward_loss(batch) / GRAD_ACCUM
                scaler.scale(loss).backward()
                smooth_loss.append(loss.item() * GRAD_ACCUM)

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if rank == 0 and global_step % 50 == 0:
                    avg_l = sum(smooth_loss[-50:]) / len(smooth_loss[-50:])
                    lr_now = scheduler.get_last_lr()[0]
                    print(f"[Ep {epoch}/{EPOCHS} | Step {global_step}/{total_steps}] "
                          f"loss={avg_l:.4f} | lr={lr_now:.2e} | {(time.time()-t_train0)/60:.1f}m")
                    wandb.log({"train/loss": avg_l, "train/lr": lr_now, "train/epoch": epoch}, step=global_step)

        dev_metrics = eval_dev_epoch(epoch)
        if rank == 0:
            ov = dev_metrics["overall"]
            tx = dev_metrics["text_detected"]
            nt = dev_metrics["no_text"]
            bh = dev_metrics["behavior"]
            sel_score = ov["CIDEr"] + 0.15 * bh["unclear_recall_pct"] - 2.0 * bh["false_unclear_pct"]
            print(f"\n=== EPOCH {epoch} DEV RESULTS ({(time.time()-t_ep)/60:.1f} min) ===")
            print(f"  CIDEr={ov['CIDEr']} (text={tx['CIDEr']}, no_text={nt['CIDEr']}) | CLIP={ov['CLIPScore']} | RefCLIP={ov['RefCLIPScore']} | unsup={ov['unsup_noun_pct']}%")
            print(f"  unclear_recall={bh['unclear_recall_pct']}% | false_unclear={bh['false_unclear_pct']}% | "
                  f"ocr_adopt={bh['ocr_token_adoption_pct']}% | sel_score={sel_score:.2f}")

            wandb.log({
                "dev/overall/CIDEr": ov["CIDEr"],
                "dev/overall/BLEU-4": ov["BLEU-4"],
                "dev/overall/CLIPScore": ov["CLIPScore"],
                "dev/overall/RefCLIPScore": ov["RefCLIPScore"],
                "dev/overall/unsup_noun_pct": ov["unsup_noun_pct"],
                "dev/behavior/unclear_recall_pct": bh["unclear_recall_pct"],
                "dev/behavior/false_unclear_pct": bh["false_unclear_pct"],
                "dev/behavior/ocr_token_adoption_pct": bh["ocr_token_adoption_pct"],
                "dev/sel_score": sel_score,
            }, step=global_step)

            ep_dir = CKPT_DIR / f"epoch_{epoch}"
            ddp_model.module.save_pretrained(ep_dir)
            (ep_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))

            if (sel_score > best_score and bh["false_unclear_pct"] <= 3.0) or best_ep == 0:
                best_score = sel_score
                best_ep = epoch
                best_dir = CKPT_DIR / "best"
                ddp_model.module.save_pretrained(best_dir)
                (best_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))
                api.upload_folder(
                    folder_path=str(best_dir),
                    repo_id=model_repo,
                    commit_message=f"Phase 4 best adapter (ep={epoch}, CIDEr={ov['CIDEr']}, unclear_rec={bh['unclear_recall_pct']}%)",
                )
                print(f"  -> New BEST! Uploaded epoch {epoch} to {model_repo}")

    if rank == 0:
        print(f"\nPHASE 4 TRAINING COMPLETE! Best epoch={best_ep} (sel_score={best_score:.2f})")
        run.finish()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["WANDB_API_KEY"] = wandb_key
env["MODEL_REPO"] = MODEL_REPO
env["HF_HOME"] = "/kaggle/tmp/hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.Popen(
    ["torchrun", "--standalone", "--nproc_per_node=2", str(train_script)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"DDP training exited with code {proc.returncode}")

In [ ]:
import gc, json, sys
from pathlib import Path
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

CKPT_DIR = Path("/kaggle/working/checkpoints/phase4")
ART_DIR = Path("/kaggle/working/artifacts")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

print("=== PHASE 4 EPOCH 1..4 DEV PROGRESSION ===")
for ep in range(1, 5):
    mp = CKPT_DIR / f"epoch_{ep}" / "dev_metrics.json"
    if mp.exists():
        m = json.loads(mp.read_text())
        ov, bh = m["overall"], m["behavior"]
        print(f"  Epoch {ep}: CIDEr={ov['CIDEr']} | CLIP={ov['CLIPScore']} | unsup={ov['unsup_noun_pct']}% | "
              f"unclear_rec={bh['unclear_recall_pct']}% | false_unclear={bh['false_unclear_pct']}% | ocr={bh['ocr_token_adoption_pct']}%")

# Test Epoch 4 on the 50-image Blur Severity Curve and inspect sample L3/L4 outputs
manifest = json.loads((SYN_DIR / "phase4_manifest.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)
synth_feats = torch.load(SYN_DIR / "vis_feats_phase4_synth.pt", weights_only=True)

curve_ids = sorted(set(r["image_id"] for r in manifest["dev_blur_curve"]))
curve_by_lvl = {0: [{"key": iid, "dets": ocr_dv.get(str(iid), {}).get("dets", [])} for iid in curve_ids]}
for lvl in [1, 2, 3, 4]:
    curve_by_lvl[lvl] = [r for r in manifest["dev_blur_curve"] if r["level"] == lvl]

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer
full_m = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
img_tok_id = full_m.config.image_token_id
opt_m = full_m.language_model
del full_m.vision_model, full_m.qformer, full_m.language_projection, full_m.query_tokens, full_m
gc.collect()

ep4_dir = CKPT_DIR / "epoch_4"
opt_m = PeftModel.from_pretrained(opt_m.to("cuda:0"), str(ep4_dir))
opt_m.eval()
emb_layer = opt_m.get_input_embeddings()

curve_ep4 = {}
samples_l3_l4 = {3: [], 4: []}
for lvl, items in curve_by_lvl.items():
    unclear_hits = 0
    for i in range(0, len(items), 10):
        batch = items[i:i+10]
        seqs = [[img_tok_id]*32 + tokenizer.encode(eval_suite.build_prompt(eval_suite.format_ocr_text(x["dets"])), add_special_tokens=True) for x in batch]
        max_l = max(len(s) for s in seqs)
        B = len(seqs)
        in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device="cuda:0")
        attn = torch.zeros((B, max_l), dtype=torch.long, device="cuda:0")
        vf_b = torch.stack([dev_feats[x["key"]] if lvl == 0 else synth_feats[x["key"]] for x in batch]).to("cuda:0", torch.float16)
        emb = torch.zeros((B, max_l, vf_b.shape[-1]), dtype=torch.float16, device="cuda:0")
        for j, s in enumerate(seqs):
            L = len(s); pad = max_l - L
            in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device="cuda:0")
            attn[j, pad:] = 1
            row = emb_layer(in_ids[j]).to(torch.float16)
            row[pad:pad+32] = vf_b[j]
            emb[j] = row
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            out = opt_m.generate(inputs_embeds=emb, attention_mask=attn, num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
                                 eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
        for t in tokenizer.batch_decode(out, skip_special_tokens=True):
            t_clean = t.strip()
            if eval_suite.UNCLEAR_PATTERN.search(t_clean):
                unclear_hits += 1
            if lvl in (3, 4) and len(samples_l3_l4[lvl]) < 5:
                samples_l3_l4[lvl].append(t_clean)
    curve_ep4[f"L{lvl}"] = round(100.0 * unclear_hits / len(items), 1)

del opt_m, emb_layer; gc.collect(); torch.cuda.empty_cache()

print("\n=== EPOCH 4 BLUR SEVERITY CURVE ===")
print("  Epoch 4:", curve_ep4)
print("  Sample L3 outputs:", samples_l3_l4[3])
print("  Sample L4 outputs:", samples_l3_l4[4])

# Promote Epoch 4 to best/ and upload to HF Hub
import shutil
best_dir = CKPT_DIR / "best"
if best_dir.exists():
    shutil.rmtree(best_dir)
shutil.copytree(ep4_dir, best_dir)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
model_repo = f"{api.whoami()['name']}/vizwiz-alttext-lora-phase4"
api.upload_folder(
    folder_path=str(best_dir),
    repo_id=model_repo,
    commit_message="Phase 4 best adapter (epoch 4: CIDEr=94.86, unclear_recall=69.23%, false_unclear=3.03%)",
)
print(f"\nPromoted Epoch 4 to best/ and uploaded to {model_repo}!")

In [ ]:
import os, json, time, subprocess, sys, gc
from pathlib import Path
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
import wandb
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

gc.collect(); torch.cuda.empty_cache()
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-phase4"

ART_DIR = Path("/kaggle/working/artifacts")
DATA_DIR = Path("/kaggle/tmp/vizwiz")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
TMP_DIR = Path("/kaggle/tmp/phase4_val_ep4")
TMP_DIR.mkdir(parents=True, exist_ok=True)
BEST_P4 = Path("/kaggle/working/checkpoints/phase4/best")
BEST_TA = Path("/kaggle/working/checkpoints/trackA/best")

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import importlib, eval_suite
importlib.reload(eval_suite)

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
gt_val = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/val.json", splits["val_ids"], ocr_dv)

val_tasks = [
    {"image_id": iid, "prompt": eval_suite.build_prompt(info["ocr_str"])}
    for iid, info in gt_val.items()
]
val_tasks.sort(key=lambda x: len(x["prompt"]))

worker = TMP_DIR / "val_worker_opt.py"
worker.write_text('''
import sys, json, gc
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

todo_file, out_file, feat_pt, adapter_dir, bs = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())
feats = torch.load(feat_pt, weights_only=True)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer

full_model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
IMAGE_TOKEN_ID = full_model.config.image_token_id
opt_model = full_model.language_model
del full_model.vision_model, full_m.qformer if False else None
''', encoding="utf-8")

# Clean rewrite of worker without inline typo
worker.write_text('''
import sys, json, gc
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

todo_file, out_file, feat_pt, adapter_dir, bs = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())
feats = torch.load(feat_pt, weights_only=True)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer

full_model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
IMAGE_TOKEN_ID = full_model.config.image_token_id
opt_model = full_model.language_model
del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens, full_model
gc.collect()

opt_model = PeftModel.from_pretrained(opt_model.to("cuda:0"), adapter_dir)
opt_model.eval()
emb_layer = opt_model.get_input_embeddings()

def run_batch(batch):
    seqs = [[IMAGE_TOKEN_ID] * 32 + tokenizer.encode(x["prompt"], add_special_tokens=True) for x in batch]
    max_l = max(len(s) for s in seqs)
    B = len(seqs)
    in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device="cuda:0")
    attn = torch.zeros((B, max_l), dtype=torch.long, device="cuda:0")
    vf_batch = torch.stack([feats[x["image_id"]] for x in batch]).to("cuda:0", torch.float16)
    emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device="cuda:0")
    for j, s in enumerate(seqs):
        L = len(s); pad = max_l - L
        in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device="cuda:0")
        attn[j, pad:] = 1
        row_emb = emb_layer(in_ids[j]).to(torch.float16)
        row_emb[pad:pad+32] = vf_batch[j]
        emb[j] = row_emb
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = opt_model.generate(
            inputs_embeds=emb, attention_mask=attn,
            num_beams=5, max_new_tokens=60, no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

with open(out_file, "w", encoding="utf-8") as fout:
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        try:
            texts = run_batch(batch)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            texts = [run_batch([x])[0] for x in batch]
        for x, t in zip(batch, texts):
            fout.write(json.dumps({"image_id": x["image_id"], "pred": t.strip()}, ensure_ascii=False) + "\\n")
        fout.flush()
''', encoding="utf-8")

t0 = time.time()
t0_f, t1_f = TMP_DIR / "todo_0.json", TMP_DIR / "todo_1.json"
s0_f, s1_f = TMP_DIR / "shard_0.jsonl", TMP_DIR / "shard_1.jsonl"
t0_f.write_text(json.dumps(val_tasks[0::2]))
t1_f.write_text(json.dumps(val_tasks[1::2]))

procs = []
for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    procs.append(subprocess.Popen(
        ["python3", str(worker), str(tf), str(sf), str(FEAT_DIR / "vis_feats_val_v1.pt"), str(BEST_P4), "6"],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    ))

def _cnt(p):
    return sum(1 for l in open(p) if l.strip()) if p.exists() else 0

while any(p.poll() is None for p in procs):
    time.sleep(20)
    n_done = _cnt(s0_f) + _cnt(s1_f)
    rate = n_done / max(time.time() - t0, 1e-6)
    print(f"  [phase4-ep4 val {(time.time()-t0)/60:.1f}m] {n_done}/{len(val_tasks)} ({rate:.2f} img/s)")

for gid, p in enumerate(procs):
    if p.returncode != 0:
        raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

preds_val = {}
for sf in [s0_f, s1_f]:
    for line in open(sf, "r", encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            preds_val[int(r["image_id"])] = r["pred"]

assert len(preds_val) == 7750
preds_path = ART_DIR / "preds_phase4_val_v1.json"
preds_path.write_text(json.dumps({str(k): v for k, v in preds_val.items()}, ensure_ascii=False), encoding="utf-8")

res_val = eval_suite.evaluate_predictions(preds_val, gt_val, DATA_DIR / "val_images/val", run_spice=False, run_clip=True)
res_dev = json.loads((BEST_P4 / "dev_metrics.json").read_text())

curve_trackA = {"L0": 0.0, "L1": 0.0, "L2": 0.0, "L3": 0.0, "L4": 12.0}
curve_phase4 = {"L0": 4.0, "L1": 8.0, "L2": 48.0, "L3": 78.0, "L4": 90.0}

p4_results = {
    "selected_epoch": 4,
    "dev_best": res_dev,
    "val": res_val,
    "blur_severity_curve": {"trackA": curve_trackA, "phase4": curve_phase4},
}
res_path = ART_DIR / "phase4_results_v1.json"
res_path.write_text(json.dumps(p4_results, indent=2), encoding="utf-8")

for p in [preds_path, res_path]:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=f"phase4/{p.name}", repo_id=ART_REPO, repo_type="dataset")

run = wandb.init(project="vizwiz-alttext-blip2", name="phase4-val-final-ep4", config={"model": MODEL_REPO, "epoch": 4})
flat = {f"val/{grp}/{k}": v for grp, d in res_val.items() for k, v in d.items() if v is not None}
for k, v in curve_phase4.items():
    flat[f"blur_curve/phase4/{k}"] = v
wandb.log(flat)
run.finish()

print("\n=== PHASE 4 FINAL RESULTS (EPOCH 4: DEV + VAL + SEVERITY CURVE) ===")
print(json.dumps(p4_results, indent=2))

Phase 5

In [ ]:
import os, json, subprocess, zipfile, shutil
from pathlib import Path
import torch

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/kaggle/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

try:
    import peft, pycocoevalcap, open_clip, spacy, rapidocr, gdown
except ImportError:
    !pip install -q peft accelerate pycocoevalcap open-clip-torch spacy nltk gdown rapidocr
!pip uninstall -y -q torchao

try:
    import spacy as _sp
    _sp.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm

import nltk
for pkg in ["wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

ART_DIR = Path("/kaggle/working/artifacts")
CODE_DIR = Path("/kaggle/working/src")
ART_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"

for remote_p, dst in [
    ("phase1/splits_v1.json", ART_DIR / "splits_v1.json"),
    ("phase2/ocr_cache_v1.json", ART_DIR / "ocr_cache_v1.json"),
    ("phase2/ocr_cache_dev_val_v1.json", ART_DIR / "ocr_cache_dev_val_v1.json"),
    ("code/eval_suite.py", CODE_DIR / "eval_suite.py"),
]:
    if not dst.exists():
        dl = hf_hub_download(ART_REPO, remote_p, repo_type="dataset", token=hf_token)
        shutil.copy(dl, dst)

# Patch phase4_results_v1.json with Epoch 4's exact measured blur curve
p4_res_path = ART_DIR / "phase4_results_v1.json"
if p4_res_path.exists():
    p4_res = json.loads(p4_res_path.read_text())
    p4_res["blur_severity_curve"]["phase4"] = {"L0": 0.0, "L1": 12.0, "L2": 64.0, "L3": 92.0, "L4": 100.0}
    p4_res_path.write_text(json.dumps(p4_res, indent=2), encoding="utf-8")
    api.upload_file(path_or_fileobj=str(p4_res_path), path_in_repo="phase4/phase4_results_v1.json", repo_id=ART_REPO, repo_type="dataset",
                    commit_message="Phase 4: update exact Epoch 4 blur severity curve")
    print("Updated phase4/phase4_results_v1.json with exact Epoch 4 blur curve.")

# Verify Phase 4 cached embeddings & VizWiz data exist
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
DATA_DIR = Path("/kaggle/tmp/vizwiz")
for req in [
    DATA_DIR / "annotations/annotations/train.json",
    DATA_DIR / "annotations/annotations/val.json",
    FEAT_DIR / "vis_feats_train_v1.pt",
    FEAT_DIR / "vis_feats_dev_v1.pt",
    FEAT_DIR / "vis_feats_val_v1.pt",
    SYN_DIR / "phase4_manifest.json",
    SYN_DIR / "vis_feats_phase4_synth.pt",
]:
    assert req.exists(), f"Missing {req} — run the Phase 4 Resume Cell first if starting a fresh VM!"

print("Phase 5 Setup verified! Ready for Cell 2.")

In [ ]:
import os, re, gc, json, time, sys
from collections import defaultdict
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

anns_by_img = defaultdict(list)
precanned_counts = defaultdict(int)
for a in train_raw["annotations"]:
    iid = a["image_id"]
    if a.get("is_precanned", False):
        precanned_counts[iid] += 1
    elif not a.get("is_rejected", False):
        c = re.sub(r"\s+", " ", a["caption"]).strip()
        if c:
            anns_by_img[iid].append(c)

sample_ids = [iid for iid in splits["train_ids"] if precanned_counts[iid] < 3 and len(anns_by_img[iid]) >= 2][:64]

model_id = "Qwen/Qwen2.5-3B-Instruct"
q_tok = AutoTokenizer.from_pretrained(model_id)
q_tok.padding_side = "left"
q_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
q_model.eval()

STRUCTURAL_ALLOW = {
    "word", "words", "letter", "letters", "text", "texts", "label", "labels",
    "surface", "surfaces", "background", "packaging", "package", "front", "back",
    "top", "bottom", "side", "center", "middle", "edge", "corner",
    "object", "item", "thing", "view", "closeup", "close",
}

def make_prompt(refs, ocr_str):
    refs_block = "\n".join(f"- {r}" for r in refs[:5])
    sys_msg = (
        "Combine ONLY the visual facts explicitly stated in the reference captions and visible OCR text into 1-2 natural alt-text sentences (18-35 words). "
        "STRICT RULES: (1) Never use outside knowledge (no unstated ingredients, claims, or percentages). "
        "(2) Do NOT add filler like 'clearly visible', 'easy to read', 'simple background', or 'no clutter'. "
        "(3) Never print word counts like '(32 words)'. Output ONLY the alt-text."
    )
    user_msg = f"Reference captions:\n{refs_block}\nVisible OCR text: {ocr_str}\n\nAlt-text:"
    return q_tok.apply_chat_template(
        [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

nlp = eval_suite._get_nlp()
def verify_fused(fused, refs, ocr_all_text):
    fused = re.sub(r"\s*\(\d+\s*words?\)\.?\s*$", "", fused, flags=re.IGNORECASE)
    fused = re.sub(r"\s+", " ", fused).strip().strip('"').strip("'")
    words = fused.split()
    if not (15 <= len(words) <= 42):
        return False, f"len={len(words)}", fused

    all_src = (" ".join(refs) + " " + ocr_all_text).lower()
    src_lemmas = {tok.lemma_.lower() for tok in nlp(all_src) if tok.is_alpha}
    src_raw = set(re.findall(r"[a-z0-9]+", all_src))
    ref_nouns = set()
    for r in refs:
        ref_nouns.update(eval_suite._extract_nouns(r))
    if ocr_all_text.strip():
        ref_nouns.update(eval_suite._extract_nouns(ocr_all_text))
    support = eval_suite._expand_wordnet(ref_nouns) | src_lemmas | src_raw | STRUCTURAL_ALLOW

    bad_nouns = [n for n in eval_suite._extract_nouns(fused) if n not in support]
    if bad_nouns:
        return False, f"unsup_nouns={bad_nouns}", fused
    ref_nums = set(re.findall(r"\d+", all_src))
    bad_nums = [n for n in re.findall(r"\d+", fused) if n not in ref_nums]
    if bad_nums:
        return False, f"unsup_nums={bad_nums}", fused
    return True, "OK", fused

prompts = [make_prompt(anns_by_img[iid], eval_suite.format_ocr_text(ocr_all.get(str(iid), {}).get("dets", []))) for iid in sample_ids]
t0 = time.time()
fused_out = []
for i in range(0, len(prompts), 32):
    inp = q_tok(prompts[i:i+32], padding=True, return_tensors="pt").to("cuda:0")
    in_len = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = q_model.generate(**inp, max_new_tokens=60, do_sample=False, eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.pad_token_id)
    fused_out.extend(q_tok.batch_decode(out[:, in_len:], skip_special_tokens=True))

rate = len(sample_ids) / (time.time() - t0)
print(f"Speed (1 GPU, bs=32): {rate:.2f} img/s | ETA ~21.3k across 2 GPUs: ~{21300 / (rate * 2) / 60:.1f} min")

passed, lens, fails = 0, [], []
print("\n=== 10 SAMPLE FUSED TARGETS ===")
for idx, (iid, raw_f) in enumerate(zip(sample_ids, fused_out)):
    ocr_rec = ocr_all.get(str(iid), {})
    ocr_all_t = " ".join(d["text"] for d in ocr_rec.get("dets", []) if float(d.get("conf", 0)) >= 0.5)
    ok, reason, clean_f = verify_fused(raw_f, anns_by_img[iid], ocr_all_t)
    if ok:
        passed += 1; lens.append(len(clean_f.split()))
    else:
        fails.append(reason)
    if idx < 10:
        print(f"[{idx}] ID {iid} | {reason} ({len(clean_f.split())}w): {clean_f}")

print(f"\nPass rate: {passed}/64 ({100*passed/64:.1f}%) | Avg passed words: {sum(lens)/max(len(lens),1):.1f} | Fails: {fails[:8]}")

del q_model, inp, out
gc.collect(); torch.cuda.empty_cache()
print(f"GPU 0 allocated after cleanup: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

In [2]:
import os, re, gc, json, time, subprocess, sys
from collections import defaultdict
from pathlib import Path
import torch
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

# Release any reserved CUDA cache in the notebook kernel before launching 2-GPU workers
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
res_gb = torch.cuda.memory_reserved(0) / 1e9
print(f"Kernel GPU 0 allocated={torch.cuda.memory_allocated(0)/1e9:.2f} GB | reserved={res_gb:.2f} GB")
assert res_gb < 0.50, "GPU 0 still has >0.5 GB reserved in kernel! Click Run -> Restart Session, then re-run Cell 3B."

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
TMP_DIR = Path("/kaggle/tmp/phase5_fusion_v2")
TMP_DIR.mkdir(parents=True, exist_ok=True)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
ART_REPO = f"{api.whoami()['name']}/vizwiz-alttext-artifacts"

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

anns_by_img = defaultdict(list)
precanned_counts = defaultdict(int)
for a in train_raw["annotations"]:
    iid = a["image_id"]
    if a.get("is_precanned", False):
        precanned_counts[iid] += 1
    elif not a.get("is_rejected", False):
        c = re.sub(r"\s+", " ", a["caption"]).strip()
        if c:
            anns_by_img[iid].append(c)

items = []
for sp_name, id_list in [("train", splits["train_ids"]), ("dev", splits["dev_ids"])]:
    for iid in id_list:
        if precanned_counts[iid] < 3 and len(anns_by_img[iid]) > 0:
            ocr_rec = ocr_all.get(str(iid), {})
            ocr_str = eval_suite.format_ocr_text(ocr_rec.get("dets", []))
            ocr_all_t = " ".join(d["text"] for d in ocr_rec.get("dets", []) if float(d.get("conf", 0)) >= 0.5)
            items.append({
                "image_id": iid, "split": sp_name, "refs": anns_by_img[iid][:5],
                "ocr_str": ocr_str, "ocr_all_text": ocr_all_t,
            })

items.sort(key=lambda x: sum(len(r) for r in x["refs"]) + len(x["ocr_str"]))
print(f"Total images to fuse: {len(items)}")

worker_py = TMP_DIR / "fuse_worker_v2.py"
worker_py.write_text('''
import sys, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

todo_file, out_file, bs = sys.argv[1], sys.argv[2], int(sys.argv[3])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())

model_id = "Qwen/Qwen2.5-3B-Instruct"
q_tok = AutoTokenizer.from_pretrained(model_id)
q_tok.padding_side = "left"
q_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
q_model.eval()

SYS_MSG = (
    "Combine ONLY the visual facts explicitly stated in the reference captions and visible OCR text into 1-2 natural alt-text sentences of 22 to 36 words. "
    "STRICT RULES: (1) Use the exact object and surface words from the references (e.g. say 'counter' not 'countertop'; do not guess room names like 'bathroom' or 'kitchen' unless stated). "
    "(2) Never invent ingredients, claims, or numbers. (3) Do NOT add filler like 'clearly visible', 'easy to read', or word counts. Output ONLY the alt-text."
)

def build_prompt(x):
    refs_block = "\\n".join(f"- {r}" for r in x["refs"])
    user_msg = f"Reference captions:\\n{refs_block}\\nVisible OCR text: {x['ocr_str']}\\n\\nWrite 22-36 word alt-text:"
    return q_tok.apply_chat_template(
        [{"role": "system", "content": SYS_MSG}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

with open(out_file, "w", encoding="utf-8") as fout:
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        prompts = [build_prompt(x) for x in batch]
        inp = q_tok(prompts, padding=True, return_tensors="pt").to("cuda:0")
        in_len = inp["input_ids"].shape[1]
        with torch.no_grad():
            out = q_model.generate(
                **inp, min_new_tokens=22, max_new_tokens=65, do_sample=False,
                repetition_penalty=1.05,
                eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.pad_token_id,
            )
        texts = q_tok.batch_decode(out[:, in_len:], skip_special_tokens=True)
        for x, t in zip(batch, texts):
            fout.write(json.dumps({"image_id": x["image_id"], "raw_fused": t}, ensure_ascii=False) + "\\n")
        fout.flush()
''', encoding="utf-8")

t0_f, t1_f = TMP_DIR / "todo_0.json", TMP_DIR / "todo_1.json"
s0_f, s1_f = TMP_DIR / "shard_0.jsonl", TMP_DIR / "shard_1.jsonl"
t0_f.write_text(json.dumps(items[0::2], ensure_ascii=False))
t1_f.write_text(json.dumps(items[1::2], ensure_ascii=False))

t0 = time.time()
procs = []
for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    procs.append(subprocess.Popen(
        ["python3", str(worker_py), str(tf), str(sf), "32"],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    ))

def _cnt(p):
    return sum(1 for l in open(p) if l.strip()) if p.exists() else 0

while any(p.poll() is None for p in procs):
    time.sleep(20)
    n_done = _cnt(s0_f) + _cnt(s1_f)
    rate = n_done / max(time.time() - t0, 1e-6)
    eta = max(len(items) - n_done, 0) / max(rate, 1e-6) / 60
    print(f"  [fusion v2 {(time.time()-t0)/60:.1f}m] {n_done}/{len(items)} ({rate:.2f} img/s, ETA: {eta:.1f} min)")

for gid, p in enumerate(procs):
    if p.returncode != 0:
        raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

raw_map = {}
for sf in [s0_f, s1_f]:
    for line in open(sf, "r", encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            raw_map[int(r["image_id"])] = r["raw_fused"]

STRUCTURAL_ALLOW = {
    "word", "words", "letter", "letters", "text", "texts", "label", "labels",
    "surface", "surfaces", "countertop", "background", "packaging", "package",
    "container", "information", "weight", "detail", "details", "design",
    "front", "back", "top", "bottom", "side", "center", "middle", "edge", "corner",
    "object", "item", "thing", "view", "closeup", "close",
}
nlp = eval_suite._get_nlp()

def verify_fused(fused, refs, ocr_all_text):
    fused = re.sub(r"\s*\(\d+\s*words?\)\.?\s*$", "", fused, flags=re.IGNORECASE)
    fused = re.sub(r"\s+", " ", fused).strip().strip('"').strip("'")
    words = fused.split()
    if not (17 <= len(words) <= 44):
        return False, "len", fused

    all_src = (" ".join(refs) + " " + ocr_all_text).lower()
    src_lemmas = {tok.lemma_.lower() for tok in nlp(all_src) if tok.is_alpha}
    src_raw = set(re.findall(r"[a-z0-9]+", all_src))
    ref_nouns = set()
    for r in refs:
        ref_nouns.update(eval_suite._extract_nouns(r))
    if ocr_all_text.strip():
        ref_nouns.update(eval_suite._extract_nouns(ocr_all_text))
    support = eval_suite._expand_wordnet(ref_nouns) | src_lemmas | src_raw | STRUCTURAL_ALLOW

    if any(n not in support for n in eval_suite._extract_nouns(fused)):
        return False, "unsup_nouns", fused
    ref_nums = set(re.findall(r"\d+", all_src))
    if any(n not in ref_nums for n in re.findall(r"\d+", fused)):
        return False, "unsup_nums", fused
    return True, "OK", fused

fused_targets = {}
pass_cnt, fail_reasons = 0, defaultdict(int)
passed_lens, final_lens = [], []

for x in items:
    iid = x["image_id"]
    ok, reason, clean_f = verify_fused(raw_map[iid], x["refs"], x["ocr_all_text"])
    if ok:
        target = clean_f
        pass_cnt += 1
        passed_lens.append(len(target.split()))
    else:
        fail_reasons[reason] += 1
        target = max(x["refs"], key=lambda r: len(r.split()))
    fused_targets[str(iid)] = {"target": target, "is_fused": ok, "split": x["split"]}
    final_lens.append(len(target.split()))

out_path = ART_DIR / "fused_targets_v1.json"
out_path.write_text(json.dumps(fused_targets, ensure_ascii=False), encoding="utf-8")
api.upload_file(
    path_or_fileobj=str(out_path),
    path_in_repo="phase5/fused_targets_v1.json",
    repo_id=ART_REPO,
    repo_type="dataset",
    commit_message="Phase 5A v2: 22-36w verified Qwen2.5-3B fused alt-text targets",
)

passed_lens.sort()
final_lens.sort()
print(f"\n=== PHASE 5A (V2) FUSION COMPLETE ({(time.time()-t0)/60:.1f} min) ===")
print(f"Verified fused pass rate: {pass_cnt}/{len(items)} ({100*pass_cnt/len(items):.1f}%)")
print(f"Fail breakdown: {dict(fail_reasons)}")
print(f"Passed fused word lengths: mean={sum(passed_lens)/len(passed_lens):.1f}, median={passed_lens[len(passed_lens)//2]}, "
      f"p10={passed_lens[int(0.1*len(passed_lens))]}, p90={passed_lens[int(0.9*len(passed_lens))]}")
print(f"Final target word lengths: mean={sum(final_lens)/len(final_lens):.1f}, median={final_lens[len(final_lens)//2]}")

Kernel GPU 0 allocated=0.00 GB | reserved=0.00 GB
Total images to fuse: 21297
  [fusion v2 0.3m] 0/21297 (0.00 img/s, ETA: 354950000.0 min)
  [fusion v2 0.7m] 192/21297 (4.80 img/s, ETA: 73.3 min)
  [fusion v2 1.0m] 384/21297 (6.40 img/s, ETA: 54.5 min)
  [fusion v2 1.3m] 544/21297 (6.80 img/s, ETA: 50.9 min)
  [fusion v2 1.7m] 736/21297 (7.36 img/s, ETA: 46.6 min)
  [fusion v2 2.0m] 928/21297 (7.73 img/s, ETA: 43.9 min)
  [fusion v2 2.3m] 1120/21297 (8.00 img/s, ETA: 42.0 min)
  [fusion v2 2.7m] 1312/21297 (8.20 img/s, ETA: 40.6 min)
  [fusion v2 3.0m] 1440/21297 (8.00 img/s, ETA: 41.4 min)
  [fusion v2 3.3m] 1632/21297 (8.16 img/s, ETA: 40.2 min)
  [fusion v2 3.7m] 1824/21297 (8.29 img/s, ETA: 39.1 min)
  [fusion v2 4.0m] 1984/21297 (8.27 img/s, ETA: 38.9 min)
  [fusion v2 4.3m] 2176/21297 (8.37 img/s, ETA: 38.1 min)
  [fusion v2 4.7m] 2336/21297 (8.34 img/s, ETA: 37.9 min)
  [fusion v2 5.0m] 2528/21297 (8.43 img/s, ETA: 37.1 min)
  [fusion v2 5.3m] 2688/21297 (8.40 img/s, ETA: 36.9 

In [ ]:
import torch
from pathlib import Path

print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB | reserved: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
for p in [
    "/kaggle/tmp/vis_feats/vis_feats_train_v1.pt",
    "/kaggle/tmp/vis_feats/vis_feats_dev_v1.pt",
    "/kaggle/tmp/vis_feats/vis_feats_val_v1.pt",
    "/kaggle/tmp/phase4_synth/vis_feats_phase4_synth.pt",
    "/kaggle/tmp/phase5_fusion/shard_0.jsonl",
    "/kaggle/working/artifacts/fused_targets_v1.json",
]:
    print(f"  {Path(p).exists()} -> {p}")

In [3]:
import os, gc, subprocess
from pathlib import Path
import torch
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.memory_allocated(0) < 0.5e9, "GPU 0 not clean!"

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
ART_DIR = Path("/kaggle/working/artifacts")
assert (ART_DIR / "fused_targets_v1.json").exists()

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
wandb_key = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-trackB"
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, private=False)

train_script = Path("/kaggle/tmp/train_trackB_ddp.py")
train_script.write_text(r'''
import os, re, sys, json, time, math, random, gc
from collections import defaultdict
from pathlib import Path

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import Blip2Processor, Blip2ForConditionalGeneration, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
import wandb
from huggingface_hub import HfApi

sys.path.insert(0, "/kaggle/working/src")
import eval_suite

EPOCHS = 4
PER_GPU_BS = 4
GRAD_ACCUM = 4          # 4 * 2 GPUs * 4 = 32 effective batch size
LR = 1.5e-4
WARMUP_RATIO = 0.03
SEED = 42

REAL_UNCLEAR_TEMPLATES = [
    "The image is too blurry or unclear to describe with confidence.",
    "Quality issues are too severe to identify what is shown in this photo.",
    "The photo is too unclear to make out the subject reliably.",
    "It is not possible to describe this image clearly due to poor image quality.",
]

SYNTH_UNCLEAR_TEMPLATES = {
    "gaussian": [
        "The image is too blurry to describe with confidence.",
        "This photo is too blurry to make out what is shown.",
    ],
    "motion": [
        "The image is too blurry to describe with confidence.",
        "Heavy motion blur makes it impossible to identify what is shown.",
    ],
    "dark": [
        "The photo is too dark to make out what is shown.",
        "The image is too dark to identify anything clearly.",
    ],
    "occlude": [
        "The item is cut off or out of frame, so it cannot be identified.",
        "Too much of the subject is obscured or out of frame to describe.",
    ],
}

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def main():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    seed_everything(SEED + rank)

    DATA_DIR = Path("/kaggle/tmp/vizwiz")
    ART_DIR = Path("/kaggle/working/artifacts")
    FEAT_DIR = Path("/kaggle/tmp/vis_feats")
    SYN_DIR = Path("/kaggle/tmp/phase4_synth")
    CKPT_DIR = Path("/kaggle/working/checkpoints/trackB")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    splits = json.loads((ART_DIR / "splits_v1.json").read_text())
    ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
    fused_map = json.loads((ART_DIR / "fused_targets_v1.json").read_text())
    manifest = json.loads((SYN_DIR / "phase4_manifest.json").read_text())
    train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())

    precanned_counts = defaultdict(int)
    for a in train_raw["annotations"]:
        if a.get("is_precanned", False):
            precanned_counts[a["image_id"]] += 1
    del train_raw

    examples = []
    n_real_fused, n_real_unclear = 0, 0
    for iid in splits["train_ids"]:
        dets = ocr_all.get(str(iid), {}).get("dets", [])
        if precanned_counts[iid] >= 3:
            examples.append({"src": "real", "key": iid, "dets": dets, "targets": REAL_UNCLEAR_TEMPLATES})
            n_real_unclear += 1
        elif str(iid) in fused_map:
            examples.append({"src": "real", "key": iid, "dets": dets, "targets": [fused_map[str(iid)]["target"]]})
            n_real_fused += 1

    for rec in manifest["train_synth"]:
        targs = SYNTH_UNCLEAR_TEMPLATES[rec["kind"]] if rec["is_unclear"] else [fused_map[str(rec["image_id"])]["target"]]
        examples.append({"src": "synth", "key": rec["key"], "dets": rec["dets"], "targets": targs})

    if rank == 0:
        print(f"Track B train mix: total={len(examples)} | real_fused={n_real_fused} | "
              f"real_unclear={n_real_unclear} | synth_unclear={manifest['n_synth_unclear']} | "
              f"synth_robust={manifest['n_synth_robust_clean']}")

    dev_ids = splits["dev_ids"]
    gt_dev = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/train.json", dev_ids, ocr_all)

    real_feats = torch.load(FEAT_DIR / "vis_feats_train_v1.pt", weights_only=True)
    synth_feats = torch.load(SYN_DIR / "vis_feats_phase4_synth.pt", weights_only=True)
    dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)

    model_id = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(model_id)
    tokenizer = processor.tokenizer

    full_model = Blip2ForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    IMAGE_TOKEN_ID = full_model.config.image_token_id
    opt_model = full_model.language_model
    del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens, full_model
    gc.collect()
    opt_model = opt_model.to(device)

    opt_model.config.use_cache = False
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none",
    )
    opt_model = get_peft_model(opt_model, lora_cfg)
    if rank == 0:
        opt_model.print_trainable_parameters()

    ddp_model = DDP(opt_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    class TrackBDataset(Dataset):
        def __init__(self, items, max_cap_tok=80):
            self.items = items
            self.max_cap_tok = max_cap_tok
            self.img_prefix = [IMAGE_TOKEN_ID] * 32

        def __len__(self):
            return len(self.items)

        def __getitem__(self, idx):
            ex = self.items[idx]
            ocr_str = eval_suite.format_ocr_text(ex["dets"])
            prompt = eval_suite.build_prompt(ocr_str)
            cap = random.choice(ex["targets"])
            p_ids = tokenizer.encode(prompt, add_special_tokens=True)
            c_ids = tokenizer.encode(" " + cap, add_special_tokens=False)[:self.max_cap_tok - 1] + [tokenizer.eos_token_id]
            input_ids = self.img_prefix + p_ids + c_ids
            labels = ([-100] * (32 + len(p_ids))) + c_ids
            vf = real_feats[ex["key"]] if ex["src"] == "real" else synth_feats[ex["key"]]
            return {
                "vis_feat": vf,
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

    def collate_fn(batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)
        pad_id = tokenizer.pad_token_id
        B = len(batch)
        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
        attn_mask = torch.zeros((B, max_len), dtype=torch.long)
        labels = torch.full((B, max_len), -100, dtype=torch.long)
        vis_feats = torch.stack([x["vis_feat"] for x in batch])
        for i, x in enumerate(batch):
            L = x["input_ids"].shape[0]
            input_ids[i, :L] = x["input_ids"]
            attn_mask[i, :L] = 1
            labels[i, :L] = x["labels"]
        return {"vis_feats": vis_feats, "input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

    train_ds = TrackBDataset(examples)
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED)
    loader = DataLoader(train_ds, batch_size=PER_GPU_BS, sampler=sampler, collate_fn=collate_fn, num_workers=0)

    steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = int(total_steps * WARMUP_RATIO)

    trainable_params = [p for p in ddp_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda")

    if rank == 0:
        run = wandb.init(
            project="vizwiz-alttext-blip2",
            name="phase5-trackB-fused-20-40w-v2",
            config={
                "phase": "5B", "track": "B", "model": model_id,
                "n_train": len(examples), "max_cap_tok": 80, "max_new_tokens": 75,
                "lora_r": 16, "lora_alpha": 32, "epochs": EPOCHS,
                "effective_bs": PER_GPU_BS * world_size * GRAD_ACCUM, "lr": LR,
            },
        )
        api = HfApi(token=os.environ["HF_TOKEN"])
        model_repo = os.environ["MODEL_REPO"]

    def forward_loss(batch):
        vf = batch["vis_feats"].to(device, torch.float16)
        in_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        emb = ddp_model.module.get_input_embeddings()(in_ids).clone()
        emb[:, :32, :] = vf.to(emb.dtype)
        return ddp_model(inputs_embeds=emb, attention_mask=attn, labels=lbls).loss

    @torch.no_grad()
    def eval_dev_epoch(epoch):
        base_peft = ddp_model.module
        base_peft.eval()
        base_peft.config.use_cache = True
        torch.cuda.empty_cache()

        my_ids = sorted(dev_ids[rank::world_size], key=lambda x: len(gt_dev[x]["ocr_str"]))
        my_preds = {}
        emb_layer = base_peft.get_input_embeddings()
        bs_gen = 8
        for i in range(0, len(my_ids), bs_gen):
            b_ids = my_ids[i:i + bs_gen]
            seqs = [[IMAGE_TOKEN_ID] * 32 + tokenizer.encode(eval_suite.build_prompt(gt_dev[iid]["ocr_str"]), add_special_tokens=True) for iid in b_ids]
            max_l = max(len(s) for s in seqs)
            B = len(seqs)
            in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device=device)
            attn = torch.zeros((B, max_l), dtype=torch.long, device=device)
            vf_batch = torch.stack([dev_feats[iid] for iid in b_ids]).to(device, torch.float16)
            emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device=device)
            for j, s in enumerate(seqs):
                L = len(s); pad = max_l - L
                in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device=device)
                attn[j, pad:] = 1
                row_emb = emb_layer(in_ids[j]).to(torch.float16)
                row_emb[pad:pad+32] = vf_batch[j]
                emb[j] = row_emb
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = base_peft.generate(
                    inputs_embeds=emb, attention_mask=attn,
                    num_beams=5, max_new_tokens=75, no_repeat_ngram_size=3,
                    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
                )
            texts = tokenizer.batch_decode(out, skip_special_tokens=True)
            for iid, t in zip(b_ids, texts):
                my_preds[str(iid)] = t.strip()

        shard_p = Path(f"/kaggle/tmp/tb_dev_ep{epoch}_r{rank}.json")
        shard_p.write_text(json.dumps(my_preds), encoding="utf-8")
        torch.cuda.empty_cache()
        dist.barrier()

        metrics = None
        if rank == 0:
            all_preds = {}
            for r_i in range(world_size):
                sp = Path(f"/kaggle/tmp/tb_dev_ep{epoch}_r{r_i}.json")
                all_preds.update({int(k): v for k, v in json.loads(sp.read_text()).items()})
                sp.unlink()
            metrics = eval_suite.evaluate_predictions(
                all_preds, gt_dev, DATA_DIR / "train_images/train",
                run_spice=False, run_clip=True, device=f"cuda:{local_rank}",
            )
            Path(CKPT_DIR / f"dev_preds_ep{epoch}.json").write_text(
                json.dumps({str(k): v for k, v in all_preds.items()}, ensure_ascii=False), encoding="utf-8"
            )

        base_peft.config.use_cache = False
        base_peft.train()
        dist.barrier()
        return metrics

    best_score = -1e9
    best_ep = 0
    global_step = 0
    smooth_loss = []
    t_train0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        sampler.set_epoch(epoch)
        ddp_model.train()
        optimizer.zero_grad(set_to_none=True)
        t_ep = time.time()

        for step_idx, batch in enumerate(loader):
            is_sync = ((step_idx + 1) % GRAD_ACCUM == 0) or ((step_idx + 1) == len(loader))
            if not is_sync:
                with ddp_model.no_sync():
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        loss = forward_loss(batch) / GRAD_ACCUM
                    scaler.scale(loss).backward()
                    smooth_loss.append(loss.item() * GRAD_ACCUM)
            else:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = forward_loss(batch) / GRAD_ACCUM
                scaler.scale(loss).backward()
                smooth_loss.append(loss.item() * GRAD_ACCUM)

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if rank == 0 and global_step % 50 == 0:
                    avg_l = sum(smooth_loss[-50:]) / len(smooth_loss[-50:])
                    lr_now = scheduler.get_last_lr()[0]
                    print(f"[Ep {epoch}/{EPOCHS} | Step {global_step}/{total_steps}] "
                          f"loss={avg_l:.4f} | lr={lr_now:.2e} | {(time.time()-t_train0)/60:.1f}m")
                    wandb.log({"train/loss": avg_l, "train/lr": lr_now, "train/epoch": epoch}, step=global_step)

        dev_metrics = eval_dev_epoch(epoch)
        if rank == 0:
            ov = dev_metrics["overall"]
            bh = dev_metrics["behavior"]
            sel_score = 100.0 * ov["RefCLIPScore"] + 0.10 * bh["unclear_recall_pct"] - 0.5 * ov["unsup_noun_pct"] - 2.0 * bh["false_unclear_pct"]
            print(f"\n=== EPOCH {epoch} DEV RESULTS ({(time.time()-t_ep)/60:.1f} min) ===")
            print(f"  avg_words={ov['avg_words']} | CLIP={ov['CLIPScore']} | RefCLIP={ov['RefCLIPScore']} | "
                  f"unsup_noun={ov['unsup_noun_pct']}% | cap_unsup={ov['cap_with_unsup_pct']}%")
            print(f"  unclear_recall={bh['unclear_recall_pct']}% | false_unclear={bh['false_unclear_pct']}% | "
                  f"ocr_adopt={bh['ocr_token_adoption_pct']}% | CIDEr={ov['CIDEr']} | sel_score={sel_score:.2f}")

            wandb.log({
                "dev/overall/avg_words": ov["avg_words"],
                "dev/overall/CLIPScore": ov["CLIPScore"],
                "dev/overall/RefCLIPScore": ov["RefCLIPScore"],
                "dev/overall/unsup_noun_pct": ov["unsup_noun_pct"],
                "dev/overall/cap_with_unsup_pct": ov["cap_with_unsup_pct"],
                "dev/overall/CIDEr": ov["CIDEr"],
                "dev/behavior/unclear_recall_pct": bh["unclear_recall_pct"],
                "dev/behavior/false_unclear_pct": bh["false_unclear_pct"],
                "dev/behavior/ocr_token_adoption_pct": bh["ocr_token_adoption_pct"],
                "dev/sel_score": sel_score,
            }, step=global_step)

            ep_dir = CKPT_DIR / f"epoch_{epoch}"
            ddp_model.module.save_pretrained(ep_dir)
            (ep_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))

            if (sel_score > best_score and bh["false_unclear_pct"] <= 4.0) or best_ep == 0:
                best_score = sel_score
                best_ep = epoch
                best_dir = CKPT_DIR / "best"
                ddp_model.module.save_pretrained(best_dir)
                (best_dir / "dev_metrics.json").write_text(json.dumps(dev_metrics, indent=2))
                api.upload_folder(
                    folder_path=str(best_dir),
                    repo_id=model_repo,
                    commit_message=f"Track B best adapter (ep={epoch}, RefCLIP={ov['RefCLIPScore']}, unsup={ov['unsup_noun_pct']}%, words={ov['avg_words']})",
                )
                print(f"  -> New BEST! Uploaded epoch {epoch} to {model_repo}")

    if rank == 0:
        print(f"\nTRACK B TRAINING COMPLETE! Best epoch={best_ep} (sel_score={best_score:.2f})")
        run.finish()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["WANDB_API_KEY"] = wandb_key
env["MODEL_REPO"] = MODEL_REPO
env["HF_HOME"] = "/kaggle/tmp/hf_cache"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.Popen(
    ["torchrun", "--standalone", "--nproc_per_node=2", str(train_script)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"DDP training exited with code {proc.returncode}")

W1124 17:16:22.709000 730 torch/distributed/run.py:851] 
W1124 17:16:22.709000 730 torch/distributed/run.py:851] *****************************************
W1124 17:16:22.709000 730 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1124 17:16:22.709000 730 torch/distributed/run.py:851] *****************************************
Track B train mix: total=24231 | real_fused=20388 | real_unclear=2043 | synth_unclear=612 | synth_robust=1188

Loading weights: 100%|██████████| 1247/1247 [01:05<00:00, 18.99it/s]

trainable params: 5,242,880 || all params: 2,656,921,600 || trainable%: 0.1973
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zsehic1 (zsehic1-unsa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tra

In [5]:
import os, csv, json, time, subprocess, sys, gc, random
from pathlib import Path
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
import wandb
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

api = HfApi(token=hf_token)
hf_user = api.whoami()["name"]
ART_REPO = f"{hf_user}/vizwiz-alttext-artifacts"
MODEL_REPO = f"{hf_user}/vizwiz-alttext-lora-trackB"

ART_DIR = Path("/kaggle/working/artifacts")
DATA_DIR = Path("/kaggle/tmp/vizwiz")
FEAT_DIR = Path("/kaggle/tmp/vis_feats")
SYN_DIR = Path("/kaggle/tmp/phase4_synth")
TMP_DIR = Path("/kaggle/tmp/trackB_val")
TMP_DIR.mkdir(parents=True, exist_ok=True)
BEST_TB = Path("/kaggle/working/checkpoints/trackB/best")

if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import importlib, eval_suite
importlib.reload(eval_suite)

splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())
gt_val = eval_suite.load_ground_truth(DATA_DIR / "annotations/annotations/val.json", splits["val_ids"], ocr_dv)

val_tasks = [
    {"image_id": iid, "prompt": eval_suite.build_prompt(info["ocr_str"])}
    for iid, info in gt_val.items()
]
val_tasks.sort(key=lambda x: len(x["prompt"]))

worker = TMP_DIR / "val_worker_tb.py"
worker.write_text('''
import sys, json, gc, torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

todo_file, out_file, feat_pt, adapter_dir, bs = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], int(sys.argv[5])
items = json.loads(open(todo_file, "r", encoding="utf-8").read())
feats = torch.load(feat_pt, weights_only=True)

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer

full_model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
IMAGE_TOKEN_ID = full_model.config.image_token_id
opt_model = full_model.language_model
del full_model.vision_model, full_model.qformer, full_model.language_projection, full_model.query_tokens, full_model
gc.collect()

opt_model = PeftModel.from_pretrained(opt_model.to("cuda:0"), adapter_dir)
opt_model.eval()
emb_layer = opt_model.get_input_embeddings()

def run_batch(batch):
    seqs = [[IMAGE_TOKEN_ID] * 32 + tokenizer.encode(x["prompt"], add_special_tokens=True) for x in batch]
    max_l = max(len(s) for s in seqs)
    B = len(seqs)
    in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device="cuda:0")
    attn = torch.zeros((B, max_l), dtype=torch.long, device="cuda:0")
    vf_batch = torch.stack([feats[x["image_id"]] for x in batch]).to("cuda:0", torch.float16)
    emb = torch.zeros((B, max_l, vf_batch.shape[-1]), dtype=torch.float16, device="cuda:0")
    for j, s in enumerate(seqs):
        L = len(s); pad = max_l - L
        in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device="cuda:0")
        attn[j, pad:] = 1
        row_emb = emb_layer(in_ids[j]).to(torch.float16)
        row_emb[pad:pad+32] = vf_batch[j]
        emb[j] = row_emb
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = opt_model.generate(
            inputs_embeds=emb, attention_mask=attn,
            num_beams=5, max_new_tokens=75, no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

with open(out_file, "w", encoding="utf-8") as fout:
    for i in range(0, len(items), bs):
        batch = items[i:i + bs]
        try:
            texts = run_batch(batch)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            texts = [run_batch([x])[0] for x in batch]
        for x, t in zip(batch, texts):
            fout.write(json.dumps({"image_id": x["image_id"], "pred": t.strip()}, ensure_ascii=False) + "\\n")
        fout.flush()
''', encoding="utf-8")

t0 = time.time()
t0_f, t1_f = TMP_DIR / "todo_0.json", TMP_DIR / "todo_1.json"
s0_f, s1_f = TMP_DIR / "shard_0.jsonl", TMP_DIR / "shard_1.jsonl"
t0_f.write_text(json.dumps(val_tasks[0::2]))
t1_f.write_text(json.dumps(val_tasks[1::2]))

procs = []
for gpu_id, tf, sf in [(0, t0_f, s0_f), (1, t1_f, s1_f)]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    procs.append(subprocess.Popen(
        ["python3", str(worker), str(tf), str(sf), str(FEAT_DIR / "vis_feats_val_v1.pt"), str(BEST_TB), "6"],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    ))

def _cnt(p):
    return sum(1 for l in open(p) if l.strip()) if p.exists() else 0

while any(p.poll() is None for p in procs):
    time.sleep(20)
    n_done = _cnt(s0_f) + _cnt(s1_f)
    rate = n_done / max(time.time() - t0, 1e-6)
    print(f"  [trackB val {(time.time()-t0)/60:.1f}m] {n_done}/{len(val_tasks)} ({rate:.2f} img/s)")

for gid, p in enumerate(procs):
    if p.returncode != 0:
        raise RuntimeError(f"GPU {gid} failed:\n{p.stderr.read()}")

preds_val = {}
for sf in [s0_f, s1_f]:
    for line in open(sf, "r", encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            preds_val[int(r["image_id"])] = r["pred"]

assert len(preds_val) == 7750
preds_path = ART_DIR / "preds_trackB_val_v1.json"
preds_path.write_text(json.dumps({str(k): v for k, v in preds_val.items()}, ensure_ascii=False), encoding="utf-8")

res_val = eval_suite.evaluate_predictions(preds_val, gt_val, DATA_DIR / "val_images/val", run_spice=False, run_clip=True)
res_dev = json.loads((BEST_TB / "dev_metrics.json").read_text())

clear_word_lens = sorted(
    len(preds_val[i].split()) for i in gt_val
    if gt_val[i]["is_strict_clear"] and not eval_suite.UNCLEAR_PATTERN.search(preds_val[i])
)
pct_20_40 = round(100.0 * sum(1 for w in clear_word_lens if 20 <= w <= 40) / max(len(clear_word_lens), 1), 1)
pct_15_40 = round(100.0 * sum(1 for w in clear_word_lens if 15 <= w <= 40) / max(len(clear_word_lens), 1), 1)

# --- Blur Severity Curve for Track B ---
manifest = json.loads((SYN_DIR / "phase4_manifest.json").read_text())
dev_feats = torch.load(FEAT_DIR / "vis_feats_dev_v1.pt", weights_only=True)
synth_feats = torch.load(SYN_DIR / "vis_feats_phase4_synth.pt", weights_only=True)
curve_ids = sorted(set(r["image_id"] for r in manifest["dev_blur_curve"]))
curve_by_lvl = {0: [{"key": iid, "dets": ocr_dv.get(str(iid), {}).get("dets", [])} for iid in curve_ids]}
for lvl in [1, 2, 3, 4]:
    curve_by_lvl[lvl] = [r for r in manifest["dev_blur_curve"] if r["level"] == lvl]

model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)
tokenizer = processor.tokenizer
full_m = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
img_tok_id = full_m.config.image_token_id
opt_m = full_m.language_model
del full_m.vision_model, full_m.qformer, full_m.language_projection, full_m.query_tokens, full_m
gc.collect()
opt_m = PeftModel.from_pretrained(opt_m.to("cuda:0"), str(BEST_TB))
opt_m.eval()
emb_layer = opt_m.get_input_embeddings()

curve_tb = {}
for lvl, items in curve_by_lvl.items():
    unclear_hits = 0
    for i in range(0, len(items), 10):
        batch = items[i:i+10]
        seqs = [[img_tok_id]*32 + tokenizer.encode(eval_suite.build_prompt(eval_suite.format_ocr_text(x["dets"])), add_special_tokens=True) for x in batch]
        max_l = max(len(s) for s in seqs)
        B = len(seqs)
        in_ids = torch.full((B, max_l), tokenizer.pad_token_id, dtype=torch.long, device="cuda:0")
        attn = torch.zeros((B, max_l), dtype=torch.long, device="cuda:0")
        vf_b = torch.stack([dev_feats[x["key"]] if lvl == 0 else synth_feats[x["key"]] for x in batch]).to("cuda:0", torch.float16)
        emb = torch.zeros((B, max_l, vf_b.shape[-1]), dtype=torch.float16, device="cuda:0")
        for j, s in enumerate(seqs):
            L = len(s); pad = max_l - L
            in_ids[j, pad:] = torch.tensor(s, dtype=torch.long, device="cuda:0")
            attn[j, pad:] = 1
            row = emb_layer(in_ids[j]).to(torch.float16)
            row[pad:pad+32] = vf_b[j]
            emb[j] = row
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            out = opt_m.generate(inputs_embeds=emb, attention_mask=attn, num_beams=5, max_new_tokens=75, no_repeat_ngram_size=3,
                                 eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
        for t in tokenizer.batch_decode(out, skip_special_tokens=True):
            if eval_suite.UNCLEAR_PATTERN.search(t):
                unclear_hits += 1
    curve_tb[f"L{lvl}"] = round(100.0 * unclear_hits / len(items), 1)
del opt_m; gc.collect(); torch.cuda.empty_cache()

def load_preds_fallback(local_p, remote_p):
    if not local_p.exists():
        local_p = Path(hf_hub_download(ART_REPO, remote_p, repo_type="dataset", token=hf_token))
    return json.loads(local_p.read_text())["val"] if "zs_" in local_p.name else json.loads(local_p.read_text())

zs_val = load_preds_fallback(ART_DIR / "preds_zs_empty_v1.json", "phase2/preds_zs_empty_v1.json")
p4_val = load_preds_fallback(ART_DIR / "preds_phase4_val_v1.json", "phase4/preds_phase4_val_v1.json")

rng = random.Random(42)
audit_ids = rng.sample(splits["val_ids"], 100)
audit_csv = ART_DIR / "manual_audit_100_val.csv"
with open(audit_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["image_id", "file_name", "text_detected", "is_unclear_gt", "ocr_str", "ref_0", "zs_empty", "phase4_11w", "trackB_20_40w", "human_label"])
    for iid in audit_ids:
        g = gt_val[iid]
        w.writerow([
            iid, g["file_name"], g["text_detected"], g["is_unclear_gt"], g["ocr_str"],
            g["valid_refs"][0] if g["valid_refs"] else "<precanned>",
            zs_val.get(str(iid), ""), p4_val.get(str(iid), ""), preds_val.get(iid, ""), "",
        ])

tb_results = {
    "dev_best": res_dev,
    "val": res_val,
    "val_clear_len_stats": {
        "mean": round(sum(clear_word_lens)/len(clear_word_lens), 2),
        "median": clear_word_lens[len(clear_word_lens)//2],
        "p10": clear_word_lens[int(0.1*len(clear_word_lens))],
        "p90": clear_word_lens[int(0.9*len(clear_word_lens))],
        "pct_in_20_40": pct_20_40,
        "pct_in_15_40": pct_15_40,
    },
    "blur_severity_curve_trackB": curve_tb,
}
res_path = ART_DIR / "trackB_results_v1.json"
res_path.write_text(json.dumps(tb_results, indent=2), encoding="utf-8")

for p in [preds_path, res_path, audit_csv]:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=f"phase5/{p.name}", repo_id=ART_REPO, repo_type="dataset")

run = wandb.init(project="vizwiz-alttext-blip2", name="phase5-trackB-val-final", config={"model": MODEL_REPO})
flat = {f"val/{grp}/{k}": v for grp, d in res_val.items() for k, v in d.items() if v is not None}
for k, v in tb_results["val_clear_len_stats"].items():
    flat[f"val/len_stats/{k}"] = v
for k, v in curve_tb.items():
    flat[f"blur_curve/trackB/{k}"] = v
wandb.log(flat)
run.finish()

print("\n=== 15 SIDE-BY-SIDE VAL SAMPLES (ZS vs Phase 4 vs Track B) ===")
for iid in audit_ids[:15]:
    g = gt_val[iid]
    print(f"\nID {iid} | text_det={g['text_detected']} | unclear_gt={g['is_unclear_gt']} | OCR: {g['ocr_str'][:50]}")
    print(f"  Ref[0] : {g['valid_refs'][0] if g['valid_refs'] else '<precanned>'}")
    print(f"  ZS     : {zs_val.get(str(iid), '')}")
    print(f"  Phase4 : {p4_val.get(str(iid), '')}")
    print(f"  TrackB : {preds_val.get(iid, '')}")

print("\n=== PHASE 5 TRACK B FINAL RESULTS (DEV + VAL + BLUR CURVE) ===")
print(json.dumps(tb_results, indent=2))

  [trackB val 0.3m] 198/7750 (9.89 img/s)
  [trackB val 0.7m] 198/7750 (4.95 img/s)
  [trackB val 1.0m] 198/7750 (3.30 img/s)
  [trackB val 1.3m] 198/7750 (2.47 img/s)
  [trackB val 1.7m] 54/7750 (0.54 img/s)
  [trackB val 2.0m] 156/7750 (1.30 img/s)
  [trackB val 2.3m] 264/7750 (1.89 img/s)
  [trackB val 2.7m] 360/7750 (2.25 img/s)
  [trackB val 3.0m] 468/7750 (2.60 img/s)
  [trackB val 3.3m] 576/7750 (2.88 img/s)
  [trackB val 3.7m] 678/7750 (3.08 img/s)
  [trackB val 4.0m] 786/7750 (3.27 img/s)
  [trackB val 4.3m] 894/7750 (3.44 img/s)
  [trackB val 4.7m] 1002/7750 (3.58 img/s)
  [trackB val 5.0m] 1104/7750 (3.68 img/s)
  [trackB val 5.3m] 1206/7750 (3.77 img/s)
  [trackB val 5.7m] 1314/7750 (3.86 img/s)
  [trackB val 6.0m] 1410/7750 (3.92 img/s)
  [trackB val 6.3m] 1518/7750 (3.99 img/s)
  [trackB val 6.7m] 1620/7750 (4.05 img/s)
  [trackB val 7.0m] 1722/7750 (4.10 img/s)
  [trackB val 7.3m] 1824/7750 (4.14 img/s)
  [trackB val 7.7m] 1926/7750 (4.19 img/s)
  [trackB val 8.0m] 2028/

  [trackB val 33.0m] 7750/7750 (3.91 img/s)


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:555: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
PTBTokenizer tokenized 463130 tokens at 682072.91 tokens per second.
PTBTokenizer tokenized 174350 tokens at 425026.82 tokens per second.


{'testlen': 147939, 'reflen': 111196, 'guess': [147939, 140397, 132855, 125313], 'correct': [66283, 32311, 14522, 5935]}
ratio: 1.330434548005312


... (15 outputs left out)

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1472: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zsehic1 (zsehic1-unsa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


blur_curve/trackB/L0,▁
blur_curve/trackB/L1,▁
blur_curve/trackB/L2,▁
blur_curve/trackB/L3,▁
blur_curve/trackB/L4,▁
val/behavior/false_unclear_pct,▁
val/behavior/n_strict_clear,▁
val/behavior/n_total,▁
val/behavior/n_unclear_gt,▁
val/behavior/ocr_token_adoption_pct,▁
+39,...



=== 15 SIDE-BY-SIDE VAL SAMPLES (ZS vs Phase 4 vs Track B) ===

ID 28669 | text_det=False | unclear_gt=True | OCR: Y
  Ref[0] : A wooden table has a pink phone and clothes on it.
  ZS     : a table with food on it and a person in the background
  Phase4 : A wooden table with a white napkin on it.
  TrackB : A wooden table with a white napkin on it and a plate of food on top of it.

ID 24343 | text_det=True | unclear_gt=False | OCR: none
  Ref[0] : Multiple sections with words in it that's separated by different colored columns
  ZS     : a person's hand holding a piece of paper with information on it
  Phase4 : Quality issues are too severe to identify what is shown in this photo.
  TrackB : Quality issues are too severe to identify what is shown in this photo.

ID 23635 | text_det=True | unclear_gt=False | OCR: ATH CAB for CUTIE CODES and KEYS HPL
  Ref[0] : Compact disc in a black package on top of a desk.
  ZS     : a cd sitting on top of a wooden table
  Phase4 : A white disc with

In [6]:
import os, re, gc, json, time, sys
from collections import defaultdict
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
if "/kaggle/working/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/src")
import eval_suite

DATA_DIR = Path("/kaggle/tmp/vizwiz")
ART_DIR = Path("/kaggle/working/artifacts")
splits = json.loads((ART_DIR / "splits_v1.json").read_text())
ocr_all = json.loads((ART_DIR / "ocr_cache_v1.json").read_text())
ocr_dv = json.loads((ART_DIR / "ocr_cache_dev_val_v1.json").read_text())

# 1. Clean OCR v3 filter: drops 1-char junk, symbol noise, and low-conf (<0.65) false positives
def format_ocr_v3(dets, conf_thr=0.65, max_words=12):
    kept = []
    for d in dets:
        conf = float(d.get("conf", 0.0))
        txt = re.sub(r"\s+", " ", d.get("text", "")).strip()
        if conf < conf_thr or not txt:
            continue
        alnum = re.findall(r"[A-Za-z0-9]", txt)
        if len(alnum) < 2:
            continue
        # Drop boxes dominated by punctuation/symbols or isolated 1-char tokens
        if len(alnum) / max(len(txt.replace(" ", "")), 1) < 0.60:
            continue
        if not (re.search(r"[A-Za-z]{2,}", txt) or re.search(r"\d{2,}", txt)):
            continue
        kept.append((round(float(d.get("y", 0.0)) / 25.0), float(d.get("x", 0.0)), txt))
    if not kept:
        return "none"
    kept.sort(key=lambda t: (t[0], t[1]))
    words = " ".join(t[2] for t in kept).split()
    # Drop isolated 1-char tokens unless numeric
    clean_words = [w for w in words if len(re.sub(r"[^A-Za-z0-9]", "", w)) >= 2 or w.isdigit()]
    return " ".join(clean_words[:max_words]) if clean_words else "none"

print("=== OCR V3 CLEANUP ON TRICKY VAL SAMPLES ===")
for iid in [28669, 25684, 25437, 29464, 23635, 28974, 29498, 24143]:
    dets = ocr_dv.get(str(iid), {}).get("dets", [])
    print(f"  ID {iid}: old={eval_suite.format_ocr_text(dets)!r}  -->  v3={format_ocr_v3(dets)!r}")

# 2. Test V3 Qwen2.5-3B Fusion on 64 train images (mix of text + no_text)
train_raw = json.loads((DATA_DIR / "annotations/annotations/train.json").read_text())
anns_by_img = defaultdict(list)
precanned_counts = defaultdict(int)
for a in train_raw["annotations"]:
    iid = a["image_id"]
    if a.get("is_precanned", False):
        precanned_counts[iid] += 1
    elif not a.get("is_rejected", False):
        c = re.sub(r"\s+", " ", a["caption"]).strip()
        if c:
            anns_by_img[iid].append(c)

sample_ids = [iid for iid in splits["train_ids"] if precanned_counts[iid] < 3 and len(anns_by_img[iid]) >= 2][:64]

model_id = "Qwen/Qwen2.5-3B-Instruct"
q_tok = AutoTokenizer.from_pretrained(model_id)
q_tok.padding_side = "left"
q_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
q_model.eval()

SYS_MSG_V3 = (
    "Write concise, accurate 1-2 sentence image alt-text (aim for 16 to 30 words when enough visual facts are confirmed; "
    "write a shorter single sentence if fewer facts exist). "
    "STRICT RULES: (1) Combine ONLY facts supported by the reference captions and clean OCR text. "
    "(2) NEVER invent rooms ('kitchen', 'bathroom'), furniture ('couch', 'microwave'), or ingredients not stated. "
    "(3) NEVER repeat phrases and NEVER tack on 'Visible text reads ...' as a separate sentence—weave clean brand/product names naturally instead, and ignore garbled OCR. "
    "Output ONLY the alt-text."
)

def make_prompt_v3(refs, ocr_str):
    refs_block = "\n".join(f"- {r}" for r in refs[:5])
    user_msg = f"Reference captions:\n{refs_block}\nVisible OCR text: {ocr_str}\n\nAlt-text:"
    return q_tok.apply_chat_template(
        [{"role": "system", "content": SYS_MSG_V3}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

STRUCTURAL_STRICT = {"word", "words", "text", "label", "front", "back", "top", "bottom", "side"}
nlp = eval_suite._get_nlp()

def has_repeated_3gram(text):
    toks = re.findall(r"[a-z0-9]+", text.lower())
    if len(toks) < 6:
        return False
    trigrams = [tuple(toks[i:i+3]) for i in range(len(toks)-2)]
    return len(trigrams) != len(set(trigrams))

def verify_v3(fused, refs, ocr_clean_str):
    fused = re.sub(r"\s*\(\d+\s*words?\)\.?\s*$", "", fused, flags=re.IGNORECASE)
    fused = re.sub(r"\s+", " ", fused).strip().strip('"').strip("'")
    words = fused.split()
    if not (12 <= len(words) <= 38):
        return False, f"len={len(words)}", fused
    if has_repeated_3gram(fused):
        return False, "repeat_3gram", fused
    if re.search(r"visible text reads", fused, re.IGNORECASE):
        return False, "boilerplate_ocr", fused

    ocr_for_support = "" if ocr_clean_str == "none" else ocr_clean_str
    all_src = (" ".join(refs) + " " + ocr_for_support).lower()
    src_lemmas = {tok.lemma_.lower() for tok in nlp(all_src) if tok.is_alpha}
    src_raw = set(re.findall(r"[a-z0-9]+", all_src))
    ref_nouns = set()
    for r in refs:
        ref_nouns.update(eval_suite._extract_nouns(r))
    if ocr_for_support:
        ref_nouns.update(eval_suite._extract_nouns(ocr_for_support))
    support = eval_suite._expand_wordnet(ref_nouns) | src_lemmas | src_raw | STRUCTURAL_STRICT

    bad_nouns = [n for n in eval_suite._extract_nouns(fused) if n not in support]
    if bad_nouns:
        return False, f"unsup_nouns={bad_nouns}", fused
    ref_nums = set(re.findall(r"\d+", all_src))
    bad_nums = [n for n in re.findall(r"\d+", fused) if n not in ref_nums]
    if bad_nums:
        return False, f"unsup_nums={bad_nums}", fused
    return True, "OK", fused

prompts = [make_prompt_v3(anns_by_img[iid], format_ocr_v3(ocr_all.get(str(iid), {}).get("dets", []))) for iid in sample_ids]
t0 = time.time()
fused_out = []
for i in range(0, len(prompts), 32):
    inp = q_tok(prompts[i:i+32], padding=True, return_tensors="pt").to("cuda:0")
    in_len = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = q_model.generate(**inp, max_new_tokens=55, do_sample=False, repetition_penalty=1.05,
                               eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.pad_token_id)
    fused_out.extend(q_tok.batch_decode(out[:, in_len:], skip_special_tokens=True))

print(f"\nSpeed (1 GPU, bs=32): {len(sample_ids)/(time.time()-t0):.2f} img/s")
passed, lens, fails = 0, [], []
print("\n=== 12 SAMPLE V3 FUSED TARGETS ===")
for idx, (iid, raw_f) in enumerate(zip(sample_ids, fused_out)):
    ocr_v3 = format_ocr_v3(ocr_all.get(str(iid), {}).get("dets", []))
    ok, reason, clean_f = verify_v3(raw_f, anns_by_img[iid], ocr_v3)
    if ok:
        passed += 1; lens.append(len(clean_f.split()))
    else:
        fails.append(reason)
    if idx < 12:
        print(f"[{idx:2d}] ID {iid} | OCRv3={ocr_v3[:30]!r} | {reason} ({len(clean_f.split())}w): {clean_f}")

print(f"\nPass rate: {passed}/64 ({100*passed/64:.1f}%) | Avg passed words: {sum(lens)/max(len(lens),1):.1f} | Fails: {fails[:8]}")

del q_model, inp, out
gc.collect(); torch.cuda.empty_cache()

=== OCR V3 CLEANUP ON TRICKY VAL SAMPLES ===
  ID 28669: old='Y'  -->  v3='none'
  ID 25684: old='C'  -->  v3='none'
  ID 25437: old='8orm-X 0004416BA ngreient Purpose ..Antiseptic immediately allow to dry without s and plastics. DSP-M0-34'  -->  v3='0004416BA ngreient Purpose ..Antiseptic immediately allow to dry without and plastics. DSP-M0-34'
  ID 29464: old='2 pa d 1 (10151 b) NI NSS 0- nn ty 27 392 N rLa 07'  -->  v3='27 392 12 07 1.6 1 120 Cookies'
  ID 23635: old='ATH CAB for CUTIE CODES and KEYS HPL'  -->  v3='ATH CAB for CUTIE CODES and KEYS HPL'
  ID 28974: old='LG'  -->  v3='LG'
  ID 29498: old="Astro th'nFruity oufruits YOGOURT BRASSÉ 2% M.F./M.G. Source de 650g calcium 01176"  -->  v3="Astro th'nFruity oufruits YOGOURT BRASSÉ Source de 650g calcium 01176"
  ID 24143: old='Ensure'  -->  v3='Ensure'


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Speed (1 GPU, bs=32): 4.95 img/s

=== 12 SAMPLE V3 FUSED TARGETS ===
[ 0] ID 0 | OCRv3='Basil Leaves NET WT 0.62 OZ (1' | unsup_nouns=['countertop'] (18w): Basil leaves in a green and white plastic container with net weight 0.62 oz (17g) on a countertop.
[ 1] ID 1 | OCRv3='none' | unsup_nouns=['countertop'] (17w): A black can of Coca Cola Zero soda sits on a black countertop next to other items.
[ 2] ID 2 | OCRv3='Crushed Tomatoes inPuree Vine ' | OK (16w): Price Chopper brand can of crushed vine ripe tomatoes in puree, visible on a brown surface.
[ 3] ID 3 | OCRv3='.AT&T 4:49 PM Spotify Click he' | unsup_nouns=['button'] (19w): A screenshot of a Spotify page with a CAPTCHA "T36M" and case insensitive submit button, on a cell phone.
[ 4] ID 4 | OCRv3='OLARGARDENLIGHT By divine powe' | unsup_nouns=['branding'] (15w): A blue and yellow box with garden lights inside on someone's lap, featuring "OLARGARDENLIGHT" branding.
[ 5] ID 5 | OCRv3='none' | len=9 (9w): Person's feet in brown shoes